In [34]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import Dropdown, HBox, VBox, Output, HTML
from IPython.display import display, clear_output, HTML
from mpl_toolkits.mplot3d import Axes3D
from scipy.signal import find_peaks
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from matplotlib import patheffects as pe

### 1. Computing Energies (Electron, Hole and Photon) and Laser Wavelength 

In [76]:
h = 6.62607015e-34
hbar = 1.054571817e-34
c = 2.99792458e8
e = 1.602176634e-19
m0 = 9.1093837015e-31
Lx = np.array([10, 20, 30, 40], dtype=float)
Ly = np.array([10, 20, 30, 40], dtype=float)
Lz = np.array([2, 4, 6, 8], dtype=float)
epsilon = np.array([10, 50, 80, 100], dtype=float)
V0e = np.array([300, 325, 350, 375], dtype=float)
V0h = np.array([100, 125, 150, 175], dtype=float)
T_C = np.array([30, 80, 130, 200], dtype=float)
nx = []
ny = []
nz = []
nx_e = []
ny_e = []
nz_e = []
nx_h = []
ny_h = []
nz_h = []
materials = ["GaAs", "InP", "GaN", "CdSe"]
mu_e_vals = np.array([0.07*m0, 0.08*m0, 0.20*m0, 0.13*m0], dtype=float)
mu_h_vals = np.array([0.50*m0, 0.60*m0, 1.25*m0, 0.45*m0], dtype=float)
E_bind_vals = np.array([4.2, 10.0, 25.0, 20.0], dtype=float)
material_index = {mat: i for i, mat in enumerate(materials)}
x = np.linspace(-50.0, 50.0, 101)
y = np.linspace(-50.0, 50.0, 101)
z = np.linspace(-15.0, 15.0, 101)

def nm_to_m(x):
    return np.asarray(x, dtype=float) * 1e-9

def kVcm_to_Vm(E):
    return np.asarray(E, dtype=float) * 1e5  

def C_to_K(TC):
    return np.asarray(TC, dtype=float) + 273.15

def meV_to_J(x):
    return np.asarray(x, dtype=float) * 1e-3 * e

def eV_to_J(x):
    return np.asarray(x, dtype=float) * e

def J_to_eV(x):
    return np.asarray(x, dtype=float) / e

def J_to_meV(x):
    return 1e3 * np.asarray(x, dtype=float) / e

def mu_e(material):
    return mu_e_vals[material_index[material]]

def mu_h(material):
    return mu_h_vals[material_index[material]]

def E_bind(material):
    return E_bind_vals[material_index[material]]

def Eg_T(material, TC):
    T_K = C_to_K(TC)
    if material == "GaAs":
        return 1.519 - 5.405e-4 * T_K**2 / (T_K + 204.0)
    elif material == "InP":
        return 1.421 - 4.8e-4 * T_K**2 / (T_K + 235.0)
    elif material == "GaN":
        return 3.507 - 9.09e-4 * T_K**2 / (T_K + 830.0)
    elif material == "CdSe":
        return 1.84 - 4.9e-4 * T_K**2 / (T_K + 250.0)
    else:
        raise ValueError(f"Unknown material: {material}")

def delta_e(material, V0e):
    return hbar / np.sqrt(2.0 * mu_e(material) * meV_to_J(V0e))

def delta_h(material, V0h):
    return hbar / np.sqrt(2.0 * mu_h(material) * meV_to_J(V0h))

def psi(nx, ny, nz, x, y, z, Lx, Ly, Lz):
    return np.sqrt(8.0 / (Lx * Ly * Lz)) * \
           np.sin(nx * np.pi * x / Lx) * \
           np.sin(ny * np.pi * y / Ly) * \
           np.sin(nz * np.pi * z / Lz)

def psisquared(nx, ny, nz, x, y, z, Lx, Ly, Lz):
    return (8.0 / (Lx * Ly * Lz)) * \
           np.sin(nx * np.pi * x / Lx)**2 * \
           np.sin(ny * np.pi * y / Ly)**2 * \
           np.sin(nz * np.pi * z / Lz)**2

def electron_energies(nx_e, ny_e, nz_e, Lx, Ly, Lz, epsilon, V0e, material):
    Lx = nm_to_m(Lx)
    Ly = nm_to_m(Ly)
    Lz = nm_to_m(Lz)
    eps = kVcm_to_Vm(epsilon)
    de = delta_e(material, V0e)
    Lx_eff = Lx + 2.0 * de
    Ly_eff = Ly + 2.0 * de
    Lz_eff = Lz + 2.0 * de
    E_e = (np.pi**2 * hbar**2 / (2.0 * mu_e(material))) * (
        nx_e**2 / Lx_eff**2 +
        ny_e**2 / Ly_eff**2 +
        nz_e**2 / Lz_eff**2
    )
    correction1e = -e * eps * (Lz_eff / 2.0)
    return E_e + correction1e

def hole_energies(nx_h, ny_h, nz_h, Lx, Ly, Lz, epsilon, V0h, material):
    Lx = nm_to_m(Lx)
    Ly = nm_to_m(Ly)
    Lz = nm_to_m(Lz)
    eps = kVcm_to_Vm(epsilon)
    dh = delta_h(material, V0h)
    Lx_eff = Lx + 2.0 * dh
    Ly_eff = Ly + 2.0 * dh
    Lz_eff = Lz + 2.0 * dh
    E_h = (np.pi**2 * hbar**2 / (2.0 * mu_h(material))) * (
        nx_h**2 / Lx_eff**2 +
        ny_h**2 / Ly_eff**2 +
        nz_h**2 / Lz_eff**2
    )
    correction1h = +e * eps * (Lz_eff / 2.0)
    return E_h + correction1h

def photon_wavelength(material,
                         nx_e, ny_e, nz_e,
                         nx_h, ny_h, nz_h,
                         Lx_nm, Ly_nm, Lz_nm,
                         epsilon_kVcm, V0e_meV, V0h_meV, TC):
    Eg = eV_to_J(Eg_T(material, TC))
    Ebind = meV_to_J(E_bind(material))
    Ee = electron_energies(
        nx_e, ny_e, nz_e,
        Lx_nm, Ly_nm, Lz_nm,
        epsilon_kVcm, V0e_meV, material
    )
    Eh = hole_energies(
        nx_h, ny_h, nz_h,
        Lx_nm, Ly_nm, Lz_nm,
        epsilon_kVcm, V0h_meV, material
    )
    E_gamma = Eg + Ee + Eh - Ebind
    lambda_gamma = h * c / E_gamma
    return lambda_gamma * 1e9

E_g = {
    mat: Eg_T(mat, T_C)
    for mat in materials
}

In [77]:
def em_spectrum_region(lambda_nm):
    if lambda_nm < 0.01:
        return "Gamma ray"
    elif lambda_nm < 10.0:
        return "X-ray"
    elif lambda_nm < 400.0:
        return "Ultraviolet (UV)"
    elif lambda_nm < 450.0:
        return "Visible light (violet)"
    elif lambda_nm < 495.0:
        return "Visible light (blue)"
    elif lambda_nm < 570.0:
        return "Visible light (green)"
    elif lambda_nm < 590.0:
        return "Visible light (yellow)"
    elif lambda_nm < 620.0:
        return "Visible light (orange)"
    elif lambda_nm < 750.0:
        return "Visible light (red)"
    elif lambda_nm < 1.0e6:
        return "Infrared (IR)"         
    elif lambda_nm < 1.0e8:
        return "Microwave"             
    else:
        return "Radio wave"

material_dropdown = Dropdown(
    options=materials,
    value=materials[0],
    description='Material:',
    layout={'width': '250px'}
)

Lx_dropdown = Dropdown(
    options=[float(v) for v in Lx],
    value=float(Lx[0]),
    description='Lₓ (nm):',
    layout={'width': '220px'}
)

Ly_dropdown = Dropdown(
    options=[float(v) for v in Ly],
    value=float(Ly[0]),
    description='Lᵧ (nm):',
    layout={'width': '220px'}
)

Lz_dropdown = Dropdown(
    options=[float(v) for v in Lz],
    value=float(Lz[0]),
    description='L𝓏 (nm):',
    layout={'width': '220px'}
)

epsilon_dropdown = Dropdown(
    options=[float(v) for v in epsilon],
    value=float(epsilon[0]),
    description='ε (kV/cm):',
    layout={'width': '220px'}
)

V0e_dropdown = Dropdown(
    options=[float(v) for v in V0e],
    value=float(V0e[0]),
    description='V₀ₑ (meV):',
    layout={'width': '220px'}
)

V0h_dropdown = Dropdown(
    options=[float(v) for v in V0h],
    value=float(V0h[0]),
    description='V₀ₕ (meV):',
    layout={'width': '220px'}
)

T_dropdown = Dropdown(
    options=[float(v) for v in T_C],
    value=float(T_C[0]),
    description='T (°C):',
    layout={'width': '220px'}
)

n_options = [1, 2, 3, 4]
nxe_dropdown = Dropdown(options=n_options, value=1, description='nₓₑ:', layout={'width': '160px'})
nye_dropdown = Dropdown(options=n_options, value=1, description='nᵧₑ:', layout={'width': '160px'})
nze_dropdown = Dropdown(options=n_options, value=1, description='n𝓏ₑ:', layout={'width': '160px'})
nxh_dropdown = Dropdown(options=n_options, value=1, description='nₓₕ:', layout={'width': '160px'})
nyh_dropdown = Dropdown(options=n_options, value=1, description='nᵧₕ:', layout={'width': '160px'})
nzh_dropdown = Dropdown(options=n_options, value=1, description='n𝓏ₕ:', layout={'width': '160px'})

output = Output()

def update_output(*args):
    with output:
        clear_output(wait=True)

        material = material_dropdown.value
        Lx_val = Lx_dropdown.value
        Ly_val = Ly_dropdown.value
        Lz_val = Lz_dropdown.value
        eps_val = epsilon_dropdown.value
        V0e_val = V0e_dropdown.value
        V0h_val = V0h_dropdown.value
        T_val = T_dropdown.value
        nx_e_val = nxe_dropdown.value
        ny_e_val = nye_dropdown.value
        nz_e_val = nze_dropdown.value
        nx_h_val = nxh_dropdown.value
        ny_h_val = nyh_dropdown.value
        nz_h_val = nzh_dropdown.value

        E_e_J = float(np.asarray(
            electron_energies(nx_e_val, ny_e_val, nz_e_val,
                              Lx_val, Ly_val, Lz_val,
                              eps_val, V0e_val, material)
        ))

        E_h_J = float(np.asarray(
            hole_energies(nx_h_val, ny_h_val, nz_h_val,
                          Lx_val, Ly_val, Lz_val,
                          eps_val, V0h_val, material)
        ))

        lam_nm = float(np.asarray(
            photon_wavelength(material,
                                 nx_e_val, ny_e_val, nz_e_val,
                                 nx_h_val, ny_h_val, nz_h_val,
                                 Lx_val, Ly_val, Lz_val,
                                 eps_val, V0e_val, V0h_val, T_val)
        ))

        spectrum_region = em_spectrum_region(lam_nm)

        E_e_eV = float(J_to_eV(E_e_J))
        E_e_meV = float(J_to_meV(E_e_J))
        E_h_eV = float(J_to_eV(E_h_J))
        E_h_meV = float(J_to_meV(E_h_J))
        Eg_eV = float(np.asarray(Eg_T(material, T_val)))
        Ebind_meV = float(np.asarray(E_bind(material)))

        print("=" * 120)
        print("QUANTUM STATE ENERGIES")
        print("=" * 120)
        print("\n" + "#" * 120)
        print(f"MATERIAL: {material}")
        print("#" * 120)

        print("\n" + "─" * 120)
        print("VARIATIONAL PARAMETERS")
        print("─" * 120)
        print(f"Lₓ   = {Lx_val:.3f} nm")
        print(f"Lᵧ   = {Ly_val:.3f} nm")
        print(f"L𝓏   = {Lz_val:.3f} nm")
        print(f"ε    = {eps_val:.3f} kV/cm")
        print(f"V₀ₑ  = {V0e_val:.3f} meV")
        print(f"V₀ₕ  = {V0h_val:.3f} meV")
        print(f"T    = {T_val:.3f} °C")
        print(f"nₓₑ  = {nx_e_val}")
        print(f"nᵧₑ  = {ny_e_val}")
        print(f"n𝓏ₑ  = {nz_e_val}")
        print(f"nₓₕ  = {nx_h_val}")
        print(f"nᵧₕ  = {ny_h_val}")
        print(f"n𝓏ₕ  = {nz_h_val}")
        print(f"Electron Quantum State: (nₓₑ,nᵧₑ,n𝓏ₑ) = ({nx_e_val},{ny_e_val},{nz_e_val})")
        print(f"Hole Quantum State: (nₓₕ,nᵧₕ,n𝓏ₕ) = ({nx_h_val},{ny_h_val},{nz_h_val})")

        print("\n" + "─" * 120)
        print("ELECTRON ENERGY")
        print("─" * 120)
        print(f"Eₑ = {E_e_J:.6e} J   =   {E_e_eV:.6e} eV   =   {E_e_meV:.6f} meV")

        print("\n" + "─" * 120)
        print("HOLE ENERGY")
        print("─" * 120)
        print(f"Eₕ = {E_h_J:.6e} J   =   {E_h_eV:.6e} eV   =   {E_h_meV:.6f} meV")

        print("\n" + "─" * 120)
        print("PHOTON WAVELENGTH")
        print("─" * 120)
        print(f"Eᵧ = {Eg_eV:.6f} eV   |   E_bind = {Ebind_meV:.6f} meV")
        print(f"λ = {lam_nm:.6f} nm")
        print(f"EM spectrum region = {spectrum_region}")

controls_row1 = HBox([material_dropdown, T_dropdown])
controls_row2 = HBox([Lx_dropdown, Ly_dropdown, Lz_dropdown])
controls_row3 = HBox([epsilon_dropdown, V0e_dropdown, V0h_dropdown])
controls_row4 = HBox([nxe_dropdown, nye_dropdown, nze_dropdown])
controls_row5 = HBox([nxh_dropdown, nyh_dropdown, nzh_dropdown])

ui = VBox([
    widgets.HTML("<h3>Parameters</h3>"),
    controls_row1,
    controls_row2,
    controls_row3,
    widgets.HTML("<b>Electron quantum state numbers</b>"),
    controls_row4,
    widgets.HTML("<b>Hole quantum state numbers</b>"),
    controls_row5,
    output
])

for w in [
    material_dropdown, Lx_dropdown, Ly_dropdown, Lz_dropdown,
    epsilon_dropdown, V0e_dropdown, V0h_dropdown, T_dropdown,
    nxe_dropdown, nye_dropdown, nze_dropdown,
    nxh_dropdown, nyh_dropdown, nzh_dropdown
]:
    w.observe(update_output, names='value')

display(ui)
update_output()

### 1. Variational Parameter Study

In [78]:
h = 6.62607015e-34
hbar = 1.054571817e-34
c = 2.99792458e8
e = 1.602176634e-19
m0 = 9.1093837015e-31
Lx_vals = np.array([10, 20, 30, 40], dtype=float)
Ly_vals = np.array([10, 20, 30, 40], dtype=float)
Lz_vals = np.array([2, 4, 6, 8], dtype=float)
epsilon_vals = np.array([10, 50, 80, 100], dtype=float)
V0e_vals = np.array([300, 325, 350, 375], dtype=float)
V0h_vals = np.array([100, 125, 150, 175], dtype=float)
T_C_vals = np.array([30, 80, 130, 200], dtype=float)
n_vals = [1, 2, 3, 4]
materials = ["GaAs", "InP", "GaN", "CdSe"]
mu_e_vals = np.array([0.06*m0, 0.08*m0, 0.20*m0, 0.13*m0], dtype=float)
mu_h_vals = np.array([0.04*m0, 0.60*m0, 1.25*m0, 0.45*m0], dtype=float)
E_bind_vals = np.array([4.2, 10.0, 25.0, 30.0], dtype=float)
material_index = {mat: i for i, mat in enumerate(materials)}

def nm_to_m(x):
    return np.asarray(x, dtype=float) * 1e-9

def kVcm_to_Vm(E):
    return np.asarray(E, dtype=float) * 1e5

def C_to_K(TC):
    return np.asarray(TC, dtype=float) + 273.15

def meV_to_J(x):
    return np.asarray(x, dtype=float) * 1e-3 * e

def eV_to_J(x):
    return np.asarray(x, dtype=float) * e

def J_to_eV(x):
    return np.asarray(x, dtype=float) / e

def J_to_meV(x):
    return 1e3 * np.asarray(x, dtype=float) / e

def mu_e(material):
    return mu_e_vals[material_index[material]]

def mu_h(material):
    return mu_h_vals[material_index[material]]

def E_bind(material):
    return E_bind_vals[material_index[material]]

def Eg_T(material, TC):
    T_K = C_to_K(TC)
    if material == "GaAs":
        return 1.519 - 5.405e-4 * T_K**2 / (T_K + 204.0)
    elif material == "InP":
        return 1.421 - 4.8e-4 * T_K**2 / (T_K + 235.0)
    elif material == "GaN":
        return 3.507 - 9.09e-4 * T_K**2 / (T_K + 830.0)
    elif material == "CdSe":
        return 1.84 - 4.9e-4 * T_K**2 / (T_K + 250.0)
    else:
        raise ValueError(f"Unknown material: {material}")

def delta_e(material, V0e):
    return hbar / np.sqrt(2.0 * mu_e(material) * meV_to_J(V0e))

def delta_h(material, V0h):
    return hbar / np.sqrt(2.0 * mu_h(material) * meV_to_J(V0h))

def L_eff_e(material, L_nm, V0e):
    return nm_to_m(L_nm) + 2.0 * delta_e(material, V0e)

def L_eff_h(material, L_nm, V0h):
    return nm_to_m(L_nm) + 2.0 * delta_h(material, V0h)

delta_e_vals = np.array([
    1e9 * delta_e(mat, V0e)
    for mat in materials
    for V0e in V0e_vals
], dtype=float)

delta_h_vals = np.array([
    1e9 * delta_h(mat, V0h)
    for mat in materials
    for V0h in V0h_vals
], dtype=float)

delta_e_vals = np.array(sorted(set(np.round(delta_e_vals, 6))), dtype=float)
delta_h_vals = np.array(sorted(set(np.round(delta_h_vals, 6))), dtype=float)

def electron_energies(nx_e, ny_e, nz_e, Lx, Ly, Lz, epsilon, V0e, material):
    Lx = nm_to_m(Lx)
    Ly = nm_to_m(Ly)
    Lz = nm_to_m(Lz)
    eps = kVcm_to_Vm(epsilon)
    de = delta_e(material, V0e)
    Lx_eff = Lx + 2.0 * de
    Ly_eff = Ly + 2.0 * de
    Lz_eff = Lz + 2.0 * de
    E_e = (np.pi**2 * hbar**2 / (2.0 * mu_e(material))) * (
        nx_e**2 / Lx_eff**2 + ny_e**2 / Ly_eff**2 + nz_e**2 / Lz_eff**2
    )
    correction1e = -e * eps * (Lz_eff / 2.0)
    return E_e + correction1e

def hole_energies(nx_h, ny_h, nz_h, Lx, Ly, Lz, epsilon, V0h, material):
    Lx = nm_to_m(Lx)
    Ly = nm_to_m(Ly)
    Lz = nm_to_m(Lz)
    eps = kVcm_to_Vm(epsilon)
    dh = delta_h(material, V0h)
    Lx_eff = Lx + 2.0 * dh
    Ly_eff = Ly + 2.0 * dh
    Lz_eff = Lz + 2.0 * dh
    E_h = (np.pi**2 * hbar**2 / (2.0 * mu_h(material))) * (
        nx_h**2 / Lx_eff**2 + ny_h**2 / Ly_eff**2 + nz_h**2 / Lz_eff**2
    )
    correction1h = +e * eps * (Lz_eff / 2.0)
    return E_h + correction1h

def photon_energy(material,
                    nx_e, ny_e, nz_e,
                    nx_h, ny_h, nz_h,
                    Lx, Ly, Lz,
                    epsilon, V0e, V0h, T_C):
    Eg = eV_to_J(Eg_T(material, T_C))
    Ee = electron_energies(nx_e, ny_e, nz_e, Lx, Ly, Lz, epsilon, V0e, material)
    Eh = hole_energies(nx_h, ny_h, nz_h, Lx, Ly, Lz, epsilon, V0h, material)
    Ebind = meV_to_J(E_bind(material))
    return Eg + Ee + Eh - Ebind

def photon_wavelength(material,
                         nx_e, ny_e, nz_e,
                         nx_h, ny_h, nz_h,
                         Lx, Ly, Lz,
                         epsilon, V0e, V0h, T_C):
    E_gamma = photon_energy(material,
                              nx_e, ny_e, nz_e,
                              nx_h, ny_h, nz_h,
                              Lx, Ly, Lz,
                              epsilon, V0e, V0h, T_C)
    return (h * c / E_gamma) * 1e9

#### a. Confinement Region Effects ($L_x, L_y, L_z$) and Energy Level Diagrams

In [79]:
confinement_material_dropdown = Dropdown(
    options=["All materials"] + materials,
    value="All materials",
    description='Material:'
)

confinement_quantity_dropdown = Dropdown(
    options=["Eₑ", "Eₕ", "Eᵧ", "λᵧ"],
    value="Eₑ",
    description='Quantity:'
)

confinement_x_dropdown = Dropdown(
    options=["Lₓ", "Lᵧ", "Lz"],
    value="Lₓ",
    description='x-axis:'
)

confinement_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[-1]), description='Lₓ,max (nm):')
confinement_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
confinement_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
confinement_epsilon_dropdown = Dropdown(options=[float(v) for v in epsilon_vals], value=float(epsilon_vals[0]), description='ε (kV/cm):')
confinement_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
confinement_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
confinement_T_dropdown = Dropdown(options=[float(v) for v in T_C_vals], value=float(T_C_vals[0]), description='T (°C):')
confinement_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
confinement_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
confinement_nze_dropdown = Dropdown(options=n_vals, value=1, description='n𝓏ₑ:')
confinement_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
confinement_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
confinement_nzh_dropdown = Dropdown(options=n_vals, value=1, description='n𝓏ₕ:')
confinement_output = Output()

def confinement_visibility(*args):
    confinement_Lx_dropdown.layout.display = ''
    confinement_Ly_dropdown.layout.display = ''
    confinement_Lz_dropdown.layout.display = ''
    if confinement_x_dropdown.value == "Lₓ":
        confinement_Lx_dropdown.description = 'Lₓ,max (nm):'
    else:
        confinement_Lx_dropdown.description = 'Lₓ (nm):'
    if confinement_x_dropdown.value == "Lᵧ":
        confinement_Ly_dropdown.description = 'Lᵧ,max (nm):'
    else:
        confinement_Ly_dropdown.description = 'Lᵧ (nm):'
    if confinement_x_dropdown.value == "Lz":
        confinement_Lz_dropdown.description = 'Lz,max (nm):'
    else:
        confinement_Lz_dropdown.description = 'Lz (nm):'

def confinement_plot(*args):
    with confinement_output:
        clear_output(wait=True)
        quantity = confinement_quantity_dropdown.value
        xaxis = confinement_x_dropdown.value
        mats = materials if confinement_material_dropdown.value == "All materials" else [confinement_material_dropdown.value]
        if xaxis == "Lₓ":
            xs = np.linspace(1.0, confinement_Lx_dropdown.value, 300)
        elif xaxis == "Lᵧ":
            xs = np.linspace(1.0, confinement_Ly_dropdown.value, 300)
        else:
            xs = np.linspace(0.5, confinement_Lz_dropdown.value, 300)
        plt.figure(figsize=(9, 6))
        for mat in mats:
            ys = []
            for x in xs:
                Lx = x if xaxis == "Lₓ" else confinement_Lx_dropdown.value
                Ly = x if xaxis == "Lᵧ" else confinement_Ly_dropdown.value
                Lz = x if xaxis == "Lz" else confinement_Lz_dropdown.value
                if quantity == "Eₑ":
                    y = J_to_meV(electron_energies(
                        confinement_nxe_dropdown.value, confinement_nye_dropdown.value, confinement_nze_dropdown.value,
                        Lx, Ly, Lz,
                        confinement_epsilon_dropdown.value,
                        confinement_V0e_dropdown.value,
                        mat
                    ))
                    ylabel = "Eₑ (meV)"
                elif quantity == "Eₕ":
                    y = J_to_meV(hole_energies(
                        confinement_nxh_dropdown.value, confinement_nyh_dropdown.value, confinement_nzh_dropdown.value,
                        Lx, Ly, Lz,
                        confinement_epsilon_dropdown.value,
                        confinement_V0h_dropdown.value,
                        mat
                    ))
                    ylabel = "Eₕ (meV)"
                elif quantity == "Eᵧ":
                    y = J_to_eV(photon_energy(
                        mat,
                        confinement_nxe_dropdown.value, confinement_nye_dropdown.value, confinement_nze_dropdown.value,
                        confinement_nxh_dropdown.value, confinement_nyh_dropdown.value, confinement_nzh_dropdown.value,
                        Lx, Ly, Lz,
                        confinement_epsilon_dropdown.value,
                        confinement_V0e_dropdown.value,
                        confinement_V0h_dropdown.value,
                        confinement_T_dropdown.value
                    ))
                    ylabel = "Eᵧ (eV)"
                else:
                    y = photon_wavelength(
                        mat,
                        confinement_nxe_dropdown.value, confinement_nye_dropdown.value, confinement_nze_dropdown.value,
                        confinement_nxh_dropdown.value, confinement_nyh_dropdown.value, confinement_nzh_dropdown.value,
                        Lx, Ly, Lz,
                        confinement_epsilon_dropdown.value,
                        confinement_V0e_dropdown.value,
                        confinement_V0h_dropdown.value,
                        confinement_T_dropdown.value
                    )
                    ylabel = "λᵧ (nm)"
                ys.append(float(np.asarray(y)))
            plt.plot(xs, ys, linewidth=2, label=mat)
        plt.xlabel(f"{xaxis} (nm)")
        plt.ylabel(ylabel)
        plt.title(f"{quantity} vs {xaxis}")
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=10, title='Material', edgecolor='black', facecolor='lightgray')
        plt.show()
        
for widget in [
    confinement_material_dropdown, confinement_quantity_dropdown, confinement_x_dropdown,
    confinement_Lx_dropdown, confinement_Ly_dropdown, confinement_Lz_dropdown,
    confinement_epsilon_dropdown, confinement_V0e_dropdown, confinement_V0h_dropdown, confinement_T_dropdown,
    confinement_nxe_dropdown, confinement_nye_dropdown, confinement_nze_dropdown,
    confinement_nxh_dropdown, confinement_nyh_dropdown, confinement_nzh_dropdown
]:
    widget.observe(confinement_plot, names='value')
confinement_x_dropdown.observe(confinement_visibility, names='value')
display(VBox([
    HBox([confinement_material_dropdown, confinement_quantity_dropdown, confinement_x_dropdown]),
    HBox([confinement_Lx_dropdown, confinement_Ly_dropdown, confinement_Lz_dropdown]),
    HBox([confinement_epsilon_dropdown, confinement_V0e_dropdown, confinement_V0h_dropdown, confinement_T_dropdown]),
    HBox([confinement_nxe_dropdown, confinement_nye_dropdown, confinement_nze_dropdown]),
    HBox([confinement_nxh_dropdown, confinement_nyh_dropdown, confinement_nzh_dropdown]),
    confinement_output
]))
confinement_visibility()
confinement_plot()

mat2_dd = Dropdown(options=["All materials"]+materials,value="All materials",description='Material:')
out2 = Output()
def energy_level(*args):
    with out2:
        clear_output(wait=True)
        mats = materials if mat2_dd.value=="All materials" else [mat2_dd.value]
        fig,axes = plt.subplots(1,len(mats),figsize=(6*len(mats),5))
        if len(mats)==1: axes=[axes]
        for ax,m in zip(axes,mats):
            for n in [1,2,3,4]:
                ax.hlines(n,0,1)
                ax.text(1.05,n,f"Eₑ({n},{n},{n})")
                ax.hlines(-n,2,3)
                ax.text(3.05,-n,f"Eₕ({n},{n},{n})")
                ax.annotate('',xy=(2,-n),xytext=(1,n),arrowprops=dict(arrowstyle='->'))
            ax.set_xticks([0.5,2.5])
            ax.set_xticklabels(['e⁻ states','h⁺ states'])
            ax.set_title(m)
        plt.show()
mat2_dd.observe(energy_level,'value')
display(VBox([mat2_dd,out2]))
energy_level()

#### b. Quantum State Effects ($n_x^{(e)}, n_y^{(e)}, n_z^{(e)}, n_x^{(h)}, n_y^{(h)}, n_z^{(h)}$) 

In [80]:
quantum_material_dropdown = Dropdown(
    options=["All materials"] + materials,
    value="All materials",
    description='Material:'
)

quantum_quantity_dropdown = Dropdown(
    options=["Eₑ", "Eₕ", "Eᵧ"],
    value="Eₑ",
    description='Quantity:'
)

quantum_x_dropdown = Dropdown(
    options=["nₓₑ", "nᵧₑ", "nzₑ", "nₓₕ", "nᵧₕ", "nzₕ"],
    value="nₓₑ",
    description='x-axis:'
)

quantum_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='Lₓ (nm):')
quantum_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
quantum_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
quantum_epsilon_dropdown = Dropdown(options=[float(v) for v in epsilon_vals], value=float(epsilon_vals[0]), description='ε (kV/cm):')
quantum_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
quantum_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
quantum_T_dropdown = Dropdown(options=[float(v) for v in T_C_vals], value=float(T_C_vals[0]), description='T (°C):')
quantum_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
quantum_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
quantum_nze_dropdown = Dropdown(options=n_vals, value=1, description='n𝓏ₑ:')
quantum_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
quantum_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
quantum_nzh_dropdown = Dropdown(options=n_vals, value=1, description='nzₕ:')
quantum_output = Output()

def quantum_visibility(*args):
    pass

def quantum_plot(*args):
    with quantum_output:
        clear_output(wait=True)
        quantity = quantum_quantity_dropdown.value
        xaxis = quantum_x_dropdown.value
        mats = materials if quantum_material_dropdown.value == "All materials" else [quantum_material_dropdown.value]
        xs = np.array(n_vals, dtype=float)
        plt.figure(figsize=(9, 6))
        for mat in mats:
            ys = []
            for x in xs:
                nxe = int(x) if xaxis == "nₓₑ" else quantum_nxe_dropdown.value
                nye = int(x) if xaxis == "nᵧₑ" else quantum_nye_dropdown.value
                nze = int(x) if xaxis == "nzₑ" else quantum_nze_dropdown.value
                nxh = int(x) if xaxis == "nₓₕ" else quantum_nxh_dropdown.value
                nyh = int(x) if xaxis == "nᵧₕ" else quantum_nyh_dropdown.value
                nzh = int(x) if xaxis == "nzₕ" else quantum_nzh_dropdown.value
                if quantity == "Eₑ":
                    y = J_to_meV(electron_energies(
                        nxe, nye, nze,
                        quantum_Lx_dropdown.value, quantum_Ly_dropdown.value, quantum_Lz_dropdown.value,
                        quantum_epsilon_dropdown.value,
                        quantum_V0e_dropdown.value,
                        mat
                    ))
                    ylabel = "Eₑ (meV)"
                elif quantity == "Eₕ":
                    y = J_to_meV(hole_energies(
                        nxh, nyh, nzh,
                        quantum_Lx_dropdown.value, quantum_Ly_dropdown.value, quantum_Lz_dropdown.value,
                        quantum_epsilon_dropdown.value,
                        quantum_V0h_dropdown.value,
                        mat
                    ))
                    ylabel = "Eₕ (meV)"
                else:
                    y = J_to_eV(photon_energy(
                        mat,
                        nxe, nye, nze,
                        nxh, nyh, nzh,
                        quantum_Lx_dropdown.value, quantum_Ly_dropdown.value, quantum_Lz_dropdown.value,
                        quantum_epsilon_dropdown.value,
                        quantum_V0e_dropdown.value,
                        quantum_V0h_dropdown.value,
                        quantum_T_dropdown.value
                    ))
                    ylabel = "Eᵧ (eV)"
                ys.append(float(np.asarray(y)))
            plt.plot(xs, ys, marker='o', linewidth=2, label=mat)
        plt.xlabel(xaxis)
        plt.ylabel(ylabel)
        plt.title(f"{quantity} vs {xaxis}")
        plt.grid(True, alpha=0.3)
        plt.xticks(n_vals)
        plt.legend(fontsize=10, title='Material', edgecolor='black', facecolor='lightgray')
        plt.show()

for widget in [
    quantum_material_dropdown, quantum_quantity_dropdown, quantum_x_dropdown,
    quantum_Lx_dropdown, quantum_Ly_dropdown, quantum_Lz_dropdown,
    quantum_epsilon_dropdown, quantum_V0e_dropdown, quantum_V0h_dropdown, quantum_T_dropdown,
    quantum_nxe_dropdown, quantum_nye_dropdown, quantum_nze_dropdown,
    quantum_nxh_dropdown, quantum_nyh_dropdown, quantum_nzh_dropdown
]:
    widget.observe(quantum_plot, names='value')
quantum_x_dropdown.observe(quantum_visibility, names='value')
display(VBox([
    HBox([quantum_material_dropdown, quantum_quantity_dropdown, quantum_x_dropdown]),
    HBox([quantum_Lx_dropdown, quantum_Ly_dropdown, quantum_Lz_dropdown]),
    HBox([quantum_epsilon_dropdown, quantum_V0e_dropdown, quantum_V0h_dropdown, quantum_T_dropdown]),
    HBox([quantum_nxe_dropdown, quantum_nye_dropdown, quantum_nze_dropdown]),
    HBox([quantum_nxh_dropdown, quantum_nyh_dropdown, quantum_nzh_dropdown]),
    quantum_output
]))
quantum_visibility()
quantum_plot()

#### c. Electric Field Effects ($\epsilon$)

In [81]:
stark_material_dropdown = Dropdown(
    options=["All materials"] + materials,
    value="All materials",
    description='Material:'
)

stark_quantity_dropdown = Dropdown(
    options=["Eₑ", "Eₕ", "Eᵧ", "λ"],
    value="Eₑ",
    description='Quantity:'
)

stark_x_dropdown = Dropdown(
    options=["ε"],
    value="ε",
    description='x-axis:'
)

stark_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='Lₓ (nm):')
stark_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
stark_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
stark_epsilon_dropdown = Dropdown(options=[float(v) for v in epsilon_vals], value=float(epsilon_vals[-1]), description='ε,max (kV/cm):')
stark_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
stark_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
stark_T_dropdown = Dropdown(options=[float(v) for v in T_C_vals], value=float(T_C_vals[0]), description='T (°C):')
stark_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
stark_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
stark_nze_dropdown = Dropdown(options=n_vals, value=1, description='n𝓏ₑ:')
stark_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
stark_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
stark_nzh_dropdown = Dropdown(options=n_vals, value=1, description='n𝓏ₕ:')
stark_output = Output()

def stark_visibility(*args):
    stark_epsilon_dropdown.description = 'ε,max (kV/cm):'

def stark_plot(*args):
    with stark_output:
        clear_output(wait=True)
        quantity = stark_quantity_dropdown.value
        xaxis = stark_x_dropdown.value
        mats = materials if stark_material_dropdown.value == "All materials" else [stark_material_dropdown.value]
        xs = np.linspace(0.0, stark_epsilon_dropdown.value, 300)
        plt.figure(figsize=(9, 6))
        for mat in mats:
            ys = []
            for x in xs:
                epsilon = x
                if quantity == "Eₑ":
                    y = J_to_meV(electron_energies(
                        stark_nxe_dropdown.value, stark_nye_dropdown.value, stark_nze_dropdown.value,
                        stark_Lx_dropdown.value, stark_Ly_dropdown.value, stark_Lz_dropdown.value,
                        epsilon,
                        stark_V0e_dropdown.value,
                        mat
                    ))
                    ylabel = "Eₑ (meV)"
                elif quantity == "Eₕ":
                    y = J_to_meV(hole_energies(
                        stark_nxh_dropdown.value, stark_nyh_dropdown.value, stark_nzh_dropdown.value,
                        stark_Lx_dropdown.value, stark_Ly_dropdown.value, stark_Lz_dropdown.value,
                        epsilon,
                        stark_V0h_dropdown.value,
                        mat
                    ))
                    ylabel = "Eₕ (meV)"
                elif quantity == "Eᵧ":
                    y = J_to_eV(photon_energy(
                        mat,
                        stark_nxe_dropdown.value, stark_nye_dropdown.value, stark_nze_dropdown.value,
                        stark_nxh_dropdown.value, stark_nyh_dropdown.value, stark_nzh_dropdown.value,
                        stark_Lx_dropdown.value, stark_Ly_dropdown.value, stark_Lz_dropdown.value,
                        epsilon,
                        stark_V0e_dropdown.value,
                        stark_V0h_dropdown.value,
                        stark_T_dropdown.value
                    ))
                    ylabel = "Eᵧ (eV)"
                else:
                    y = photon_wavelength(
                        mat,
                        stark_nxe_dropdown.value, stark_nye_dropdown.value, stark_nze_dropdown.value,
                        stark_nxh_dropdown.value, stark_nyh_dropdown.value, stark_nzh_dropdown.value,
                        stark_Lx_dropdown.value, stark_Ly_dropdown.value, stark_Lz_dropdown.value,
                        epsilon,
                        stark_V0e_dropdown.value,
                        stark_V0h_dropdown.value,
                        stark_T_dropdown.value
                    )
                    ylabel = "λ (nm)"
                ys.append(float(np.asarray(y)))
            plt.plot(xs, ys, linewidth=2, label=mat)
        plt.xlabel(f"{xaxis} (kV/cm)")
        plt.ylabel(ylabel)
        plt.title(f"{quantity} vs {xaxis}")
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=10, title='Material', edgecolor='black', facecolor='lightgray')
        plt.show()
        
for widget in [
    stark_material_dropdown, stark_quantity_dropdown, stark_x_dropdown,
    stark_Lx_dropdown, stark_Ly_dropdown, stark_Lz_dropdown,
    stark_epsilon_dropdown, stark_V0e_dropdown, stark_V0h_dropdown, stark_T_dropdown,
    stark_nxe_dropdown, stark_nye_dropdown, stark_nze_dropdown,
    stark_nxh_dropdown, stark_nyh_dropdown, stark_nzh_dropdown
]:
    widget.observe(stark_plot, names='value')
stark_x_dropdown.observe(stark_visibility, names='value')
display(VBox([
    HBox([stark_material_dropdown, stark_quantity_dropdown, stark_x_dropdown]),
    HBox([stark_Lx_dropdown, stark_Ly_dropdown, stark_Lz_dropdown]),
    HBox([stark_epsilon_dropdown, stark_V0e_dropdown, stark_V0h_dropdown, stark_T_dropdown]),
    HBox([stark_nxe_dropdown, stark_nye_dropdown, stark_nze_dropdown]),
    HBox([stark_nxh_dropdown, stark_nyh_dropdown, stark_nzh_dropdown]),
    stark_output
]))
stark_visibility()
stark_plot()

#### d. Finite Barrier/Penetration Depth Effects ($V_{0e} (\delta_e), V_{0h} (\delta_h)$)

In [82]:
barrier_material_dropdown = Dropdown(
    options=["All materials"] + materials,
    value="All materials",
    description='Material:'
)

barrier_quantity_dropdown = Dropdown(
    options=["Eₑ", "Eₕ"],
    value="Eₑ",
    description='Quantity:'
)

barrier_x_dropdown = Dropdown(
    options=["V₀ₑ", "V₀ₕ", "δₑ", "δₕ"],
    value="V₀ₑ",
    description='x-axis:'
)

barrier_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='Lₓ (nm):')
barrier_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
barrier_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
barrier_epsilon_dropdown = Dropdown(options=[float(v) for v in epsilon_vals], value=float(epsilon_vals[0]), description='ε (kV/cm):')
barrier_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[-1]), description='V₀ₑ,max (meV):')
barrier_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[-1]), description='V₀ₕ,max (meV):')
barrier_deltae_dropdown = Dropdown(options=[float(v) for v in delta_e_vals], value=float(delta_e_vals[-1]), description='δₑ,max (nm):')
barrier_deltah_dropdown = Dropdown(options=[float(v) for v in delta_h_vals], value=float(delta_h_vals[-1]), description='δₕ,max (nm):')
barrier_T_dropdown = Dropdown(options=[float(v) for v in T_C_vals], value=float(T_C_vals[0]), description='T (°C):')
barrier_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
barrier_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
barrier_nze_dropdown = Dropdown(options=n_vals, value=1, description='nzₑ:')
barrier_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
barrier_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
barrier_nzh_dropdown = Dropdown(options=n_vals, value=1, description='nzₕ:')
barrier_output = Output()

def barrier_visibility(*args):
    if barrier_x_dropdown.value == "V₀ₑ":
        barrier_V0e_dropdown.description = 'V₀ₑ,max (meV):'
    else:
        barrier_V0e_dropdown.description = 'V₀ₑ (meV):'
    if barrier_x_dropdown.value == "V₀ₕ":
        barrier_V0h_dropdown.description = 'V₀ₕ,max (meV):'
    else:
        barrier_V0h_dropdown.description = 'V₀ₕ (meV):'
    if barrier_x_dropdown.value == "δₑ":
        barrier_deltae_dropdown.description = 'δₑ,max (nm):'
    else:
        barrier_deltae_dropdown.description = 'δₑ (nm):'
    if barrier_x_dropdown.value == "δₕ":
        barrier_deltah_dropdown.description = 'δₕ,max (nm):'
    else:
        barrier_deltah_dropdown.description = 'δₕ (nm):'

def barrier_plot(*args):
    with barrier_output:
        clear_output(wait=True)
        quantity = barrier_quantity_dropdown.value
        xaxis = barrier_x_dropdown.value
        mats = materials if barrier_material_dropdown.value == "All materials" else [barrier_material_dropdown.value]
        if xaxis == "V₀ₑ":
            xs = np.linspace(1.0, barrier_V0e_dropdown.value, 300)
        elif xaxis == "V₀ₕ":
            xs = np.linspace(1.0, barrier_V0h_dropdown.value, 300)
        elif xaxis == "δₑ":
            xs = np.linspace(0.01, barrier_deltae_dropdown.value, 300)
        else:
            xs = np.linspace(0.01, barrier_deltah_dropdown.value, 300)
        plt.figure(figsize=(9, 6))
        for mat in mats:
            ys = []
            for x in xs:
                V0e = x if xaxis == "V₀ₑ" else barrier_V0e_dropdown.value
                V0h = x if xaxis == "V₀ₕ" else barrier_V0h_dropdown.value
                deltae = x if xaxis == "δₑ" else barrier_deltae_dropdown.value
                deltah = x if xaxis == "δₕ" else barrier_deltah_dropdown.value
                if quantity == "Eₑ":
                    if xaxis == "δₑ":
                        y = J_to_meV(electron_energies(
                            barrier_nxe_dropdown.value, barrier_nye_dropdown.value, barrier_nze_dropdown.value,
                            barrier_Lx_dropdown.value + 2.0*deltae, barrier_Ly_dropdown.value + 2.0*deltae, barrier_Lz_dropdown.value + 2.0*deltae,
                            barrier_epsilon_dropdown.value,
                            barrier_V0e_dropdown.value,
                            mat
                        ))
                    else:
                        y = J_to_meV(electron_energies(
                            barrier_nxe_dropdown.value, barrier_nye_dropdown.value, barrier_nze_dropdown.value,
                            barrier_Lx_dropdown.value, barrier_Ly_dropdown.value, barrier_Lz_dropdown.value,
                            barrier_epsilon_dropdown.value,
                            V0e,
                            mat
                        ))
                    ylabel = "Eₑ (meV)"
                else:
                    if xaxis == "δₕ":
                        y = J_to_meV(hole_energies(
                            barrier_nxh_dropdown.value, barrier_nyh_dropdown.value, barrier_nzh_dropdown.value,
                            barrier_Lx_dropdown.value + 2.0*deltah, barrier_Ly_dropdown.value + 2.0*deltah, barrier_Lz_dropdown.value + 2.0*deltah,
                            barrier_epsilon_dropdown.value,
                            barrier_V0h_dropdown.value,
                            mat
                        ))
                    else:
                        y = J_to_meV(hole_energies(
                            barrier_nxh_dropdown.value, barrier_nyh_dropdown.value, barrier_nzh_dropdown.value,
                            barrier_Lx_dropdown.value, barrier_Ly_dropdown.value, barrier_Lz_dropdown.value,
                            barrier_epsilon_dropdown.value,
                            V0h,
                            mat
                        ))
                    ylabel = "Eₕ (meV)"
                ys.append(float(np.asarray(y)))
            plt.plot(xs, ys, linewidth=2, label=mat)
        if xaxis == "V₀ₑ":
            plt.xlabel("V₀ₑ (meV)")
        elif xaxis == "V₀ₕ":
            plt.xlabel("V₀ₕ (meV)")
        elif xaxis == "δₑ":
            plt.xlabel("δₑ (nm)")
        else:
            plt.xlabel("δₕ (nm)")
        plt.ylabel(ylabel)
        plt.title(f"{quantity} vs {xaxis}")
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=10, title='Material', edgecolor='black', facecolor='lightgray')
        plt.show()
        
for widget in [
    barrier_material_dropdown, barrier_quantity_dropdown, barrier_x_dropdown,
    barrier_Lx_dropdown, barrier_Ly_dropdown, barrier_Lz_dropdown,
    barrier_epsilon_dropdown, barrier_V0e_dropdown, barrier_V0h_dropdown,
    barrier_deltae_dropdown, barrier_deltah_dropdown, barrier_T_dropdown,
    barrier_nxe_dropdown, barrier_nye_dropdown, barrier_nze_dropdown,
    barrier_nxh_dropdown, barrier_nyh_dropdown, barrier_nzh_dropdown
]:
    widget.observe(barrier_plot, names='value')
barrier_x_dropdown.observe(barrier_visibility, names='value')
display(VBox([
    HBox([barrier_material_dropdown, barrier_quantity_dropdown, barrier_x_dropdown]),
    HBox([barrier_Lx_dropdown, barrier_Ly_dropdown, barrier_Lz_dropdown]),
    HBox([barrier_epsilon_dropdown, barrier_V0e_dropdown, barrier_V0h_dropdown, barrier_T_dropdown]),
    HBox([barrier_deltae_dropdown, barrier_deltah_dropdown]),
    HBox([barrier_nxe_dropdown, barrier_nye_dropdown, barrier_nze_dropdown]),
    HBox([barrier_nxh_dropdown, barrier_nyh_dropdown, barrier_nzh_dropdown]),
    barrier_output
]))
barrier_visibility()
barrier_plot()

#### e. Temperature Effects ($T$)

In [83]:
temperature_material_dropdown = Dropdown(
    options=["All materials"] + materials,
    value="All materials",
    description='Material:'
)

temperature_quantity_dropdown = Dropdown(
    options=["Eᵧ", "λ"],
    value="Eᵧ",
    description='Quantity:'
)

temperature_x_dropdown = Dropdown(
    options=["T"],
    value="T",
    description='x-axis:'
)

temperature_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='Lₓ (nm):')
temperature_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
temperature_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
temperature_epsilon_dropdown = Dropdown(options=[float(v) for v in epsilon_vals], value=float(epsilon_vals[0]), description='ε (kV/cm):')
temperature_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
temperature_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
temperature_T_dropdown = Dropdown(options=[float(v) for v in T_C_vals], value=float(T_C_vals[-1]), description='T,max (°C):')
temperature_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
temperature_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
temperature_nze_dropdown = Dropdown(options=n_vals, value=1, description='n𝓏ₑ:')
temperature_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
temperature_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
temperature_nzh_dropdown = Dropdown(options=n_vals, value=1, description='n𝓏ₕ:')
temperature_output = Output()

def temperature_visibility(*args):
    if temperature_x_dropdown.value == "T":
        temperature_T_dropdown.description = 'T,max (°C):'
    else:
        temperature_T_dropdown.description = 'T (°C):'

def temperature_plot(*args):
    with temperature_output:
        clear_output(wait=True)
        quantity = temperature_quantity_dropdown.value
        xaxis = temperature_x_dropdown.value
        mats = materials if temperature_material_dropdown.value == "All materials" else [temperature_material_dropdown.value]
        xs = np.linspace(1.0, temperature_T_dropdown.value, 300)
        plt.figure(figsize=(9, 6))
        for mat in mats:
            ys = []
            for x in xs:
                T = x
                if quantity == "Eᵧ":
                    y = J_to_eV(photon_energy(
                        mat,
                        temperature_nxe_dropdown.value, temperature_nye_dropdown.value, temperature_nze_dropdown.value,
                        temperature_nxh_dropdown.value, temperature_nyh_dropdown.value, temperature_nzh_dropdown.value,
                        temperature_Lx_dropdown.value, temperature_Ly_dropdown.value, temperature_Lz_dropdown.value,
                        temperature_epsilon_dropdown.value,
                        temperature_V0e_dropdown.value,
                        temperature_V0h_dropdown.value,
                        T
                    ))
                    ylabel = "Eᵧ (eV)"
                else:
                    y = photon_wavelength(
                        mat,
                        temperature_nxe_dropdown.value, temperature_nye_dropdown.value, temperature_nze_dropdown.value,
                        temperature_nxh_dropdown.value, temperature_nyh_dropdown.value, temperature_nzh_dropdown.value,
                        temperature_Lx_dropdown.value, temperature_Ly_dropdown.value, temperature_Lz_dropdown.value,
                        temperature_epsilon_dropdown.value,
                        temperature_V0e_dropdown.value,
                        temperature_V0h_dropdown.value,
                        T
                    )
                    ylabel = "λ (nm)"
                ys.append(float(np.asarray(y)))
            plt.plot(xs, ys, linewidth=2, label=mat)
        plt.xlabel("T (°C)")
        plt.ylabel(ylabel)
        plt.title(f"{quantity} vs {xaxis}")
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=10, title='Material', edgecolor='black', facecolor='lightgray')
        plt.show()
        
for widget in [
    temperature_material_dropdown, temperature_quantity_dropdown, temperature_x_dropdown,
    temperature_Lx_dropdown, temperature_Ly_dropdown, temperature_Lz_dropdown,
    temperature_epsilon_dropdown, temperature_V0e_dropdown, temperature_V0h_dropdown, temperature_T_dropdown,
    temperature_nxe_dropdown, temperature_nye_dropdown, temperature_nze_dropdown,
    temperature_nxh_dropdown, temperature_nyh_dropdown, temperature_nzh_dropdown
]:
    widget.observe(temperature_plot, names='value')
temperature_x_dropdown.observe(temperature_visibility, names='value')
display(VBox([
    HBox([temperature_material_dropdown, temperature_quantity_dropdown, temperature_x_dropdown]),
    HBox([temperature_Lx_dropdown, temperature_Ly_dropdown, temperature_Lz_dropdown]),
    HBox([temperature_epsilon_dropdown, temperature_V0e_dropdown, temperature_V0h_dropdown, temperature_T_dropdown]),
    HBox([temperature_nxe_dropdown, temperature_nye_dropdown, temperature_nze_dropdown]),
    HBox([temperature_nxh_dropdown, temperature_nyh_dropdown, temperature_nzh_dropdown]),
    temperature_output
]))
temperature_visibility()
temperature_plot()

### 3. Wavefunction and Probability Density Visualization

In [84]:
def psi_e(nx, ny, nz, x, y, z, Lx, Ly, Lz, V0e, material):
    delta = delta_e(material, V0e)
    Lx_eff = nm_to_m(Lx) + 2.0 * delta
    Ly_eff = nm_to_m(Ly) + 2.0 * delta
    Lz_eff = nm_to_m(Lz) + 2.0 * delta
    x_m = nm_to_m(x)
    y_m = nm_to_m(y)
    z_m = nm_to_m(z)
    return np.sqrt(8.0 / (Lx_eff * Ly_eff * Lz_eff)) * \
           np.sin(nx * np.pi * x_m / Lx_eff) * \
           np.sin(ny * np.pi * y_m / Ly_eff) * \
           np.sin(nz * np.pi * z_m / Lz_eff)

def psi_h(nx, ny, nz, x, y, z, Lx, Ly, Lz, V0h, material):
    delta = delta_h(material, V0h)
    Lx_eff = nm_to_m(Lx) + 2.0 * delta
    Ly_eff = nm_to_m(Ly) + 2.0 * delta
    Lz_eff = nm_to_m(Lz) + 2.0 * delta
    x_m = nm_to_m(x)
    y_m = nm_to_m(y)
    z_m = nm_to_m(z)
    return np.sqrt(8.0 / (Lx_eff * Ly_eff * Lz_eff)) * \
           np.sin(nx * np.pi * x_m / Lx_eff) * \
           np.sin(ny * np.pi * y_m / Ly_eff) * \
           np.sin(nz * np.pi * z_m / Lz_eff)

def psi_e_squared(nx, ny, nz, x, y, z, Lx, Ly, Lz, V0e, material):
    delta = delta_e(material, V0e)
    Lx_eff = nm_to_m(Lx) + 2.0 * delta
    Ly_eff = nm_to_m(Ly) + 2.0 * delta
    Lz_eff = nm_to_m(Lz) + 2.0 * delta
    x_m = nm_to_m(x)
    y_m = nm_to_m(y)
    z_m = nm_to_m(z)
    return (8.0 / (Lx_eff * Ly_eff * Lz_eff)) * \
           np.sin(nx * np.pi * x_m / Lx_eff)**2 * \
           np.sin(ny * np.pi * y_m / Ly_eff)**2 * \
           np.sin(nz * np.pi * z_m / Lz_eff)**2

def psi_h_squared(nx, ny, nz, x, y, z, Lx, Ly, Lz, V0h, material):
    delta = delta_h(material, V0h)
    Lx_eff = nm_to_m(Lx) + 2.0 * delta
    Ly_eff = nm_to_m(Ly) + 2.0 * delta
    Lz_eff = nm_to_m(Lz) + 2.0 * delta
    x_m = nm_to_m(x)
    y_m = nm_to_m(y)
    z_m = nm_to_m(z)
    return (8.0 / (Lx_eff * Ly_eff * Lz_eff)) * \
           np.sin(nx * np.pi * x_m / Lx_eff)**2 * \
           np.sin(ny * np.pi * y_m / Ly_eff)**2 * \
           np.sin(nz * np.pi * z_m / Lz_eff)**2

#### a. 3D Wavefunction 

In [85]:
wavefunction_material_dropdown = Dropdown(
    options=materials,
    value=materials[0],
    description='Material:'
)

wavefunction_particle_dropdown = Dropdown(
    options=["Ψₑ", "Ψₕ"],
    value="Ψₑ",
    description='Wavefunction:'
)

wavefunction_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='Lₓ (nm):')
wavefunction_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
wavefunction_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
wavefunction_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
wavefunction_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
wavefunction_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
wavefunction_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
wavefunction_nze_dropdown = Dropdown(options=n_vals, value=1, description='n𝓏ₑ:')
wavefunction_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
wavefunction_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
wavefunction_nzh_dropdown = Dropdown(options=n_vals, value=1, description='n𝓏ₕ:')
wavefunction_xmax_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='xₘₐₓ (nm):')
wavefunction_ymax_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='yₘₐₓ (nm):')
wavefunction_zmax_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='zₘₐₓ (nm):')
wavefunction_points_dropdown = Dropdown(options=[8, 10, 12, 15, 20], value=10, description='N:')
wavefunction_output = Output()

def wavefunction_visibility(*args):
    if wavefunction_particle_dropdown.value == "Ψₑ":
        wavefunction_V0e_dropdown.layout.display = ''
        wavefunction_V0h_dropdown.layout.display = 'none'
        wavefunction_nxe_dropdown.layout.display = ''
        wavefunction_nye_dropdown.layout.display = ''
        wavefunction_nze_dropdown.layout.display = ''
        wavefunction_nxh_dropdown.layout.display = 'none'
        wavefunction_nyh_dropdown.layout.display = 'none'
        wavefunction_nzh_dropdown.layout.display = 'none'
    else:
        wavefunction_V0e_dropdown.layout.display = 'none'
        wavefunction_V0h_dropdown.layout.display = ''
        wavefunction_nxe_dropdown.layout.display = 'none'
        wavefunction_nye_dropdown.layout.display = 'none'
        wavefunction_nze_dropdown.layout.display = 'none'
        wavefunction_nxh_dropdown.layout.display = ''
        wavefunction_nyh_dropdown.layout.display = ''
        wavefunction_nzh_dropdown.layout.display = ''

def wavefunction_plot(*args):
    with wavefunction_output:
        clear_output(wait=True)
        material = wavefunction_material_dropdown.value
        particle = wavefunction_particle_dropdown.value
        if particle == "Ψₑ":
            delta_nm = 1e9 * delta_e(material, wavefunction_V0e_dropdown.value)
        else:
            delta_nm = 1e9 * delta_h(material, wavefunction_V0h_dropdown.value)
        x = np.linspace(
            0.0,
            wavefunction_Lx_dropdown.value + 2.0 * delta_nm,
            wavefunction_points_dropdown.value
        )
        y = np.linspace(
            0.0,
            wavefunction_Ly_dropdown.value + 2.0 * delta_nm,
            wavefunction_points_dropdown.value
        )
        z = np.linspace(
            0.0,
            wavefunction_Lz_dropdown.value + 2.0 * delta_nm,
            wavefunction_points_dropdown.value
        )
        X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
        if particle == "Ψₑ":
            Psi = psi_e(
                wavefunction_nxe_dropdown.value, wavefunction_nye_dropdown.value, wavefunction_nze_dropdown.value,
                X, Y, Z,
                wavefunction_Lx_dropdown.value, wavefunction_Ly_dropdown.value, wavefunction_Lz_dropdown.value,
                wavefunction_V0e_dropdown.value,
                material
            )
            title_txt = f"Ψₑ(x, y, z) for {material}"
            cbar_label = "Ψₑ(x, y, z)"
        else:
            Psi = psi_h(
                wavefunction_nxh_dropdown.value, wavefunction_nyh_dropdown.value, wavefunction_nzh_dropdown.value,
                X, Y, Z,
                wavefunction_Lx_dropdown.value, wavefunction_Ly_dropdown.value, wavefunction_Lz_dropdown.value,
                wavefunction_V0h_dropdown.value,
                material
            )
            title_txt = f"Ψₕ(x, y, z) for {material}"
            cbar_label = "Ψₕ(x, y, z)"
        fig = plt.figure(figsize=(11, 8), constrained_layout=True)
        ax = fig.add_subplot(111, projection='3d')
        sc = ax.scatter(
            X.ravel(), Y.ravel(), Z.ravel(),
            c=Psi.ravel(),
            cmap='inferno',
            s=18,
            alpha=0.9
        )
        ax.set_xlabel('x (nm)', labelpad=10)
        ax.set_ylabel('y (nm)', labelpad=10)
        ax.set_zlabel('z (nm)', labelpad=10)
        ax.set_title(title_txt, pad=18)
        cbar = fig.colorbar(sc, ax=ax, pad=0.12, shrink=0.78)
        cbar.set_label(cbar_label)
        plt.show()

for widget in [
    wavefunction_material_dropdown, wavefunction_particle_dropdown,
    wavefunction_Lx_dropdown, wavefunction_Ly_dropdown, wavefunction_Lz_dropdown,
    wavefunction_V0e_dropdown, wavefunction_V0h_dropdown,
    wavefunction_nxe_dropdown, wavefunction_nye_dropdown, wavefunction_nze_dropdown,
    wavefunction_nxh_dropdown, wavefunction_nyh_dropdown, wavefunction_nzh_dropdown,
    wavefunction_xmax_dropdown, wavefunction_ymax_dropdown, wavefunction_zmax_dropdown,
    wavefunction_points_dropdown
]:
    widget.observe(wavefunction_plot, names='value')
wavefunction_particle_dropdown.observe(wavefunction_visibility, names='value')
display(VBox([
    HBox([wavefunction_material_dropdown, wavefunction_particle_dropdown]),
    HBox([wavefunction_Lx_dropdown, wavefunction_Ly_dropdown, wavefunction_Lz_dropdown]),
    HBox([wavefunction_V0e_dropdown, wavefunction_V0h_dropdown]),
    HBox([wavefunction_nxe_dropdown, wavefunction_nye_dropdown, wavefunction_nze_dropdown]),
    HBox([wavefunction_nxh_dropdown, wavefunction_nyh_dropdown, wavefunction_nzh_dropdown]),
    HBox([wavefunction_xmax_dropdown, wavefunction_ymax_dropdown, wavefunction_zmax_dropdown, wavefunction_points_dropdown]),
    wavefunction_output
]))
wavefunction_visibility()
wavefunction_plot()

#### b. 3D Probability Density 

In [86]:
probability_material_dropdown = Dropdown(
    options=materials,
    value=materials[0],
    description='Material:'
)

probability_particle_dropdown = Dropdown(
    options=["|Ψₑ|²", "|Ψₕ|²"],
    value="|Ψₑ|²",
    description='Density:'
)

probability_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='Lₓ (nm):')
probability_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
probability_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
probability_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
probability_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
probability_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
probability_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
probability_nze_dropdown = Dropdown(options=n_vals, value=1, description='n𝓏ₑ:')
probability_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
probability_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
probability_nzh_dropdown = Dropdown(options=n_vals, value=1, description='n𝓏ₕ:')
probability_xmax_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='xₘₐₓ (nm):')
probability_ymax_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='yₘₐₓ (nm):')
probability_zmax_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='zₘₐₓ (nm):')
probability_points_dropdown = Dropdown(options=[8, 10, 12, 15, 20], value=10, description='N:')
probability_output = Output()

def probability_visibility(*args):
    if probability_particle_dropdown.value == "|Ψₑ|²":
        probability_V0e_dropdown.layout.display = ''
        probability_V0h_dropdown.layout.display = 'none'
        probability_nxe_dropdown.layout.display = ''
        probability_nye_dropdown.layout.display = ''
        probability_nze_dropdown.layout.display = ''
        probability_nxh_dropdown.layout.display = 'none'
        probability_nyh_dropdown.layout.display = 'none'
        probability_nzh_dropdown.layout.display = 'none'
    else:
        probability_V0e_dropdown.layout.display = 'none'
        probability_V0h_dropdown.layout.display = ''
        probability_nxe_dropdown.layout.display = 'none'
        probability_nye_dropdown.layout.display = 'none'
        probability_nze_dropdown.layout.display = 'none'
        probability_nxh_dropdown.layout.display = ''
        probability_nyh_dropdown.layout.display = ''
        probability_nzh_dropdown.layout.display = ''

def probability_plot(*args):
    with probability_output:
        clear_output(wait=True)
        material = probability_material_dropdown.value
        particle = probability_particle_dropdown.value
        if particle == "|Ψₑ|²":
            delta_nm = 1e9 * delta_e(material, probability_V0e_dropdown.value)
        else:
            delta_nm = 1e9 * delta_h(material, probability_V0h_dropdown.value)
        x = np.linspace(
            0.0,
            probability_Lx_dropdown.value + 2.0 * delta_nm,
            probability_points_dropdown.value
        )
        y = np.linspace(
            0.0,
            probability_Ly_dropdown.value + 2.0 * delta_nm,
            probability_points_dropdown.value
        )
        z = np.linspace(
            0.0,
            probability_Lz_dropdown.value + 2.0 * delta_nm,
            probability_points_dropdown.value
        )
        X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
        if particle == "|Ψₑ|²":
            Psi2 = psi_e_squared(
                probability_nxe_dropdown.value, probability_nye_dropdown.value, probability_nze_dropdown.value,
                X, Y, Z,
                probability_Lx_dropdown.value, probability_Ly_dropdown.value, probability_Lz_dropdown.value,
                probability_V0e_dropdown.value,
                material
            )
            title_txt = f"|Ψₑ(x, y, z)|² for {material}"
            cbar_label = "|Ψₑ(x, y, z)|²"
        else:
            Psi2 = psi_h_squared(
                probability_nxh_dropdown.value, probability_nyh_dropdown.value, probability_nzh_dropdown.value,
                X, Y, Z,
                probability_Lx_dropdown.value, probability_Ly_dropdown.value, probability_Lz_dropdown.value,
                probability_V0h_dropdown.value,
                material
            )
            title_txt = f"|Ψₕ(x, y, z)|² for {material}"
            cbar_label = "|Ψₕ(x, y, z)|²"
        fig = plt.figure(figsize=(11, 8), constrained_layout=True)
        ax = fig.add_subplot(111, projection='3d')
        sc = ax.scatter(
            X.ravel(), Y.ravel(), Z.ravel(),
            c=Psi2.ravel(),
            cmap='inferno',
            s=18,
            alpha=0.9
        )
        ax.set_xlabel('x (nm)', labelpad=10)
        ax.set_ylabel('y (nm)', labelpad=10)
        ax.set_zlabel('z (nm)', labelpad=10)
        ax.set_title(title_txt, pad=18)
        cbar = fig.colorbar(sc, ax=ax, pad=0.12, shrink=0.78)
        cbar.set_label(cbar_label)
        plt.show()

for widget in [
    probability_material_dropdown, probability_particle_dropdown,
    probability_Lx_dropdown, probability_Ly_dropdown, probability_Lz_dropdown,
    probability_V0e_dropdown, probability_V0h_dropdown,
    probability_nxe_dropdown, probability_nye_dropdown, probability_nze_dropdown,
    probability_nxh_dropdown, probability_nyh_dropdown, probability_nzh_dropdown,
    probability_xmax_dropdown, probability_ymax_dropdown, probability_zmax_dropdown,
    probability_points_dropdown
]:
    widget.observe(probability_plot, names='value')
probability_particle_dropdown.observe(probability_visibility, names='value')
display(VBox([
    HBox([probability_material_dropdown, probability_particle_dropdown]),
    HBox([probability_Lx_dropdown, probability_Ly_dropdown, probability_Lz_dropdown]),
    HBox([probability_V0e_dropdown, probability_V0h_dropdown]),
    HBox([probability_nxe_dropdown, probability_nye_dropdown, probability_nze_dropdown]),
    HBox([probability_nxh_dropdown, probability_nyh_dropdown, probability_nzh_dropdown]),
    HBox([probability_xmax_dropdown, probability_ymax_dropdown, probability_zmax_dropdown, probability_points_dropdown]),
    probability_output
]))
probability_visibility()
probability_plot()

In [87]:
Leff_x_vals = np.array(sorted(set(np.round(np.concatenate([
    [
        Lx + 2.0e9 * delta_e(mat, V0e)
        for mat in materials
        for Lx in Lx_vals
        for V0e in V0e_vals
    ],
    [
        Lx + 2.0e9 * delta_h(mat, V0h)
        for mat in materials
        for Lx in Lx_vals
        for V0h in V0h_vals
    ]
]), 6))), dtype=float)

Leff_y_vals = np.array(sorted(set(np.round(np.concatenate([
    [
        Ly + 2.0e9 * delta_e(mat, V0e)
        for mat in materials
        for Ly in Ly_vals
        for V0e in V0e_vals
    ],
    [
        Ly + 2.0e9 * delta_h(mat, V0h)
        for mat in materials
        for Ly in Ly_vals
        for V0h in V0h_vals
    ]
]), 6))), dtype=float)

Leff_z_vals = np.array(sorted(set(np.round(np.concatenate([
    [
        Lz + 2.0e9 * delta_e(mat, V0e)
        for mat in materials
        for Lz in Lz_vals
        for V0e in V0e_vals
    ],
    [
        Lz + 2.0e9 * delta_h(mat, V0h)
        for mat in materials
        for Lz in Lz_vals
        for V0h in V0h_vals
    ]
]), 6))), dtype=float)

x0_vals = np.array(sorted(set(np.round(np.concatenate([
    Leff_x_vals / 4.0,
    Leff_x_vals / 2.0,
    3.0 * Leff_x_vals / 4.0
]), 6))), dtype=float)

y0_vals = np.array(sorted(set(np.round(np.concatenate([
    Leff_y_vals / 4.0,
    Leff_y_vals / 2.0,
    3.0 * Leff_y_vals / 4.0
]), 6))), dtype=float)

z0_vals = np.array(sorted(set(np.round(np.concatenate([
    Leff_z_vals / 4.0,
    Leff_z_vals / 2.0,
    3.0 * Leff_z_vals / 4.0
]), 6))), dtype=float)

#### c. Wavefunction 1D Slices

In [88]:
def psi_x_1D_slice(nx, ny, nz, x, y0, z0, Lx, Ly, Lz, V0, material, particle):
    if particle == "Ψₑ":
        return psi_e(nx, ny, nz, x, y0, z0, Lx, Ly, Lz, V0, material)
    else:
        return psi_h(nx, ny, nz, x, y0, z0, Lx, Ly, Lz, V0, material)

def psi_y_1D_slice(nx, ny, nz, x0, y, z0, Lx, Ly, Lz, V0, material, particle):
    if particle == "Ψₑ":
        return psi_e(nx, ny, nz, x0, y, z0, Lx, Ly, Lz, V0, material)
    else:
        return psi_h(nx, ny, nz, x0, y, z0, Lx, Ly, Lz, V0, material)

def psi_z_1D_slice(nx, ny, nz, x0, y0, z, Lx, Ly, Lz, V0, material, particle):
    if particle == "Ψₑ":
        return psi_e(nx, ny, nz, x0, y0, z, Lx, Ly, Lz, V0, material)
    else:
        return psi_h(nx, ny, nz, x0, y0, z, Lx, Ly, Lz, V0, material)

def add_state_label_and_extrema(ax, xs, ys, nx, ny, nz, particle="Ψₑ", color=None):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    if len(xs) < 3:
        return
    x_min, x_max = np.min(xs), np.max(xs)
    y_min, y_max = np.min(ys), np.max(ys)
    x_span = x_max - x_min
    y_span = y_max - y_min
    if y_span == 0:
        y_span = 1.0
    x_margin = 0.08 * x_span
    y_margin = 0.12 * y_span
    def clamp_inside(x_text, y_text):
        x_text = np.clip(x_text, x_min + x_margin, x_max - x_margin)
        y_text = np.clip(y_text, y_min + y_margin, y_max - y_margin)
        return x_text, y_text
    used_text_positions = []
    def shift_if_overlap(x_text, y_text):
        for _ in range(10):
            overlap = False
            for (ux, uy) in used_text_positions:
                if abs(x_text - ux) < 0.12 * x_span and abs(y_text - uy) < 0.12 * y_span:
                    y_text += 0.10 * y_span
                    x_text += 0.04 * x_span
                    x_text, y_text = clamp_inside(x_text, y_text)
                    overlap = True
                    break
            if not overlap:
                break
        used_text_positions.append((x_text, y_text))
        return x_text, y_text
    if particle == "Ψₑ":
        state_txt = f"(nₓₑ, nᵧₑ, nzₑ)=({nx}, {ny}, {nz})"
    else:
        state_txt = f"(nₓₕ, nᵧₕ, nzₕ)=({nx}, {ny}, {nz})"
    label_idx = int(0.78 * (len(xs) - 1))
    x_text = xs[label_idx] + 0.04 * x_span
    y_text = ys[label_idx] + 0.08 * y_span
    x_text, y_text = clamp_inside(x_text, y_text)
    x_text, y_text = shift_if_overlap(x_text, y_text)
    ax.annotate(
        state_txt,
        xy=(xs[label_idx], ys[label_idx]),
        xytext=(x_text, y_text),
        textcoords='data',
        fontsize=11,
        ha='left',
        va='bottom',
        color=color,
        bbox=dict(facecolor='white', edgecolor='black', alpha=0.75, pad=1.5),
        arrowprops=dict(arrowstyle='-', lw=1, color=color)
    )
    peak_idx, _ = find_peaks(ys)
    trough_idx, _ = find_peaks(-ys)
    def place_extrema_label(i, label_text, marker_style, prefer_up=True):
        xpt = xs[i]
        ypt = ys[i]
        ax.plot(xpt, ypt, marker=marker_style, markersize=6, color=color)
        x_text = xpt + 0.04 * x_span
        y_text = ypt + (0.10 * y_span if prefer_up else -0.10 * y_span)
        x_text, y_text = clamp_inside(x_text, y_text)
        x_text, y_text = shift_if_overlap(x_text, y_text)
        ax.annotate(
            label_text,
            xy=(xpt, ypt),
            xytext=(x_text, y_text),
            textcoords='data',
            fontsize=9,
            ha='left',
            va='bottom' if prefer_up else 'top',
            color=color,
            bbox=dict(facecolor='white', edgecolor='black', alpha=0.75, pad=1.0),
            arrowprops=dict(arrowstyle='-', lw=0.8, color=color)
        )
    for i in peak_idx:
        place_extrema_label(
            i,
            f"max\n({xs[i]:.2f}, {ys[i]:.2e})",
            'o',
            prefer_up=True
        )
    for i in trough_idx:
        place_extrema_label(
            i,
            f"min\n({xs[i]:.2f}, {ys[i]:.2e})",
            's',
            prefer_up=False
        )

def get_slice_curve_color(axis, nx, ny, nz):
    if axis == "x":
        color_map = {
            1: 'orangered',
            2: 'darkgreen',
            3: 'teal',
            4: 'magenta'
        }
        return color_map.get(nx, 'black')
    elif axis == "y":
        color_map = {
            1: 'darkred',
            2: 'gold',
            3: 'navy',
            4: 'indigo'
        }
        return color_map.get(ny, 'black')
    elif axis == "z":
        color_map = {
            1: 'crimson',
            2: 'brown',
            3: 'skyblue',
            4: 'darkviolet'
        }
        return color_map.get(nz, 'black')
    return 'black'

In [89]:
slice_material_dropdown = Dropdown(
    options=materials,
    value=materials[0],
    description='Material:'
)

slice_particle_dropdown = Dropdown(
    options=["Ψₑ", "Ψₕ"],
    value="Ψₑ",
    description='Wavefunction:'
)

slice_axis_dropdown = Dropdown(
    options=["x", "y", "z"],
    value="x",
    description='Slice:'
)

slice_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='Lₓ (nm):')
slice_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
slice_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
slice_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
slice_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
slice_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
slice_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
slice_nze_dropdown = Dropdown(options=n_vals, value=1, description='nzₑ:')
slice_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
slice_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
slice_nzh_dropdown = Dropdown(options=n_vals, value=1, description='nzₕ:')
slice_x0_dropdown = Dropdown(options=[float(v) for v in x0_vals], value=float(x0_vals[len(x0_vals)//2]), description='x₀ (nm):')
slice_y0_dropdown = Dropdown(options=[float(v) for v in y0_vals], value=float(y0_vals[len(y0_vals)//2]), description='y₀ (nm):')
slice_z0_dropdown = Dropdown(options=[float(v) for v in z0_vals], value=float(z0_vals[len(z0_vals)//2]), description='z₀ (nm):')
slice_points_dropdown = Dropdown(options=[100, 150, 200, 300, 400], value=300, description='N:')
slice_output = Output()

def slice_visibility(*args):
    if slice_particle_dropdown.value == "Ψₑ":
        slice_V0e_dropdown.layout.display = ''
        slice_V0h_dropdown.layout.display = 'none'
        slice_nxe_dropdown.layout.display = ''
        slice_nye_dropdown.layout.display = ''
        slice_nze_dropdown.layout.display = ''
        slice_nxh_dropdown.layout.display = 'none'
        slice_nyh_dropdown.layout.display = 'none'
        slice_nzh_dropdown.layout.display = 'none'
    else:
        slice_V0e_dropdown.layout.display = 'none'
        slice_V0h_dropdown.layout.display = ''
        slice_nxe_dropdown.layout.display = 'none'
        slice_nye_dropdown.layout.display = 'none'
        slice_nze_dropdown.layout.display = 'none'
        slice_nxh_dropdown.layout.display = ''
        slice_nyh_dropdown.layout.display = ''
        slice_nzh_dropdown.layout.display = ''
    slice_x0_dropdown.layout.display = ''
    slice_y0_dropdown.layout.display = ''
    slice_z0_dropdown.layout.display = ''

def slice_plot(*args):
    with slice_output:
        clear_output(wait=True)
        material = slice_material_dropdown.value
        particle = slice_particle_dropdown.value
        axis = slice_axis_dropdown.value
        if particle == "Ψₑ":
            nx = slice_nxe_dropdown.value
            ny = slice_nye_dropdown.value
            nz = slice_nze_dropdown.value
            V0 = slice_V0e_dropdown.value
            delta_nm = 1e9 * delta_e(material, V0)
        else:
            nx = slice_nxh_dropdown.value
            ny = slice_nyh_dropdown.value
            nz = slice_nzh_dropdown.value
            V0 = slice_V0h_dropdown.value
            delta_nm = 1e9 * delta_h(material, V0)
        Lx_eff_nm = slice_Lx_dropdown.value + 2.0 * delta_nm
        Ly_eff_nm = slice_Ly_dropdown.value + 2.0 * delta_nm
        Lz_eff_nm = slice_Lz_dropdown.value + 2.0 * delta_nm
        if axis == "x":
            xs = np.linspace(0.0, Lx_eff_nm, slice_points_dropdown.value)
            ys = psi_x_1D_slice(
                nx, ny, nz,
                xs,
                slice_y0_dropdown.value,
                slice_z0_dropdown.value,
                slice_Lx_dropdown.value, slice_Ly_dropdown.value, slice_Lz_dropdown.value,
                V0,
                material,
                particle
            )
            xlabel = "x (nm)"
            ylabel = f"{particle}(x, y₀, z₀)"
            title_txt = f"{particle}(x, y₀, z₀) for {material}"
        elif axis == "y":
            xs = np.linspace(0.0, Ly_eff_nm, slice_points_dropdown.value)
            ys = psi_y_1D_slice(
                nx, ny, nz,
                slice_x0_dropdown.value,
                xs,
                slice_z0_dropdown.value,
                slice_Lx_dropdown.value, slice_Ly_dropdown.value, slice_Lz_dropdown.value,
                V0,
                material,
                particle
            )
            xlabel = "y (nm)"
            ylabel = f"{particle}(x₀, y, z₀)"
            title_txt = f"{particle}(x₀, y, z₀) for {material}"
        else:
            xs = np.linspace(0.0, Lz_eff_nm, slice_points_dropdown.value)
            ys = psi_z_1D_slice(
                nx, ny, nz,
                slice_x0_dropdown.value,
                slice_y0_dropdown.value,
                xs,
                slice_Lx_dropdown.value, slice_Ly_dropdown.value, slice_Lz_dropdown.value,
                V0,
                material,
                particle
            )
            xlabel = "z (nm)"
            ylabel = f"{particle}(x₀, y₀, z)"
            title_txt = f"{particle}(x₀, y₀, z) for {material}"
        fig, ax = plt.subplots(figsize=(9, 6))
        curve_color = get_slice_curve_color(axis, nx, ny, nz)
        line, = ax.plot(xs, ys, linewidth=2, color=curve_color)
        add_state_label_and_extrema(
            ax,
            xs,
            ys,
            nx,
            ny,
            nz,
            particle=particle,
            color=curve_color
        )
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title_txt)
        ax.grid(True, alpha=0.3)
        plt.show()
        plt.close(fig)

for widget in [
    slice_material_dropdown, slice_particle_dropdown, slice_axis_dropdown,
    slice_Lx_dropdown, slice_Ly_dropdown, slice_Lz_dropdown,
    slice_V0e_dropdown, slice_V0h_dropdown,
    slice_nxe_dropdown, slice_nye_dropdown, slice_nze_dropdown,
    slice_nxh_dropdown, slice_nyh_dropdown, slice_nzh_dropdown,
    slice_x0_dropdown, slice_y0_dropdown, slice_z0_dropdown,
    slice_points_dropdown
]:
    widget.observe(slice_plot, names='value')
slice_particle_dropdown.observe(slice_visibility, names='value')
display(VBox([
    HBox([slice_material_dropdown, slice_particle_dropdown, slice_axis_dropdown]),
    HBox([slice_Lx_dropdown, slice_Ly_dropdown, slice_Lz_dropdown]),
    HBox([slice_V0e_dropdown, slice_V0h_dropdown]),
    HBox([slice_nxe_dropdown, slice_nye_dropdown, slice_nze_dropdown]),
    HBox([slice_nxh_dropdown, slice_nyh_dropdown, slice_nzh_dropdown]),
    HBox([slice_x0_dropdown, slice_y0_dropdown, slice_z0_dropdown, slice_points_dropdown]),
    slice_output
]))
slice_visibility()
slice_plot()

#### d. Probability Density 1D Slices

In [90]:
def psi_x_1D_density_slice(nx, ny, nz, x, y0, z0, Lx, Ly, Lz, V0, material, particle):
    if particle == "|Ψₑ|²":
        return psi_e_squared(nx, ny, nz, x, y0, z0, Lx, Ly, Lz, V0, material)
    else:
        return psi_h_squared(nx, ny, nz, x, y0, z0, Lx, Ly, Lz, V0, material)

def psi_y_1D_density_slice(nx, ny, nz, x0, y, z0, Lx, Ly, Lz, V0, material, particle):
    if particle == "|Ψₑ|²":
        return psi_e_squared(nx, ny, nz, x0, y, z0, Lx, Ly, Lz, V0, material)
    else:
        return psi_h_squared(nx, ny, nz, x0, y, z0, Lx, Ly, Lz, V0, material)

def psi_z_1D_density_slice(nx, ny, nz, x0, y0, z, Lx, Ly, Lz, V0, material, particle):
    if particle == "|Ψₑ|²":
        return psi_e_squared(nx, ny, nz, x0, y0, z, Lx, Ly, Lz, V0, material)
    else:
        return psi_h_squared(nx, ny, nz, x0, y0, z, Lx, Ly, Lz, V0, material)

def add_state_label_and_extrema_density(ax, xs, ys, nx, ny, nz, particle="|Ψₑ|²", color=None):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    if len(xs) < 3:
        return
    x_min, x_max = np.min(xs), np.max(xs)
    y_min, y_max = np.min(ys), np.max(ys)
    x_span = x_max - x_min
    y_span = y_max - y_min
    if y_span == 0:
        y_span = 1.0
    x_margin = 0.08 * x_span
    y_margin = 0.12 * y_span
    def clamp_inside(x_text, y_text):
        x_text = np.clip(x_text, x_min + x_margin, x_max - x_margin)
        y_text = np.clip(y_text, y_min + y_margin, y_max - y_margin)
        return x_text, y_text
    used_text_positions = []
    def shift_if_overlap(x_text, y_text):
        for _ in range(10):
            overlap = False
            for (ux, uy) in used_text_positions:
                if abs(x_text - ux) < 0.12 * x_span and abs(y_text - uy) < 0.12 * y_span:
                    y_text += 0.10 * y_span
                    x_text += 0.04 * x_span
                    x_text, y_text = clamp_inside(x_text, y_text)
                    overlap = True
                    break
            if not overlap:
                break
        used_text_positions.append((x_text, y_text))
        return x_text, y_text
    if particle == "|Ψₑ|²":
        state_txt = f"(nₓₑ, nᵧₑ, nzₑ)=({nx}, {ny}, {nz})"
    else:
        state_txt = f"(nₓₕ, nᵧₕ, nzₕ)=({nx}, {ny}, {nz})"
    label_idx = int(0.78 * (len(xs) - 1))
    x_text = xs[label_idx] + 0.04 * x_span
    y_text = ys[label_idx] + 0.08 * y_span
    x_text, y_text = clamp_inside(x_text, y_text)
    x_text, y_text = shift_if_overlap(x_text, y_text)
    ax.annotate(
        state_txt,
        xy=(xs[label_idx], ys[label_idx]),
        xytext=(x_text, y_text),
        textcoords='data',
        fontsize=11,
        ha='left',
        va='bottom',
        color=color,
        bbox=dict(facecolor='white', edgecolor='black', alpha=0.75, pad=1.5),
        arrowprops=dict(arrowstyle='-', lw=1, color=color)
    )
    peak_idx, _ = find_peaks(ys)
    trough_idx, _ = find_peaks(-ys)
    def place_extrema_label(i, label_text, marker_style, prefer_up=True):
        xpt = xs[i]
        ypt = ys[i]
        ax.plot(xpt, ypt, marker=marker_style, markersize=6, color=color)
        x_text = xpt + 0.04 * x_span
        y_text = ypt + (0.10 * y_span if prefer_up else -0.10 * y_span)
        x_text, y_text = clamp_inside(x_text, y_text)
        x_text, y_text = shift_if_overlap(x_text, y_text)
        ax.annotate(
            label_text,
            xy=(xpt, ypt),
            xytext=(x_text, y_text),
            textcoords='data',
            fontsize=9,
            ha='left',
            va='bottom' if prefer_up else 'top',
            color=color,
            bbox=dict(facecolor='white', edgecolor='black', alpha=0.75, pad=1.0),
            arrowprops=dict(arrowstyle='-', lw=0.8, color=color)
        )
    for i in peak_idx:
        place_extrema_label(
            i,
            f"max\n({xs[i]:.2f}, {ys[i]:.2e})",
            'o',
            prefer_up=True
        )
    for i in trough_idx:
        place_extrema_label(
            i,
            f"min\n({xs[i]:.2f}, {ys[i]:.2e})",
            's',
            prefer_up=False
        )

def get_density_slice_curve_color(axis, nx, ny, nz):
    if axis == "x":
        color_map = {
            1: 'orangered',
            2: 'darkgreen',
            3: 'teal',
            4: 'magenta'
        }
        return color_map.get(nx, 'black')
    elif axis == "y":
        color_map = {
            1: 'darkred',
            2: 'gold',
            3: 'navy',
            4: 'indigo'
        }
        return color_map.get(ny, 'black')
    elif axis == "z":
        color_map = {
            1: 'crimson',
            2: 'brown',
            3: 'skyblue',
            4: 'darkviolet'
        }
        return color_map.get(nz, 'black')
    return 'black'

In [91]:
density_slice_material_dropdown = Dropdown(
    options=materials,
    value=materials[0],
    description='Material:'
)

density_slice_particle_dropdown = Dropdown(
    options=["|Ψₑ|²", "|Ψₕ|²"],
    value="|Ψₑ|²",
    description='Density:'
)

density_slice_axis_dropdown = Dropdown(
    options=["x", "y", "z"],
    value="x",
    description='Slice:'
)

density_slice_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='Lₓ (nm):')
density_slice_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
density_slice_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
density_slice_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
density_slice_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
density_slice_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
density_slice_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
density_slice_nze_dropdown = Dropdown(options=n_vals, value=1, description='nzₑ:')
density_slice_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
density_slice_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
density_slice_nzh_dropdown = Dropdown(options=n_vals, value=1, description='nzₕ:')
density_slice_x0_dropdown = Dropdown(options=[float(v) for v in x0_vals], value=float(x0_vals[len(x0_vals)//2]), description='x₀ (nm):')
density_slice_y0_dropdown = Dropdown(options=[float(v) for v in y0_vals], value=float(y0_vals[len(y0_vals)//2]), description='y₀ (nm):')
density_slice_z0_dropdown = Dropdown(options=[float(v) for v in z0_vals], value=float(z0_vals[len(z0_vals)//2]), description='z₀ (nm):')
density_slice_points_dropdown = Dropdown(options=[100, 150, 200, 300, 400], value=300, description='N:')
density_slice_output = Output()

def density_slice_visibility(*args):
    if density_slice_particle_dropdown.value == "|Ψₑ|²":
        density_slice_V0e_dropdown.layout.display = ''
        density_slice_V0h_dropdown.layout.display = 'none'
        density_slice_nxe_dropdown.layout.display = ''
        density_slice_nye_dropdown.layout.display = ''
        density_slice_nze_dropdown.layout.display = ''
        density_slice_nxh_dropdown.layout.display = 'none'
        density_slice_nyh_dropdown.layout.display = 'none'
        density_slice_nzh_dropdown.layout.display = 'none'
    else:
        density_slice_V0e_dropdown.layout.display = 'none'
        density_slice_V0h_dropdown.layout.display = ''
        density_slice_nxe_dropdown.layout.display = 'none'
        density_slice_nye_dropdown.layout.display = 'none'
        density_slice_nze_dropdown.layout.display = 'none'
        density_slice_nxh_dropdown.layout.display = ''
        density_slice_nyh_dropdown.layout.display = ''
        density_slice_nzh_dropdown.layout.display = ''
    density_slice_x0_dropdown.layout.display = ''
    density_slice_y0_dropdown.layout.display = ''
    density_slice_z0_dropdown.layout.display = ''

def density_slice_plot(*args):
    with density_slice_output:
        clear_output(wait=True)
        material = density_slice_material_dropdown.value
        particle = density_slice_particle_dropdown.value
        axis = density_slice_axis_dropdown.value
        if particle == "|Ψₑ|²":
            nx = density_slice_nxe_dropdown.value
            ny = density_slice_nye_dropdown.value
            nz = density_slice_nze_dropdown.value
            V0 = density_slice_V0e_dropdown.value
            delta_nm = 1e9 * delta_e(material, V0)
        else:
            nx = density_slice_nxh_dropdown.value
            ny = density_slice_nyh_dropdown.value
            nz = density_slice_nzh_dropdown.value
            V0 = density_slice_V0h_dropdown.value
            delta_nm = 1e9 * delta_h(material, V0)
        Lx_eff_nm = density_slice_Lx_dropdown.value + 2.0 * delta_nm
        Ly_eff_nm = density_slice_Ly_dropdown.value + 2.0 * delta_nm
        Lz_eff_nm = density_slice_Lz_dropdown.value + 2.0 * delta_nm
        if axis == "x":
            xs = np.linspace(0.0, Lx_eff_nm, density_slice_points_dropdown.value)
            ys = psi_x_1D_density_slice(
                nx, ny, nz,
                xs,
                density_slice_y0_dropdown.value,
                density_slice_z0_dropdown.value,
                density_slice_Lx_dropdown.value, density_slice_Ly_dropdown.value, density_slice_Lz_dropdown.value,
                V0,
                material,
                particle
            )
            xlabel = "x (nm)"
            ylabel = f"{particle}(x, y₀, z₀)"
            title_txt = f"{particle}(x, y₀, z₀) for {material}"
        elif axis == "y":
            xs = np.linspace(0.0, Ly_eff_nm, density_slice_points_dropdown.value)
            ys = psi_y_1D_density_slice(
                nx, ny, nz,
                density_slice_x0_dropdown.value,
                xs,
                density_slice_z0_dropdown.value,
                density_slice_Lx_dropdown.value, density_slice_Ly_dropdown.value, density_slice_Lz_dropdown.value,
                V0,
                material,
                particle
            )
            xlabel = "y (nm)"
            ylabel = f"{particle}(x₀, y, z₀)"
            title_txt = f"{particle}(x₀, y, z₀) for {material}"
        else:
            xs = np.linspace(0.0, Lz_eff_nm, density_slice_points_dropdown.value)
            ys = psi_z_1D_density_slice(
                nx, ny, nz,
                density_slice_x0_dropdown.value,
                density_slice_y0_dropdown.value,
                xs,
                density_slice_Lx_dropdown.value, density_slice_Ly_dropdown.value, density_slice_Lz_dropdown.value,
                V0,
                material,
                particle
            )
            xlabel = "z (nm)"
            ylabel = f"{particle}(x₀, y₀, z)"
            title_txt = f"{particle}(x₀, y₀, z) for {material}"
        fig, ax = plt.subplots(figsize=(9, 6))
        curve_color = get_density_slice_curve_color(axis, nx, ny, nz)
        line, = ax.plot(xs, ys, linewidth=2, color=curve_color)
        add_state_label_and_extrema_density(
            ax,
            xs,
            ys,
            nx,
            ny,
            nz,
            particle=particle,
            color=curve_color
        )
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title_txt)
        ax.grid(True, alpha=0.3)
        plt.show()
        plt.close(fig)

for widget in [
    density_slice_material_dropdown, density_slice_particle_dropdown, density_slice_axis_dropdown,
    density_slice_Lx_dropdown, density_slice_Ly_dropdown, density_slice_Lz_dropdown,
    density_slice_V0e_dropdown, density_slice_V0h_dropdown,
    density_slice_nxe_dropdown, density_slice_nye_dropdown, density_slice_nze_dropdown,
    density_slice_nxh_dropdown, density_slice_nyh_dropdown, density_slice_nzh_dropdown,
    density_slice_x0_dropdown, density_slice_y0_dropdown, density_slice_z0_dropdown,
    density_slice_points_dropdown
]:
    widget.observe(density_slice_plot, names='value')
density_slice_particle_dropdown.observe(density_slice_visibility, names='value')
display(VBox([
    HBox([density_slice_material_dropdown, density_slice_particle_dropdown, density_slice_axis_dropdown]),
    HBox([density_slice_Lx_dropdown, density_slice_Ly_dropdown, density_slice_Lz_dropdown]),
    HBox([density_slice_V0e_dropdown, density_slice_V0h_dropdown]),
    HBox([density_slice_nxe_dropdown, density_slice_nye_dropdown, density_slice_nze_dropdown]),
    HBox([density_slice_nxh_dropdown, density_slice_nyh_dropdown, density_slice_nzh_dropdown]),
    HBox([density_slice_x0_dropdown, density_slice_y0_dropdown, density_slice_z0_dropdown, density_slice_points_dropdown]),
    density_slice_output
]))
density_slice_visibility()
density_slice_plot()

#### e. 2D Probability Density Slices 

In [92]:
density_2D_material_dropdown = Dropdown(
    options=materials,
    value=materials[0],
    description='Material:'
)

density_2D_particle_dropdown = Dropdown(
    options=["|Ψₑ|²", "|Ψₕ|²"],
    value="|Ψₑ|²",
    description='Density:'
)

density_2D_plane_dropdown = Dropdown(
    options=["xy", "xz", "yz"],
    value="xy",
    description='Plane:'
)

density_2D_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='Lₓ (nm):')
density_2D_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
density_2D_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
density_2D_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
density_2D_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
density_2D_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
density_2D_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
density_2D_nze_dropdown = Dropdown(options=n_vals, value=1, description='nzₑ:')
density_2D_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
density_2D_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
density_2D_nzh_dropdown = Dropdown(options=n_vals, value=1, description='nzₕ:')
density_2D_x0_dropdown = Dropdown(options=[float(v) for v in x0_vals], value=float(x0_vals[len(x0_vals)//2]), description='x₀ (nm):')
density_2D_y0_dropdown = Dropdown(options=[float(v) for v in y0_vals], value=float(y0_vals[len(y0_vals)//2]), description='y₀ (nm):')
density_2D_z0_dropdown = Dropdown(options=[float(v) for v in z0_vals], value=float(z0_vals[len(z0_vals)//2]), description='z₀ (nm):')
density_2D_points_dropdown = Dropdown(options=[50, 75, 100, 150, 200], value=100, description='N:')
density_2D_output = Output()

def density_2D_visibility(*args):
    if density_2D_particle_dropdown.value == "|Ψₑ|²":
        density_2D_V0e_dropdown.layout.display = ''
        density_2D_V0h_dropdown.layout.display = 'none'
        density_2D_nxe_dropdown.layout.display = ''
        density_2D_nye_dropdown.layout.display = ''
        density_2D_nze_dropdown.layout.display = ''
        density_2D_nxh_dropdown.layout.display = 'none'
        density_2D_nyh_dropdown.layout.display = 'none'
        density_2D_nzh_dropdown.layout.display = 'none'
    else:
        density_2D_V0e_dropdown.layout.display = 'none'
        density_2D_V0h_dropdown.layout.display = ''
        density_2D_nxe_dropdown.layout.display = 'none'
        density_2D_nye_dropdown.layout.display = 'none'
        density_2D_nze_dropdown.layout.display = 'none'
        density_2D_nxh_dropdown.layout.display = ''
        density_2D_nyh_dropdown.layout.display = ''
        density_2D_nzh_dropdown.layout.display = ''
    if density_2D_plane_dropdown.value == "xy":
        density_2D_x0_dropdown.layout.display = 'none'
        density_2D_y0_dropdown.layout.display = 'none'
        density_2D_z0_dropdown.layout.display = ''
    elif density_2D_plane_dropdown.value == "xz":
        density_2D_x0_dropdown.layout.display = 'none'
        density_2D_y0_dropdown.layout.display = ''
        density_2D_z0_dropdown.layout.display = 'none'
    else:
        density_2D_x0_dropdown.layout.display = ''
        density_2D_y0_dropdown.layout.display = 'none'
        density_2D_z0_dropdown.layout.display = 'none'

def density_2D_plot(*args):
    with density_2D_output:
        clear_output(wait=True)
        material = density_2D_material_dropdown.value
        particle = density_2D_particle_dropdown.value
        plane = density_2D_plane_dropdown.value
        if particle == "|Ψₑ|²":
            nx = density_2D_nxe_dropdown.value
            ny = density_2D_nye_dropdown.value
            nz = density_2D_nze_dropdown.value
            V0 = density_2D_V0e_dropdown.value
            delta_nm = 1e9 * delta_e(material, V0)
        else:
            nx = density_2D_nxh_dropdown.value
            ny = density_2D_nyh_dropdown.value
            nz = density_2D_nzh_dropdown.value
            V0 = density_2D_V0h_dropdown.value
            delta_nm = 1e9 * delta_h(material, V0)
        Lx_eff_nm = density_2D_Lx_dropdown.value + 2.0 * delta_nm
        Ly_eff_nm = density_2D_Ly_dropdown.value + 2.0 * delta_nm
        Lz_eff_nm = density_2D_Lz_dropdown.value + 2.0 * delta_nm
        if plane == "xy":
            x = np.linspace(0.0, Lx_eff_nm, density_2D_points_dropdown.value)
            y = np.linspace(0.0, Ly_eff_nm, density_2D_points_dropdown.value)
            X, Y = np.meshgrid(x, y, indexing='ij')
            Z = np.full_like(X, density_2D_z0_dropdown.value)
            if particle == "|Ψₑ|²":
                D = psi_e_squared(
                    nx, ny, nz,
                    X, Y, Z,
                    density_2D_Lx_dropdown.value, density_2D_Ly_dropdown.value, density_2D_Lz_dropdown.value,
                    V0,
                    material
                )
            else:
                D = psi_h_squared(
                    nx, ny, nz,
                    X, Y, Z,
                    density_2D_Lx_dropdown.value, density_2D_Ly_dropdown.value, density_2D_Lz_dropdown.value,
                    V0,
                    material
                )
            xlabel = "x (nm)"
            ylabel = "y (nm)"
            title_txt = f"{particle}(x, y, z₀) for {material}"
            cbar_label = f"{particle}(x, y, z₀)"
            A = X
            B = Y
        elif plane == "xz":
            x = np.linspace(0.0, Lx_eff_nm, density_2D_points_dropdown.value)
            z = np.linspace(0.0, Lz_eff_nm, density_2D_points_dropdown.value)
            X, Z = np.meshgrid(x, z, indexing='ij')
            Y = np.full_like(X, density_2D_y0_dropdown.value)
            if particle == "|Ψₑ|²":
                D = psi_e_squared(
                    nx, ny, nz,
                    X, Y, Z,
                    density_2D_Lx_dropdown.value, density_2D_Ly_dropdown.value, density_2D_Lz_dropdown.value,
                    V0,
                    material
                )
            else:
                D = psi_h_squared(
                    nx, ny, nz,
                    X, Y, Z,
                    density_2D_Lx_dropdown.value, density_2D_Ly_dropdown.value, density_2D_Lz_dropdown.value,
                    V0,
                    material
                )
            xlabel = "x (nm)"
            ylabel = "z (nm)"
            title_txt = f"{particle}(x, y₀, z) for {material}"
            cbar_label = f"{particle}(x, y₀, z)"
            A = X
            B = Z
        else:
            y = np.linspace(0.0, Ly_eff_nm, density_2D_points_dropdown.value)
            z = np.linspace(0.0, Lz_eff_nm, density_2D_points_dropdown.value)
            Y, Z = np.meshgrid(y, z, indexing='ij')
            X = np.full_like(Y, density_2D_x0_dropdown.value)
            if particle == "|Ψₑ|²":
                D = psi_e_squared(
                    nx, ny, nz,
                    X, Y, Z,
                    density_2D_Lx_dropdown.value, density_2D_Ly_dropdown.value, density_2D_Lz_dropdown.value,
                    V0,
                    material
                )
            else:
                D = psi_h_squared(
                    nx, ny, nz,
                    X, Y, Z,
                    density_2D_Lx_dropdown.value, density_2D_Ly_dropdown.value, density_2D_Lz_dropdown.value,
                    V0,
                    material
                )
            xlabel = "y (nm)"
            ylabel = "z (nm)"
            title_txt = f"{particle}(x₀, y, z) for {material}"
            cbar_label = f"{particle}(x₀, y, z)"
            A = Y
            B = Z
        fig, ax = plt.subplots(figsize=(9, 6))
        pcm = ax.contourf(A, B, D, levels=50, cmap='inferno')
        cbar = fig.colorbar(pcm, ax=ax, pad=0.08)
        cbar.set_label(cbar_label)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title_txt)
        plt.show()
        plt.close(fig)

for widget in [
    density_2D_material_dropdown, density_2D_particle_dropdown, density_2D_plane_dropdown,
    density_2D_Lx_dropdown, density_2D_Ly_dropdown, density_2D_Lz_dropdown,
    density_2D_V0e_dropdown, density_2D_V0h_dropdown,
    density_2D_nxe_dropdown, density_2D_nye_dropdown, density_2D_nze_dropdown,
    density_2D_nxh_dropdown, density_2D_nyh_dropdown, density_2D_nzh_dropdown,
    density_2D_x0_dropdown, density_2D_y0_dropdown, density_2D_z0_dropdown,
    density_2D_points_dropdown
]:
    widget.observe(density_2D_plot, names='value')
density_2D_particle_dropdown.observe(density_2D_visibility, names='value')
density_2D_plane_dropdown.observe(density_2D_visibility, names='value')
display(VBox([
    HBox([density_2D_material_dropdown, density_2D_particle_dropdown, density_2D_plane_dropdown]),
    HBox([density_2D_Lx_dropdown, density_2D_Ly_dropdown, density_2D_Lz_dropdown]),
    HBox([density_2D_V0e_dropdown, density_2D_V0h_dropdown]),
    HBox([density_2D_nxe_dropdown, density_2D_nye_dropdown, density_2D_nze_dropdown]),
    HBox([density_2D_nxh_dropdown, density_2D_nyh_dropdown, density_2D_nzh_dropdown]),
    HBox([density_2D_x0_dropdown, density_2D_y0_dropdown, density_2D_z0_dropdown, density_2D_points_dropdown]),
    density_2D_output
]))
density_2D_visibility()
density_2D_plot()

#### f. 2D Electron-Hole Product Probability Density Slices

In [93]:
density_product_material_dropdown = Dropdown(
    options=materials,
    value=materials[0],
    description='Material:'
)

density_product_plane_dropdown = Dropdown(
    options=["xy", "xz", "yz"],
    value="xy",
    description='Plane:'
)

density_product_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='Lₓ (nm):')
density_product_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
density_product_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
density_product_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
density_product_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
density_product_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
density_product_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
density_product_nze_dropdown = Dropdown(options=n_vals, value=1, description='nzₑ:')
density_product_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
density_product_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
density_product_nzh_dropdown = Dropdown(options=n_vals, value=1, description='nzₕ:')
density_product_x0_dropdown = Dropdown(options=[float(v) for v in x0_vals], value=float(x0_vals[len(x0_vals)//2]), description='x₀ (nm):')
density_product_y0_dropdown = Dropdown(options=[float(v) for v in y0_vals], value=float(y0_vals[len(y0_vals)//2]), description='y₀ (nm):')
density_product_z0_dropdown = Dropdown(options=[float(v) for v in z0_vals], value=float(z0_vals[len(z0_vals)//2]), description='z₀ (nm):')
density_product_points_dropdown = Dropdown(options=[50, 75, 100, 150, 200], value=100, description='N:')
density_product_output = Output()

def density_product_visibility(*args):
    if density_product_plane_dropdown.value == "xy":
        density_product_x0_dropdown.layout.display = 'none'
        density_product_y0_dropdown.layout.display = 'none'
        density_product_z0_dropdown.layout.display = ''
    elif density_product_plane_dropdown.value == "xz":
        density_product_x0_dropdown.layout.display = 'none'
        density_product_y0_dropdown.layout.display = ''
        density_product_z0_dropdown.layout.display = 'none'
    else:
        density_product_x0_dropdown.layout.display = ''
        density_product_y0_dropdown.layout.display = 'none'
        density_product_z0_dropdown.layout.display = 'none'

def density_product_plot(*args):
    with density_product_output:
        clear_output(wait=True)
        material = density_product_material_dropdown.value
        plane = density_product_plane_dropdown.value
        deltae_nm = 1e9 * delta_e(material, density_product_V0e_dropdown.value)
        deltah_nm = 1e9 * delta_h(material, density_product_V0h_dropdown.value)
        Lx_eff_nm = max(density_product_Lx_dropdown.value + 2.0 * deltae_nm, density_product_Lx_dropdown.value + 2.0 * deltah_nm)
        Ly_eff_nm = max(density_product_Ly_dropdown.value + 2.0 * deltae_nm, density_product_Ly_dropdown.value + 2.0 * deltah_nm)
        Lz_eff_nm = max(density_product_Lz_dropdown.value + 2.0 * deltae_nm, density_product_Lz_dropdown.value + 2.0 * deltah_nm)
        if plane == "xy":
            x = np.linspace(0.0, Lx_eff_nm, density_product_points_dropdown.value)
            y = np.linspace(0.0, Ly_eff_nm, density_product_points_dropdown.value)
            X, Y = np.meshgrid(x, y, indexing='ij')
            Z = np.full_like(X, density_product_z0_dropdown.value)
            De = psi_e_squared(
                density_product_nxe_dropdown.value, density_product_nye_dropdown.value, density_product_nze_dropdown.value,
                X, Y, Z,
                density_product_Lx_dropdown.value, density_product_Ly_dropdown.value, density_product_Lz_dropdown.value,
                density_product_V0e_dropdown.value,
                material
            )
            Dh = psi_h_squared(
                density_product_nxh_dropdown.value, density_product_nyh_dropdown.value, density_product_nzh_dropdown.value,
                X, Y, Z,
                density_product_Lx_dropdown.value, density_product_Ly_dropdown.value, density_product_Lz_dropdown.value,
                density_product_V0h_dropdown.value,
                material
            )
            xlabel = "x (nm)"
            ylabel = "y (nm)"
            title_txt = f"|Ψₑ|²|Ψₕ|²(x, y, z₀) for {material}"
            cbar_label = "|Ψₑ|²|Ψₕ|²"
            A = X
            B = Y
        elif plane == "xz":
            x = np.linspace(0.0, Lx_eff_nm, density_product_points_dropdown.value)
            z = np.linspace(0.0, Lz_eff_nm, density_product_points_dropdown.value)
            X, Z = np.meshgrid(x, z, indexing='ij')
            Y = np.full_like(X, density_product_y0_dropdown.value)
            De = psi_e_squared(
                density_product_nxe_dropdown.value, density_product_nye_dropdown.value, density_product_nze_dropdown.value,
                X, Y, Z,
                density_product_Lx_dropdown.value, density_product_Ly_dropdown.value, density_product_Lz_dropdown.value,
                density_product_V0e_dropdown.value,
                material
            )
            Dh = psi_h_squared(
                density_product_nxh_dropdown.value, density_product_nyh_dropdown.value, density_product_nzh_dropdown.value,
                X, Y, Z,
                density_product_Lx_dropdown.value, density_product_Ly_dropdown.value, density_product_Lz_dropdown.value,
                density_product_V0h_dropdown.value,
                material
            )
            xlabel = "x (nm)"
            ylabel = "z (nm)"
            title_txt = f"|Ψₑ|²|Ψₕ|²(x, y₀, z) for {material}"
            cbar_label = "|Ψₑ|²|Ψₕ|²"
            A = X
            B = Z
        else:
            y = np.linspace(0.0, Ly_eff_nm, density_product_points_dropdown.value)
            z = np.linspace(0.0, Lz_eff_nm, density_product_points_dropdown.value)
            Y, Z = np.meshgrid(y, z, indexing='ij')
            X = np.full_like(Y, density_product_x0_dropdown.value)
            De = psi_e_squared(
                density_product_nxe_dropdown.value, density_product_nye_dropdown.value, density_product_nze_dropdown.value,
                X, Y, Z,
                density_product_Lx_dropdown.value, density_product_Ly_dropdown.value, density_product_Lz_dropdown.value,
                density_product_V0e_dropdown.value,
                material
            )
            Dh = psi_h_squared(
                density_product_nxh_dropdown.value, density_product_nyh_dropdown.value, density_product_nzh_dropdown.value,
                X, Y, Z,
                density_product_Lx_dropdown.value, density_product_Ly_dropdown.value, density_product_Lz_dropdown.value,
                density_product_V0h_dropdown.value,
                material
            )
            xlabel = "y (nm)"
            ylabel = "z (nm)"
            title_txt = f"|Ψₑ|²|Ψₕ|²(x₀, y, z) for {material}"
            cbar_label = "|Ψₑ|²|Ψₕ|²"
            A = Y
            B = Z
        D = De * Dh
        fig, ax = plt.subplots(figsize=(9, 6))
        pcm = ax.contourf(A, B, D, levels=50, cmap='inferno')
        cbar = fig.colorbar(pcm, ax=ax, pad=0.08)
        cbar.set_label(cbar_label)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title_txt)
        plt.show()
        plt.close(fig)

for widget in [
    density_product_material_dropdown, density_product_plane_dropdown,
    density_product_Lx_dropdown, density_product_Ly_dropdown, density_product_Lz_dropdown,
    density_product_V0e_dropdown, density_product_V0h_dropdown,
    density_product_nxe_dropdown, density_product_nye_dropdown, density_product_nze_dropdown,
    density_product_nxh_dropdown, density_product_nyh_dropdown, density_product_nzh_dropdown,
    density_product_x0_dropdown, density_product_y0_dropdown, density_product_z0_dropdown,
    density_product_points_dropdown
]:
    widget.observe(density_product_plot, names='value')
density_product_plane_dropdown.observe(density_product_visibility, names='value')
display(VBox([
    HBox([density_product_material_dropdown, density_product_plane_dropdown]),
    HBox([density_product_Lx_dropdown, density_product_Ly_dropdown, density_product_Lz_dropdown]),
    HBox([density_product_V0e_dropdown, density_product_V0h_dropdown]),
    HBox([density_product_nxe_dropdown, density_product_nye_dropdown, density_product_nze_dropdown]),
    HBox([density_product_nxh_dropdown, density_product_nyh_dropdown, density_product_nzh_dropdown]),
    HBox([density_product_x0_dropdown, density_product_y0_dropdown, density_product_z0_dropdown, density_product_points_dropdown]),
    density_product_output
]))
density_product_visibility()
density_product_plot()

### 3. Overlap Study 

$$
\Psi_e(x,y,z)
=
\sqrt{
\frac{8}{
\left(L_x+\frac{2\hbar}{\sqrt{2\mu_eV_{0e}}}\right)
\left(L_y+\frac{2\hbar}{\sqrt{2\mu_eV_{0e}}}\right)
\left(L_z+\frac{2\hbar}{\sqrt{2\mu_eV_{0e}}}\right)
}
}
\sin\!\left(
\frac{n_x^{(e)}\pi x}{
L_x+\frac{2\hbar}{\sqrt{2\mu_eV_{0e}}}
}
\right)
\sin\!\left(
\frac{n_y^{(e)}\pi y}{
L_y+\frac{2\hbar}{\sqrt{2\mu_eV_{0e}}}
}
\right)
\sin\!\left(
\frac{n_z^{(e)}\pi z}{
L_z+\frac{2\hbar}{\sqrt{2\mu_eV_{0e}}}
}
\right)
$$

$$
\Psi_h(x,y,z)
=
\sqrt{
\frac{8}{
\left(L_x+\frac{2\hbar}{\sqrt{2\mu_hV_{0h}}}\right)
\left(L_y+\frac{2\hbar}{\sqrt{2\mu_hV_{0h}}}\right)
\left(L_z+\frac{2\hbar}{\sqrt{2\mu_hV_{0h}}}\right)
}
}
\sin\!\left(
\frac{n_x^{(h)}\pi x}{
L_x+\frac{2\hbar}{\sqrt{2\mu_hV_{0h}}}
}
\right)
\sin\!\left(
\frac{n_y^{(h)}\pi y}{
L_y+\frac{2\hbar}{\sqrt{2\mu_hV_{0h}}}
}
\right)
\sin\!\left(
\frac{n_z^{(h)}\pi z}{
L_z+\frac{2\hbar}{\sqrt{2\mu_hV_{0h}}}
}
\right)
$$

$$
L_{xe}
=
L_x+\frac{2\hbar}{\sqrt{2\mu_eV_{0e}}},
\qquad
L_{ye}
=
L_y+\frac{2\hbar}{\sqrt{2\mu_eV_{0e}}},
\qquad
L_{ze}
=
L_z+\frac{2\hbar}{\sqrt{2\mu_eV_{0e}}}
$$

$$
L_{xh}
=
L_x+\frac{2\hbar}{\sqrt{2\mu_hV_{0h}}},
\qquad
L_{yh}
=
L_y+\frac{2\hbar}{\sqrt{2\mu_hV_{0h}}},
\qquad
L_{zh}
=
L_z+\frac{2\hbar}{\sqrt{2\mu_hV_{0h}}}
$$

$$
L_x^{(c)}=\min(L_{xe},L_{xh}),
\qquad
L_y^{(c)}=\min(L_{ye},L_{yh}),
\qquad
L_z^{(c)}=\min(L_{ze},L_{zh})
$$

$$
N_e
=
\sqrt{\frac{8}{L_{xe}L_{ye}L_{ze}}}
$$

$$
N_h
=
\sqrt{\frac{8}{L_{xh}L_{yh}L_{zh}}}
$$

$$
\Psi_e(x,y,z)
=
N_e
\sin\!\left(\frac{n_x^{(e)}\pi x}{L_{xe}}\right)
\sin\!\left(\frac{n_y^{(e)}\pi y}{L_{ye}}\right)
\sin\!\left(\frac{n_z^{(e)}\pi z}{L_{ze}}\right)
$$

$$
\Psi_h(x,y,z)
=
N_h
\sin\!\left(\frac{n_x^{(h)}\pi x}{L_{xh}}\right)
\sin\!\left(\frac{n_y^{(h)}\pi y}{L_{yh}}\right)
\sin\!\left(\frac{n_z^{(h)}\pi z}{L_{zh}}\right)
$$

$$
\mathcal{O}
=
\iiint
\Psi_e(x,y,z)\Psi_h(x,y,z)\,dx\,dy\,dz
$$

$$
\mathcal{O}
=
\int_0^{L_x^{(c)}}
\int_0^{L_y^{(c)}}
\int_0^{L_z^{(c)}}
\Psi_e(x,y,z)\Psi_h(x,y,z)\,dz\,dy\,dx
$$

$$
\mathcal{O}
=
\int_0^{L_x^{(c)}}
\int_0^{L_y^{(c)}}
\int_0^{L_z^{(c)}}
N_eN_h
\sin\!\left(\frac{n_x^{(e)}\pi x}{L_{xe}}\right)
\sin\!\left(\frac{n_y^{(e)}\pi y}{L_{ye}}\right)
\sin\!\left(\frac{n_z^{(e)}\pi z}{L_{ze}}\right)
\sin\!\left(\frac{n_x^{(h)}\pi x}{L_{xh}}\right)
\sin\!\left(\frac{n_y^{(h)}\pi y}{L_{yh}}\right)
\sin\!\left(\frac{n_z^{(h)}\pi z}{L_{zh}}\right)
\,dz\,dy\,dx
$$

$$
\mathcal{O}
=
N_eN_h
\int_0^{L_x^{(c)}}
\sin\!\left(\frac{n_x^{(e)}\pi x}{L_{xe}}\right)
\sin\!\left(\frac{n_x^{(h)}\pi x}{L_{xh}}\right)\,dx
\int_0^{L_y^{(c)}}
\sin\!\left(\frac{n_y^{(e)}\pi y}{L_{ye}}\right)
\sin\!\left(\frac{n_y^{(h)}\pi y}{L_{yh}}\right)\,dy
\int_0^{L_z^{(c)}}
\sin\!\left(\frac{n_z^{(e)}\pi z}{L_{ze}}\right)
\sin\!\left(\frac{n_z^{(h)}\pi z}{L_{zh}}\right)\,dz
$$

$$
\mathcal{O}
=
N_eN_h\,I_xI_yI_z
$$

$$
I_x
=
\int_0^{L_x^{(c)}}
\sin\!\left(\frac{n_x^{(e)}\pi x}{L_{xe}}\right)
\sin\!\left(\frac{n_x^{(h)}\pi x}{L_{xh}}\right)\,dx
$$

$$
A_x
=
\frac{n_x^{(e)}\pi x}{L_{xe}},
\qquad
B_x
=
\frac{n_x^{(h)}\pi x}{L_{xh}}
$$

$$
I_x
=
\int_0^{L_x^{(c)}} \sin(A_x)\sin(B_x)\,dx
$$

$$
\sin(A_x)\sin(B_x)
=
\frac{1}{2}\left[\cos(A_x-B_x)-\cos(A_x+B_x)\right]
$$

$$
I_x
=
\frac{1}{2}
\int_0^{L_x^{(c)}}
\left[
\cos\!\left(
\frac{n_x^{(e)}\pi x}{L_{xe}}
-
\frac{n_x^{(h)}\pi x}{L_{xh}}
\right)
-
\cos\!\left(
\frac{n_x^{(e)}\pi x}{L_{xe}}
+
\frac{n_x^{(h)}\pi x}{L_{xh}}
\right)
\right]dx
$$

$$
I_x
=
\frac{1}{2}
\int_0^{L_x^{(c)}}
\left[
\cos\!\left(
\left(
\frac{n_x^{(e)}\pi}{L_{xe}}
-
\frac{n_x^{(h)}\pi}{L_{xh}}
\right)x
\right)
-
\cos\!\left(
\left(
\frac{n_x^{(e)}\pi}{L_{xe}}
+
\frac{n_x^{(h)}\pi}{L_{xh}}
\right)x
\right)
\right]dx
$$

$$
\alpha_x
=
\frac{n_x^{(e)}\pi}{L_{xe}}
-
\frac{n_x^{(h)}\pi}{L_{xh}}
$$

$$
\beta_x
=
\frac{n_x^{(e)}\pi}{L_{xe}}
+
\frac{n_x^{(h)}\pi}{L_{xh}}
$$

$$
I_x
=
\frac{1}{2}
\int_0^{L_x^{(c)}}
\left[
\cos(\alpha_x x)-\cos(\beta_x x)
\right]dx
$$

$$
I_x
=
\frac{1}{2}
\left[
\int_0^{L_x^{(c)}} \cos(\alpha_x x)\,dx
-
\int_0^{L_x^{(c)}} \cos(\beta_x x)\,dx
\right]
$$

$$
\int \cos(\alpha_x x)\,dx
=
\frac{\sin(\alpha_x x)}{\alpha_x}
$$

$$
\int \cos(\beta_x x)\,dx
=
\frac{\sin(\beta_x x)}{\beta_x}
$$

$$
I_x
=
\frac{1}{2}
\left[
\frac{\sin(\alpha_x x)}{\alpha_x}
-
\frac{\sin(\beta_x x)}{\beta_x}
\right]_{0}^{L_x^{(c)}}
$$

$$
I_x
=
\frac{1}{2}
\left[
\left(
\frac{\sin(\alpha_x L_x^{(c)})}{\alpha_x}
-
\frac{\sin(\beta_x L_x^{(c)})}{\beta_x}
\right)
-
\left(
\frac{\sin(0)}{\alpha_x}
-
\frac{\sin(0)}{\beta_x}
\right)
\right]
$$

$$
\sin(0)=0
$$

$$
I_x
=
\frac{1}{2}
\left[
\frac{\sin(\alpha_x L_x^{(c)})}{\alpha_x}
-
\frac{\sin(\beta_x L_x^{(c)})}{\beta_x}
\right]
$$

$$
I_x
=
\frac{1}{2}
\left[
\frac{
\sin\!\left(
\left(
\frac{n_x^{(e)}\pi}{L_{xe}}
-
\frac{n_x^{(h)}\pi}{L_{xh}}
\right)L_x^{(c)}
\right)
}{
\frac{n_x^{(e)}\pi}{L_{xe}}
-
\frac{n_x^{(h)}\pi}{L_{xh}}
}
-
\frac{
\sin\!\left(
\left(
\frac{n_x^{(e)}\pi}{L_{xe}}
+
\frac{n_x^{(h)}\pi}{L_{xh}}
\right)L_x^{(c)}
\right)
}{
\frac{n_x^{(e)}\pi}{L_{xe}}
+
\frac{n_x^{(h)}\pi}{L_{xh}}
}
\right]
$$

$$
I_y
=
\int_0^{L_y^{(c)}}
\sin\!\left(\frac{n_y^{(e)}\pi y}{L_{ye}}\right)
\sin\!\left(\frac{n_y^{(h)}\pi y}{L_{yh}}\right)\,dy
$$

$$
A_y
=
\frac{n_y^{(e)}\pi y}{L_{ye}},
\qquad
B_y
=
\frac{n_y^{(h)}\pi y}{L_{yh}}
$$

$$
I_y
=
\int_0^{L_y^{(c)}} \sin(A_y)\sin(B_y)\,dy
$$

$$
\sin(A_y)\sin(B_y)
=
\frac{1}{2}\left[\cos(A_y-B_y)-\cos(A_y+B_y)\right]
$$

$$
I_y
=
\frac{1}{2}
\int_0^{L_y^{(c)}}
\left[
\cos\!\left(
\frac{n_y^{(e)}\pi y}{L_{ye}}
-
\frac{n_y^{(h)}\pi y}{L_{yh}}
\right)
-
\cos\!\left(
\frac{n_y^{(e)}\pi y}{L_{ye}}
+
\frac{n_y^{(h)}\pi y}{L_{yh}}
\right)
\right]dy
$$

$$
I_y
=
\frac{1}{2}
\int_0^{L_y^{(c)}}
\left[
\cos\!\left(
\left(
\frac{n_y^{(e)}\pi}{L_{ye}}
-
\frac{n_y^{(h)}\pi}{L_{yh}}
\right)y
\right)
-
\cos\!\left(
\left(
\frac{n_y^{(e)}\pi}{L_{ye}}
+
\frac{n_y^{(h)}\pi}{L_{yh}}
\right)y
\right)
\right]dy
$$

$$
\alpha_y
=
\frac{n_y^{(e)}\pi}{L_{ye}}
-
\frac{n_y^{(h)}\pi}{L_{yh}}
$$

$$
\beta_y
=
\frac{n_y^{(e)}\pi}{L_{ye}}
+
\frac{n_y^{(h)}\pi}{L_{yh}}
$$

$$
I_y
=
\frac{1}{2}
\int_0^{L_y^{(c)}}
\left[
\cos(\alpha_y y)-\cos(\beta_y y)
\right]dy
$$

$$
I_y
=
\frac{1}{2}
\left[
\int_0^{L_y^{(c)}} \cos(\alpha_y y)\,dy
-
\int_0^{L_y^{(c)}} \cos(\beta_y y)\,dy
\right]
$$

$$
\int \cos(\alpha_y y)\,dy
=
\frac{\sin(\alpha_y y)}{\alpha_y}
$$

$$
\int \cos(\beta_y y)\,dy
=
\frac{\sin(\beta_y y)}{\beta_y}
$$

$$
I_y
=
\frac{1}{2}
\left[
\frac{\sin(\alpha_y y)}{\alpha_y}
-
\frac{\sin(\beta_y y)}{\beta_y}
\right]_{0}^{L_y^{(c)}}
$$

$$
I_y
=
\frac{1}{2}
\left[
\left(
\frac{\sin(\alpha_y L_y^{(c)})}{\alpha_y}
-
\frac{\sin(\beta_y L_y^{(c)})}{\beta_y}
\right)
-
\left(
\frac{\sin(0)}{\alpha_y}
-
\frac{\sin(0)}{\beta_y}
\right)
\right]
$$

$$
\sin(0)=0
$$

$$
I_y
=
\frac{1}{2}
\left[
\frac{\sin(\alpha_y L_y^{(c)})}{\alpha_y}
-
\frac{\sin(\beta_y L_y^{(c)})}{\beta_y}
\right]
$$

$$
I_y
=
\frac{1}{2}
\left[
\frac{
\sin\!\left(
\left(
\frac{n_y^{(e)}\pi}{L_{ye}}
-
\frac{n_y^{(h)}\pi}{L_{yh}}
\right)L_y^{(c)}
\right)
}{
\frac{n_y^{(e)}\pi}{L_{ye}}
-
\frac{n_y^{(h)}\pi}{L_{yh}}
}
-
\frac{
\sin\!\left(
\left(
\frac{n_y^{(e)}\pi}{L_{ye}}
+
\frac{n_y^{(h)}\pi}{L_{yh}}
\right)L_y^{(c)}
\right)
}{
\frac{n_y^{(e)}\pi}{L_{ye}}
+
\frac{n_y^{(h)}\pi}{L_{yh}}
}
\right]
$$

$$
I_z
=
\int_0^{L_z^{(c)}}
\sin\!\left(\frac{n_z^{(e)}\pi z}{L_{ze}}\right)
\sin\!\left(\frac{n_z^{(h)}\pi z}{L_{zh}}\right)\,dz
$$

$$
A_z
=
\frac{n_z^{(e)}\pi z}{L_{ze}},
\qquad
B_z
=
\frac{n_z^{(h)}\pi z}{L_{zh}}
$$

$$
I_z
=
\int_0^{L_z^{(c)}} \sin(A_z)\sin(B_z)\,dz
$$

$$
\sin(A_z)\sin(B_z)
=
\frac{1}{2}\left[\cos(A_z-B_z)-\cos(A_z+B_z)\right]
$$

$$
I_z
=
\frac{1}{2}
\int_0^{L_z^{(c)}}
\left[
\cos\!\left(
\frac{n_z^{(e)}\pi z}{L_{ze}}
-
\frac{n_z^{(h)}\pi z}{L_{zh}}
\right)
-
\cos\!\left(
\frac{n_z^{(e)}\pi z}{L_{ze}}
+
\frac{n_z^{(h)}\pi z}{L_{zh}}
\right)
\right]dz
$$

$$
I_z
=
\frac{1}{2}
\int_0^{L_z^{(c)}}
\left[
\cos\!\left(
\left(
\frac{n_z^{(e)}\pi}{L_{ze}}
-
\frac{n_z^{(h)}\pi}{L_{zh}}
\right)z
\right)
-
\cos\!\left(
\left(
\frac{n_z^{(e)}\pi}{L_{ze}}
+
\frac{n_z^{(h)}\pi}{L_{zh}}
\right)z
\right)
\right]dz
$$

$$
\alpha_z
=
\frac{n_z^{(e)}\pi}{L_{ze}}
-
\frac{n_z^{(h)}\pi}{L_{zh}}
$$

$$
\beta_z
=
\frac{n_z^{(e)}\pi}{L_{ze}}
+
\frac{n_z^{(h)}\pi}{L_{zh}}
$$

$$
I_z
=
\frac{1}{2}
\int_0^{L_z^{(c)}}
\left[
\cos(\alpha_z z)-\cos(\beta_z z)
\right]dz
$$

$$
I_z
=
\frac{1}{2}
\left[
\int_0^{L_z^{(c)}} \cos(\alpha_z z)\,dz
-
\int_0^{L_z^{(c)}} \cos(\beta_z z)\,dz
\right]
$$

$$
\int \cos(\alpha_z z)\,dz
=
\frac{\sin(\alpha_z z)}{\alpha_z}
$$

$$
\int \cos(\beta_z z)\,dz
=
\frac{\sin(\beta_z z)}{\beta_z}
$$

$$
I_z
=
\frac{1}{2}
\left[
\frac{\sin(\alpha_z z)}{\alpha_z}
-
\frac{\sin(\beta_z z)}{\beta_z}
\right]_{0}^{L_z^{(c)}}
$$

$$
I_z
=
\frac{1}{2}
\left[
\left(
\frac{\sin(\alpha_z L_z^{(c)})}{\alpha_z}
-
\frac{\sin(\beta_z L_z^{(c)})}{\beta_z}
\right)
-
\left(
\frac{\sin(0)}{\alpha_z}
-
\frac{\sin(0)}{\beta_z}
\right)
\right]
$$

$$
\sin(0)=0
$$

$$
I_z
=
\frac{1}{2}
\left[
\frac{\sin(\alpha_z L_z^{(c)})}{\alpha_z}
-
\frac{\sin(\beta_z L_z^{(c)})}{\beta_z}
\right]
$$

$$
I_z
=
\frac{1}{2}
\left[
\frac{
\sin\!\left(
\left(
\frac{n_z^{(e)}\pi}{L_{ze}}
-
\frac{n_z^{(h)}\pi}{L_{zh}}
\right)L_z^{(c)}
\right)
}{
\frac{n_z^{(e)}\pi}{L_{ze}}
-
\frac{n_z^{(h)}\pi}{L_{zh}}
}
-
\frac{
\sin\!\left(
\left(
\frac{n_z^{(e)}\pi}{L_{ze}}
+
\frac{n_z^{(h)}\pi}{L_{zh}}
\right)L_z^{(c)}
\right)
}{
\frac{n_z^{(e)}\pi}{L_{ze}}
+
\frac{n_z^{(h)}\pi}{L_{zh}}
}
\right]
$$

$$
N_eN_h
=
\sqrt{\frac{8}{L_{xe}L_{ye}L_{ze}}}
\sqrt{\frac{8}{L_{xh}L_{yh}L_{zh}}}
$$

$$
N_eN_h
=
\frac{8}{\sqrt{L_{xe}L_{ye}L_{ze}L_{xh}L_{yh}L_{zh}}}
$$

$$
\mathcal{O}
=
\frac{8}{\sqrt{L_{xe}L_{ye}L_{ze}L_{xh}L_{yh}L_{zh}}}
\left(\frac{1}{2}\right)
\left(\frac{1}{2}\right)
\left(\frac{1}{2}\right)
\left[
\frac{
\sin\!\left(
\left(
\frac{n_x^{(e)}\pi}{L_{xe}}
-
\frac{n_x^{(h)}\pi}{L_{xh}}
\right)L_x^{(c)}
\right)
}{
\frac{n_x^{(e)}\pi}{L_{xe}}
-
\frac{n_x^{(h)}\pi}{L_{xh}}
}
-
\frac{
\sin\!\left(
\left(
\frac{n_x^{(e)}\pi}{L_{xe}}
+
\frac{n_x^{(h)}\pi}{L_{xh}}
\right)L_x^{(c)}
\right)
}{
\frac{n_x^{(e)}\pi}{L_{xe}}
+
\frac{n_x^{(h)}\pi}{L_{xh}}
}
\right]
$$

$$
\qquad\qquad\qquad\qquad\qquad\qquad
\times
\left[
\frac{
\sin\!\left(
\left(
\frac{n_y^{(e)}\pi}{L_{ye}}
-
\frac{n_y^{(h)}\pi}{L_{yh}}
\right)L_y^{(c)}
\right)
}{
\frac{n_y^{(e)}\pi}{L_{ye}}
-
\frac{n_y^{(h)}\pi}{L_{yh}}
}
-
\frac{
\sin\!\left(
\left(
\frac{n_y^{(e)}\pi}{L_{ye}}
+
\frac{n_y^{(h)}\pi}{L_{yh}}
\right)L_y^{(c)}
\right)
}{
\frac{n_y^{(e)}\pi}{L_{ye}}
+
\frac{n_y^{(h)}\pi}{L_{yh}}
}
\right]
$$

$$
\qquad\qquad\qquad\qquad\qquad\qquad
\times
\left[
\frac{
\sin\!\left(
\left(
\frac{n_z^{(e)}\pi}{L_{ze}}
-
\frac{n_z^{(h)}\pi}{L_{zh}}
\right)L_z^{(c)}
\right)
}{
\frac{n_z^{(e)}\pi}{L_{ze}}
-
\frac{n_z^{(h)}\pi}{L_{zh}}
}
-
\frac{
\sin\!\left(
\left(
\frac{n_z^{(e)}\pi}{L_{ze}}
+
\frac{n_z^{(h)}\pi}{L_{zh}}
\right)L_z^{(c)}
\right)
}{
\frac{n_z^{(e)}\pi}{L_{ze}}
+
\frac{n_z^{(h)}\pi}{L_{zh}}
}
\right]
$$

$$
\left(\frac{1}{2}\right)
\left(\frac{1}{2}\right)
\left(\frac{1}{2}\right)
=
\frac{1}{8}
$$

$$
\frac{8}{\sqrt{L_{xe}L_{ye}L_{ze}L_{xh}L_{yh}L_{zh}}}
\cdot
\frac{1}{8}
=
\frac{1}{\sqrt{L_{xe}L_{ye}L_{ze}L_{xh}L_{yh}L_{zh}}}
$$

$$
\boxed{
\begin{aligned}
\mathcal{O}
&=
\frac{1}{
\sqrt{L_{xe}L_{ye}L_{ze}L_{xh}L_{yh}L_{zh}}
}
\left[
\frac{
\sin\!\left(
\left(
\frac{n_x^{(e)}\pi}{L_{xe}}
-
\frac{n_x^{(h)}\pi}{L_{xh}}
\right)L_x^{(c)}
\right)
}{
\frac{n_x^{(e)}\pi}{L_{xe}}
-
\frac{n_x^{(h)}\pi}{L_{xh}}
}
-
\frac{
\sin\!\left(
\left(
\frac{n_x^{(e)}\pi}{L_{xe}}
+
\frac{n_x^{(h)}\pi}{L_{xh}}
\right)L_x^{(c)}
\right)
}{
\frac{n_x^{(e)}\pi}{L_{xe}}
+
\frac{n_x^{(h)}\pi}{L_{xh}}
}
\right]
\\[1.2em]
&\qquad\times
\left[
\frac{
\sin\!\left(
\left(
\frac{n_y^{(e)}\pi}{L_{ye}}
-
\frac{n_y^{(h)}\pi}{L_{yh}}
\right)L_y^{(c)}
\right)
}{
\frac{n_y^{(e)}\pi}{L_{ye}}
-
\frac{n_y^{(h)}\pi}{L_{yh}}
}
-
\frac{
\sin\!\left(
\left(
\frac{n_y^{(e)}\pi}{L_{ye}}
+
\frac{n_y^{(h)}\pi}{L_{yh}}
\right)L_y^{(c)}
\right)
}{
\frac{n_y^{(e)}\pi}{L_{ye}}
+
\frac{n_y^{(h)}\pi}{L_{yh}}
}
\right]
\\[1.2em]
&\qquad\times
\left[
\frac{
\sin\!\left(
\left(
\frac{n_z^{(e)}\pi}{L_{ze}}
-
\frac{n_z^{(h)}\pi}{L_{zh}}
\right)L_z^{(c)}
\right)
}{
\frac{n_z^{(e)}\pi}{L_{ze}}
-
\frac{n_z^{(h)}\pi}{L_{zh}}
}
-
\frac{
\sin\!\left(
\left(
\frac{n_z^{(e)}\pi}{L_{ze}}
+
\frac{n_z^{(h)}\pi}{L_{zh}}
\right)L_z^{(c)}
\right)
}{
\frac{n_z^{(e)}\pi}{L_{ze}}
+
\frac{n_z^{(h)}\pi}{L_{zh}}
}
\right]
\end{aligned}
}
$$

$$
\boxed{
\begin{aligned}
|{\mathcal O}|^2
&=
\frac{1}{
L_{xe}L_{ye}L_{ze}L_{xh}L_{yh}L_{zh}
}
\left[
\frac{
\sin\!\left(
\left(
\frac{n_x^{(e)}\pi}{L_{xe}}
-
\frac{n_x^{(h)}\pi}{L_{xh}}
\right)L_x^{(c)}
\right)
}{
\frac{n_x^{(e)}\pi}{L_{xe}}
-
\frac{n_x^{(h)}\pi}{L_{xh}}
}
-
\frac{
\sin\!\left(
\left(
\frac{n_x^{(e)}\pi}{L_{xe}}
+
\frac{n_x^{(h)}\pi}{L_{xh}}
\right)L_x^{(c)}
\right)
}{
\frac{n_x^{(e)}\pi}{L_{xe}}
+
\frac{n_x^{(h)}\pi}{L_{xh}}
}
\right]^2
\\[1.2em]
&\qquad\times
\left[
\frac{
\sin\!\left(
\left(
\frac{n_y^{(e)}\pi}{L_{ye}}
-
\frac{n_y^{(h)}\pi}{L_{yh}}
\right)L_y^{(c)}
\right)
}{
\frac{n_y^{(e)}\pi}{L_{ye}}
-
\frac{n_y^{(h)}\pi}{L_{yh}}
}
-
\frac{
\sin\!\left(
\left(
\frac{n_y^{(e)}\pi}{L_{ye}}
+
\frac{n_y^{(h)}\pi}{L_{yh}}
\right)L_y^{(c)}
\right)
}{
\frac{n_y^{(e)}\pi}{L_{ye}}
+
\frac{n_y^{(h)}\pi}{L_{yh}}
}
\right]^2
\\[1.2em]
&\qquad\times
\left[
\frac{
\sin\!\left(
\left(
\frac{n_z^{(e)}\pi}{L_{ze}}
-
\frac{n_z^{(h)}\pi}{L_{zh}}
\right)L_z^{(c)}
\right)
}{
\frac{n_z^{(e)}\pi}{L_{ze}}
-
\frac{n_z^{(h)}\pi}{L_{zh}}
}
-
\frac{
\sin\!\left(
\left(
\frac{n_z^{(e)}\pi}{L_{ze}}
+
\frac{n_z^{(h)}\pi}{L_{zh}}
\right)L_z^{(c)}
\right)
}{
\frac{n_z^{(e)}\pi}{L_{ze}}
+
\frac{n_z^{(h)}\pi}{L_{zh}}
}
\right]^2
\end{aligned}
}
$$

In [94]:
def overlap_from_parameters(
    Lx, Ly, Lz,
    deltae_nm, deltah_nm,
    nxe, nye, nze,
    nxh, nyh, nzh
):
    Lxe = Lx + 2.0 * deltae_nm
    Lye = Ly + 2.0 * deltae_nm
    Lze = Lz + 2.0 * deltae_nm
    Lxh = Lx + 2.0 * deltah_nm
    Lyh = Ly + 2.0 * deltah_nm
    Lzh = Lz + 2.0 * deltah_nm
    Lxc = min(Lxe, Lxh)
    Lyc = min(Lye, Lyh)
    Lzc = min(Lze, Lzh)
    def overlap_1D(ne, nh, Le, Lh, Lc):
        ae = ne * np.pi / Le
        ah = nh * np.pi / Lh
        denom_minus = ae - ah
        denom_plus = ae + ah
        term_minus = np.where(
            np.abs(denom_minus) < 1e-12,
            Lc,
            np.sin(denom_minus * Lc) / denom_minus
        )
        term_plus = np.sin(denom_plus * Lc) / denom_plus
        return term_minus - term_plus
    Ox = overlap_1D(nxe, nxh, Lxe, Lxh, Lxc)
    Oy = overlap_1D(nye, nyh, Lye, Lyh, Lyc)
    Oz = overlap_1D(nze, nzh, Lze, Lzh, Lzc)
    return (Ox * Oy * Oz) / np.sqrt(Lxe * Lye * Lze * Lxh * Lyh * Lzh)

overlap_material_dropdown = Dropdown(
    options=materials,
    value=materials[0],
    description='Material:'
)

overlap_quantity_dropdown = Dropdown(
    options=["O", "|O|²"],
    value="O",
    description='Quantity:'
)

overlap_x_dropdown = Dropdown(
    options=["Lₓ", "Lᵧ", "Lz", "δₑ", "δₕ", "V₀ₑ", "V₀ₕ", "nₓₑ", "nᵧₑ", "nzₑ", "nₓₕ", "nᵧₕ", "nzₕ"],
    value="Lₓ",
    description='x-axis:'
)

overlap_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[-1]), description='Lₓ,max (nm):')
overlap_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
overlap_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
overlap_deltae_dropdown = Dropdown(options=[float(v) for v in delta_e_vals], value=float(delta_e_vals[0]), description='δₑ (nm):')
overlap_deltah_dropdown = Dropdown(options=[float(v) for v in delta_h_vals], value=float(delta_h_vals[0]), description='δₕ (nm):')
overlap_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
overlap_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
overlap_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
overlap_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
overlap_nze_dropdown = Dropdown(options=n_vals, value=1, description='nzₑ:')
overlap_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
overlap_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
overlap_nzh_dropdown = Dropdown(options=n_vals, value=1, description='nzₕ:')
overlap_output = Output()

def overlap_visibility(*args):
    if overlap_x_dropdown.value == "Lₓ":
        overlap_Lx_dropdown.description = 'Lₓ,max (nm):'
    else:
        overlap_Lx_dropdown.description = 'Lₓ (nm):'
    if overlap_x_dropdown.value == "Lᵧ":
        overlap_Ly_dropdown.description = 'Lᵧ,max (nm):'
    else:
        overlap_Ly_dropdown.description = 'Lᵧ (nm):'
    if overlap_x_dropdown.value == "Lz":
        overlap_Lz_dropdown.description = 'Lz,max (nm):'
    else:
        overlap_Lz_dropdown.description = 'Lz (nm):'
    if overlap_x_dropdown.value == "δₑ":
        overlap_deltae_dropdown.description = 'δₑ,max (nm):'
    else:
        overlap_deltae_dropdown.description = 'δₑ (nm):'
    if overlap_x_dropdown.value == "δₕ":
        overlap_deltah_dropdown.description = 'δₕ,max (nm):'
    else:
        overlap_deltah_dropdown.description = 'δₕ (nm):'
    if overlap_x_dropdown.value == "V₀ₑ":
        overlap_V0e_dropdown.description = 'V₀ₑ,max (meV):'
    else:
        overlap_V0e_dropdown.description = 'V₀ₑ (meV):'
    if overlap_x_dropdown.value == "V₀ₕ":
        overlap_V0h_dropdown.description = 'V₀ₕ,max (meV):'
    else:
        overlap_V0h_dropdown.description = 'V₀ₕ (meV):'

def overlap_plot(*args):
    with overlap_output:
        clear_output(wait=True)
        quantity = overlap_quantity_dropdown.value
        xaxis = overlap_x_dropdown.value
        material = overlap_material_dropdown.value
        if xaxis == "Lₓ":
            xs = np.linspace(1.0, overlap_Lx_dropdown.value, 300)
        elif xaxis == "Lᵧ":
            xs = np.linspace(1.0, overlap_Ly_dropdown.value, 300)
        elif xaxis == "Lz":
            xs = np.linspace(0.5, overlap_Lz_dropdown.value, 300)
        elif xaxis == "δₑ":
            xs = np.linspace(0.01, overlap_deltae_dropdown.value, 300)
        elif xaxis == "δₕ":
            xs = np.linspace(0.01, overlap_deltah_dropdown.value, 300)
        elif xaxis == "V₀ₑ":
            xs = np.linspace(1.0, overlap_V0e_dropdown.value, 300)
        elif xaxis == "V₀ₕ":
            xs = np.linspace(1.0, overlap_V0h_dropdown.value, 300)
        else:
            xs = np.array(n_vals, dtype=float)
        ys = []
        for x in xs:
            Lx = x if xaxis == "Lₓ" else overlap_Lx_dropdown.value
            Ly = x if xaxis == "Lᵧ" else overlap_Ly_dropdown.value
            Lz = x if xaxis == "Lz" else overlap_Lz_dropdown.value
            V0e = x if xaxis == "V₀ₑ" else overlap_V0e_dropdown.value
            V0h = x if xaxis == "V₀ₕ" else overlap_V0h_dropdown.value
            deltae_nm = x if xaxis == "δₑ" else float(1e9 * delta_e(material, V0e))
            deltah_nm = x if xaxis == "δₕ" else float(1e9 * delta_h(material, V0h))
            nxe = int(x) if xaxis == "nₓₑ" else overlap_nxe_dropdown.value
            nye = int(x) if xaxis == "nᵧₑ" else overlap_nye_dropdown.value
            nze = int(x) if xaxis == "nzₑ" else overlap_nze_dropdown.value
            nxh = int(x) if xaxis == "nₓₕ" else overlap_nxh_dropdown.value
            nyh = int(x) if xaxis == "nᵧₕ" else overlap_nyh_dropdown.value
            nzh = int(x) if xaxis == "nzₕ" else overlap_nzh_dropdown.value
            O = overlap_from_parameters(
                Lx, Ly, Lz,
                deltae_nm, deltah_nm,
                nxe, nye, nze,
                nxh, nyh, nzh
            )
            if quantity == "O":
                y = O
                ylabel = "O"
            else:
                y = np.abs(O)**2
                ylabel = "|O|²"
            ys.append(float(np.asarray(y)))
        plt.figure(figsize=(9, 6))
        plt.plot(xs, ys, linewidth=2.0, color='limegreen')
        if xaxis == "Lₓ":
            plt.xlabel("Lₓ (nm)")
        elif xaxis == "Lᵧ":
            plt.xlabel("Lᵧ (nm)")
        elif xaxis == "Lz":
            plt.xlabel("Lz (nm)")
        elif xaxis == "δₑ":
            plt.xlabel("δₑ (nm)")
        elif xaxis == "δₕ":
            plt.xlabel("δₕ (nm)")
        elif xaxis == "V₀ₑ":
            plt.xlabel("V₀ₑ (meV)")
        elif xaxis == "V₀ₕ":
            plt.xlabel("V₀ₕ (meV)")
        elif xaxis == "nₓₑ":
            plt.xlabel("nₓₑ")
            plt.xticks(n_vals)
        elif xaxis == "nᵧₑ":
            plt.xlabel("nᵧₑ")
            plt.xticks(n_vals)
        elif xaxis == "nzₑ":
            plt.xlabel("nzₑ")
            plt.xticks(n_vals)
        elif xaxis == "nₓₕ":
            plt.xlabel("nₓₕ")
            plt.xticks(n_vals)
        elif xaxis == "nᵧₕ":
            plt.xlabel("nᵧₕ")
            plt.xticks(n_vals)
        else:
            plt.xlabel("nzₕ")
            plt.xticks(n_vals)
        plt.ylabel(ylabel)
        plt.title(f"{quantity} vs {xaxis} for {material}")
        plt.grid(True, alpha=0.3)
        plt.show()

for widget in [
    overlap_material_dropdown, overlap_quantity_dropdown, overlap_x_dropdown,
    overlap_Lx_dropdown, overlap_Ly_dropdown, overlap_Lz_dropdown,
    overlap_deltae_dropdown, overlap_deltah_dropdown,
    overlap_V0e_dropdown, overlap_V0h_dropdown,
    overlap_nxe_dropdown, overlap_nye_dropdown, overlap_nze_dropdown,
    overlap_nxh_dropdown, overlap_nyh_dropdown, overlap_nzh_dropdown
]:
    widget.observe(overlap_plot, names='value')
overlap_x_dropdown.observe(overlap_visibility, names='value')
display(VBox([
    HBox([overlap_material_dropdown, overlap_quantity_dropdown, overlap_x_dropdown]),
    HBox([overlap_Lx_dropdown, overlap_Ly_dropdown, overlap_Lz_dropdown]),
    HBox([overlap_deltae_dropdown, overlap_deltah_dropdown, overlap_V0e_dropdown, overlap_V0h_dropdown]),
    HBox([overlap_nxe_dropdown, overlap_nye_dropdown, overlap_nze_dropdown]),
    HBox([overlap_nxh_dropdown, overlap_nyh_dropdown, overlap_nzh_dropdown]),
    overlap_output
]))
overlap_visibility()
overlap_plot()

In [95]:
state_match_material_dropdown = Dropdown(
    options=materials,
    value=materials[0],
    description='Material:'
)

state_match_quantity_dropdown = Dropdown(
    options=["O", "|O|²"],
    value="|O|²",
    description='Quantity:'
)

state_match_pair_dropdown = Dropdown(
    options=["nₓₑ vs nₓₕ", "nᵧₑ vs nᵧₕ", "nzₑ vs nzₕ"],
    value="nₓₑ vs nₓₕ",
    description='State Pair:'
)

state_match_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='Lₓ (nm):')
state_match_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
state_match_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
state_match_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
state_match_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
state_match_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
state_match_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
state_match_nze_dropdown = Dropdown(options=n_vals, value=1, description='nzₑ:')
state_match_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
state_match_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
state_match_nzh_dropdown = Dropdown(options=n_vals, value=1, description='nzₕ:')
state_match_output = Output()

def state_match_plot(*args):
    with state_match_output:
        clear_output(wait=True)
        material = state_match_material_dropdown.value
        quantity = state_match_quantity_dropdown.value
        pair = state_match_pair_dropdown.value
        deltae_nm = float(1e9 * delta_e(material, state_match_V0e_dropdown.value))
        deltah_nm = float(1e9 * delta_h(material, state_match_V0h_dropdown.value))
        X, Y = np.meshgrid(n_vals, n_vals, indexing='ij')
        Z = np.zeros_like(X, dtype=float)
        for i, ne in enumerate(n_vals):
            for j, nh in enumerate(n_vals):
                nxe = ne if pair == "nₓₑ vs nₓₕ" else state_match_nxe_dropdown.value
                nye = ne if pair == "nᵧₑ vs nᵧₕ" else state_match_nye_dropdown.value
                nze = ne if pair == "nzₑ vs nzₕ" else state_match_nze_dropdown.value
                nxh = nh if pair == "nₓₑ vs nₓₕ" else state_match_nxh_dropdown.value
                nyh = nh if pair == "nᵧₑ vs nᵧₕ" else state_match_nyh_dropdown.value
                nzh = nh if pair == "nzₑ vs nzₕ" else state_match_nzh_dropdown.value
                O = overlap_from_parameters(
                    state_match_Lx_dropdown.value, state_match_Ly_dropdown.value, state_match_Lz_dropdown.value,
                    deltae_nm, deltah_nm,
                    nxe, nye, nze,
                    nxh, nyh, nzh
                )
                if quantity == "O":
                    Z[i, j] = O
                    cbar_label = "O"
                else:
                    Z[i, j] = np.abs(O)**2
                    cbar_label = "|O|²"
        fig, ax = plt.subplots(figsize=(9, 6))
        pcm = ax.contourf(X, Y, Z, levels=30, cmap='viridis')
        cbar = fig.colorbar(pcm, ax=ax, pad=0.08)
        cbar.set_label(cbar_label)
        if pair == "nₓₑ vs nₓₕ":
            ax.set_xlabel("nₓₕ")
            ax.set_ylabel("nₓₑ")
        elif pair == "nᵧₑ vs nᵧₕ":
            ax.set_xlabel("nᵧₕ")
            ax.set_ylabel("nᵧₑ")
        else:
            ax.set_xlabel("nzₕ")
            ax.set_ylabel("nzₑ")
        ax.set_xticks(n_vals)
        ax.set_yticks(n_vals)
        ax.set_title(f"{quantity} : State Matching for {material}")
        plt.show()
        plt.close(fig)

for widget in [
    state_match_material_dropdown, state_match_quantity_dropdown, state_match_pair_dropdown,
    state_match_Lx_dropdown, state_match_Ly_dropdown, state_match_Lz_dropdown,
    state_match_V0e_dropdown, state_match_V0h_dropdown,
    state_match_nxe_dropdown, state_match_nye_dropdown, state_match_nze_dropdown,
    state_match_nxh_dropdown, state_match_nyh_dropdown, state_match_nzh_dropdown
]:
    widget.observe(state_match_plot, names='value')
display(VBox([
    HBox([state_match_material_dropdown, state_match_quantity_dropdown, state_match_pair_dropdown]),
    HBox([state_match_Lx_dropdown, state_match_Ly_dropdown, state_match_Lz_dropdown]),
    HBox([state_match_V0e_dropdown, state_match_V0h_dropdown]),
    HBox([state_match_nxe_dropdown, state_match_nye_dropdown, state_match_nze_dropdown]),
    HBox([state_match_nxh_dropdown, state_match_nyh_dropdown, state_match_nzh_dropdown]),
    state_match_output
]))
state_match_plot()

In [96]:
overlap_emission_material_dropdown = Dropdown(
    options=materials,
    value=materials[0],
    description='Material:'
)

overlap_emission_quantity_dropdown = Dropdown(
    options=["O", "|O|²"],
    value="|O|²",
    description='Quantity:'
)

overlap_emission_x_dropdown = Dropdown(
    options=["Eᵧ", "λ"],
    value="Eᵧ",
    description='x-axis:'
)

overlap_emission_study_dropdown = Dropdown(
    options=["Lₓ", "Lᵧ", "Lz", "ε", "V₀ₑ", "V₀ₕ", "T", "nₓₑ", "nᵧₑ", "nzₑ", "nₓₕ", "nᵧₕ", "nzₕ"],
    value="Lₓ",
    description='Study:'
)

overlap_emission_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[-1]), description='Lₓ,max (nm):')
overlap_emission_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
overlap_emission_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
overlap_emission_epsilon_dropdown = Dropdown(options=[float(v) for v in epsilon_vals], value=float(epsilon_vals[0]), description='ε (kV/cm):')
overlap_emission_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
overlap_emission_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
overlap_emission_T_dropdown = Dropdown(options=[float(v) for v in T_C_vals], value=float(T_C_vals[0]), description='T (°C):')
overlap_emission_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
overlap_emission_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
overlap_emission_nze_dropdown = Dropdown(options=n_vals, value=1, description='nzₑ:')
overlap_emission_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
overlap_emission_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
overlap_emission_nzh_dropdown = Dropdown(options=n_vals, value=1, description='nzₕ:')
overlap_emission_output = Output()

def overlap_emission_visibility(*args):
    if overlap_emission_study_dropdown.value == "Lₓ":
        overlap_emission_Lx_dropdown.description = 'Lₓ,max (nm):'
    else:
        overlap_emission_Lx_dropdown.description = 'Lₓ (nm):'
    if overlap_emission_study_dropdown.value == "Lᵧ":
        overlap_emission_Ly_dropdown.description = 'Lᵧ,max (nm):'
    else:
        overlap_emission_Ly_dropdown.description = 'Lᵧ (nm):'
    if overlap_emission_study_dropdown.value == "Lz":
        overlap_emission_Lz_dropdown.description = 'Lz,max (nm):'
    else:
        overlap_emission_Lz_dropdown.description = 'Lz (nm):'
    if overlap_emission_study_dropdown.value == "ε":
        overlap_emission_epsilon_dropdown.description = 'ε,max (kV/cm):'
    else:
        overlap_emission_epsilon_dropdown.description = 'ε (kV/cm):'
    if overlap_emission_study_dropdown.value == "V₀ₑ":
        overlap_emission_V0e_dropdown.description = 'V₀ₑ,max (meV):'
    else:
        overlap_emission_V0e_dropdown.description = 'V₀ₑ (meV):'
    if overlap_emission_study_dropdown.value == "V₀ₕ":
        overlap_emission_V0h_dropdown.description = 'V₀ₕ,max (meV):'
    else:
        overlap_emission_V0h_dropdown.description = 'V₀ₕ (meV):'
    if overlap_emission_study_dropdown.value == "T":
        overlap_emission_T_dropdown.description = 'T,max (°C):'
    else:
        overlap_emission_T_dropdown.description = 'T (°C):'

def overlap_emission_plot(*args):
    with overlap_emission_output:
        clear_output(wait=True)
        material = overlap_emission_material_dropdown.value
        quantity = overlap_emission_quantity_dropdown.value
        xaxis = overlap_emission_x_dropdown.value
        study = overlap_emission_study_dropdown.value
        if study == "Lₓ":
            ws = np.linspace(1.0, overlap_emission_Lx_dropdown.value, 300)
        elif study == "Lᵧ":
            ws = np.linspace(1.0, overlap_emission_Ly_dropdown.value, 300)
        elif study == "Lz":
            ws = np.linspace(0.5, overlap_emission_Lz_dropdown.value, 300)
        elif study == "ε":
            ws = np.linspace(1.0, overlap_emission_epsilon_dropdown.value, 300)
        elif study == "V₀ₑ":
            ws = np.linspace(1.0, overlap_emission_V0e_dropdown.value, 300)
        elif study == "V₀ₕ":
            ws = np.linspace(1.0, overlap_emission_V0h_dropdown.value, 300)
        elif study == "T":
            ws = np.linspace(1.0, overlap_emission_T_dropdown.value, 300)
        else:
            ws = np.array(n_vals, dtype=float)
        xs = []
        ys = []
        for w in ws:
            Lx = w if study == "Lₓ" else overlap_emission_Lx_dropdown.value
            Ly = w if study == "Lᵧ" else overlap_emission_Ly_dropdown.value
            Lz = w if study == "Lz" else overlap_emission_Lz_dropdown.value
            epsilon = w if study == "ε" else overlap_emission_epsilon_dropdown.value
            V0e = w if study == "V₀ₑ" else overlap_emission_V0e_dropdown.value
            V0h = w if study == "V₀ₕ" else overlap_emission_V0h_dropdown.value
            T = w if study == "T" else overlap_emission_T_dropdown.value
            nxe = int(w) if study == "nₓₑ" else overlap_emission_nxe_dropdown.value
            nye = int(w) if study == "nᵧₑ" else overlap_emission_nye_dropdown.value
            nze = int(w) if study == "nzₑ" else overlap_emission_nze_dropdown.value
            nxh = int(w) if study == "nₓₕ" else overlap_emission_nxh_dropdown.value
            nyh = int(w) if study == "nᵧₕ" else overlap_emission_nyh_dropdown.value
            nzh = int(w) if study == "nzₕ" else overlap_emission_nzh_dropdown.value
            deltae_nm = float(1e9 * delta_e(material, V0e))
            deltah_nm = float(1e9 * delta_h(material, V0h))
            O = overlap_from_parameters(
                Lx, Ly, Lz,
                deltae_nm, deltah_nm,
                nxe, nye, nze,
                nxh, nyh, nzh
            )
            if quantity == "O":
                y = O
                ylabel = "O"
            else:
                y = np.abs(O)**2
                ylabel = "|O|²"
            if xaxis == "Eᵧ":
                x = J_to_eV(photon_energy(
                    material,
                    nxe, nye, nze,
                    nxh, nyh, nzh,
                    Lx, Ly, Lz,
                    epsilon,
                    V0e,
                    V0h,
                    T
                ))
                xlabel = "Eᵧ (eV)"
            else:
                x = photon_wavelength(
                    material,
                    nxe, nye, nze,
                    nxh, nyh, nzh,
                    Lx, Ly, Lz,
                    epsilon,
                    V0e,
                    V0h,
                    T
                )
                xlabel = "λ (nm)"
            xs.append(float(np.asarray(x)))
            ys.append(float(np.asarray(y)))
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.plot(xs, ys, linewidth=2, color='hotpink')
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{quantity} vs {xaxis} for {material}")
        ax.grid(True, alpha=0.3)
        plt.show()
        plt.close(fig)

for widget in [
    overlap_emission_material_dropdown, overlap_emission_quantity_dropdown, overlap_emission_x_dropdown, overlap_emission_study_dropdown,
    overlap_emission_Lx_dropdown, overlap_emission_Ly_dropdown, overlap_emission_Lz_dropdown,
    overlap_emission_epsilon_dropdown, overlap_emission_V0e_dropdown, overlap_emission_V0h_dropdown, overlap_emission_T_dropdown,
    overlap_emission_nxe_dropdown, overlap_emission_nye_dropdown, overlap_emission_nze_dropdown,
    overlap_emission_nxh_dropdown, overlap_emission_nyh_dropdown, overlap_emission_nzh_dropdown
]:
    widget.observe(overlap_emission_plot, names='value')
overlap_emission_study_dropdown.observe(overlap_emission_visibility, names='value')
display(VBox([
    HBox([overlap_emission_material_dropdown, overlap_emission_quantity_dropdown, overlap_emission_x_dropdown, overlap_emission_study_dropdown]),
    HBox([overlap_emission_Lx_dropdown, overlap_emission_Ly_dropdown, overlap_emission_Lz_dropdown]),
    HBox([overlap_emission_epsilon_dropdown, overlap_emission_V0e_dropdown, overlap_emission_V0h_dropdown, overlap_emission_T_dropdown]),
    HBox([overlap_emission_nxe_dropdown, overlap_emission_nye_dropdown, overlap_emission_nze_dropdown]),
    HBox([overlap_emission_nxh_dropdown, overlap_emission_nyh_dropdown, overlap_emission_nzh_dropdown]),
    overlap_emission_output
]))
overlap_emission_visibility()
overlap_emission_plot()

### 4. Materials Comparison

In [97]:
material_bar_quantity_dropdown = Dropdown(
    options=["Eₑ", "Eₕ", "Eᵧ", "λ", "O", "|O|²"],
    value="Eₑ",
    description='Quantity:'
)

material_bar_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='Lₓ (nm):')
material_bar_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
material_bar_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
material_bar_epsilon_dropdown = Dropdown(options=[float(v) for v in epsilon_vals], value=float(epsilon_vals[0]), description='ε (kV/cm):')
material_bar_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
material_bar_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
material_bar_T_dropdown = Dropdown(options=[float(v) for v in T_C_vals], value=float(T_C_vals[0]), description='T (°C):')
material_bar_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
material_bar_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
material_bar_nze_dropdown = Dropdown(options=n_vals, value=1, description='nzₑ:')
material_bar_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
material_bar_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
material_bar_nzh_dropdown = Dropdown(options=n_vals, value=1, description='nzₕ:')
material_bar_output = Output()

def material_bar_plot(*args):
    with material_bar_output:
        clear_output(wait=True)
        quantity = material_bar_quantity_dropdown.value
        xs = np.arange(len(materials))
        ys = []
        colors = ['slateblue', 'tomato', 'darkturquoise', 'peru']
        for mat in materials:
            if quantity == "Eₑ":
                y = J_to_meV(electron_energies(
                    material_bar_nxe_dropdown.value, material_bar_nye_dropdown.value, material_bar_nze_dropdown.value,
                    material_bar_Lx_dropdown.value, material_bar_Ly_dropdown.value, material_bar_Lz_dropdown.value,
                    material_bar_epsilon_dropdown.value,
                    material_bar_V0e_dropdown.value,
                    mat
                ))
                ylabel = "Eₑ (meV)"
            elif quantity == "Eₕ":
                y = J_to_meV(hole_energies(
                    material_bar_nxh_dropdown.value, material_bar_nyh_dropdown.value, material_bar_nzh_dropdown.value,
                    material_bar_Lx_dropdown.value, material_bar_Ly_dropdown.value, material_bar_Lz_dropdown.value,
                    material_bar_epsilon_dropdown.value,
                    material_bar_V0h_dropdown.value,
                    mat
                ))
                ylabel = "Eₕ (meV)"
            elif quantity == "Eᵧ":
                y = J_to_eV(photon_energy(
                    mat,
                    material_bar_nxe_dropdown.value, material_bar_nye_dropdown.value, material_bar_nze_dropdown.value,
                    material_bar_nxh_dropdown.value, material_bar_nyh_dropdown.value, material_bar_nzh_dropdown.value,
                    material_bar_Lx_dropdown.value, material_bar_Ly_dropdown.value, material_bar_Lz_dropdown.value,
                    material_bar_epsilon_dropdown.value,
                    material_bar_V0e_dropdown.value,
                    material_bar_V0h_dropdown.value,
                    material_bar_T_dropdown.value
                ))
                ylabel = "Eᵧ (eV)"
            elif quantity == "λ":
                y = photon_wavelength(
                    mat,
                    material_bar_nxe_dropdown.value, material_bar_nye_dropdown.value, material_bar_nze_dropdown.value,
                    material_bar_nxh_dropdown.value, material_bar_nyh_dropdown.value, material_bar_nzh_dropdown.value,
                    material_bar_Lx_dropdown.value, material_bar_Ly_dropdown.value, material_bar_Lz_dropdown.value,
                    material_bar_epsilon_dropdown.value,
                    material_bar_V0e_dropdown.value,
                    material_bar_V0h_dropdown.value,
                    material_bar_T_dropdown.value
                )
                ylabel = "λ (nm)"
            elif quantity == "O":
                deltae_nm = float(1e9 * delta_e(mat, material_bar_V0e_dropdown.value))
                deltah_nm = float(1e9 * delta_h(mat, material_bar_V0h_dropdown.value))
                y = overlap_from_parameters(
                    material_bar_Lx_dropdown.value, material_bar_Ly_dropdown.value, material_bar_Lz_dropdown.value,
                    deltae_nm, deltah_nm,
                    material_bar_nxe_dropdown.value, material_bar_nye_dropdown.value, material_bar_nze_dropdown.value,
                    material_bar_nxh_dropdown.value, material_bar_nyh_dropdown.value, material_bar_nzh_dropdown.value
                )
                ylabel = "O"
            else:
                deltae_nm = float(1e9 * delta_e(mat, material_bar_V0e_dropdown.value))
                deltah_nm = float(1e9 * delta_h(mat, material_bar_V0h_dropdown.value))
                O = overlap_from_parameters(
                    material_bar_Lx_dropdown.value, material_bar_Ly_dropdown.value, material_bar_Lz_dropdown.value,
                    deltae_nm, deltah_nm,
                    material_bar_nxe_dropdown.value, material_bar_nye_dropdown.value, material_bar_nze_dropdown.value,
                    material_bar_nxh_dropdown.value, material_bar_nyh_dropdown.value, material_bar_nzh_dropdown.value
                )
                y = np.abs(O)**2
                ylabel = "|O|²"
            ys.append(float(np.asarray(y)))
        plt.figure(figsize=(9, 6))
        bars = plt.bar(xs, ys, color=colors, edgecolor='black')
        plt.xticks(xs, materials)
        plt.ylabel(ylabel)
        plt.title(f"{quantity} : Comparison of Materials")
        plt.grid(True, alpha=0.3, axis='y')
        for bar, y in zip(bars, ys):
            plt.text(
                bar.get_x() + bar.get_width()/2.0,
                bar.get_height(),
                f"{y:.3e}" if abs(y) < 1e-2 or abs(y) >= 1e3 else f"{y:.3f}",
                ha='center',
                va='bottom',
                fontsize=10
            )
        plt.show()

for widget in [
    material_bar_quantity_dropdown,
    material_bar_Lx_dropdown, material_bar_Ly_dropdown, material_bar_Lz_dropdown,
    material_bar_epsilon_dropdown, material_bar_V0e_dropdown, material_bar_V0h_dropdown, material_bar_T_dropdown,
    material_bar_nxe_dropdown, material_bar_nye_dropdown, material_bar_nze_dropdown,
    material_bar_nxh_dropdown, material_bar_nyh_dropdown, material_bar_nzh_dropdown
]:
    widget.observe(material_bar_plot, names='value')
display(VBox([
    HBox([material_bar_quantity_dropdown]),
    HBox([material_bar_Lx_dropdown, material_bar_Ly_dropdown, material_bar_Lz_dropdown]),
    HBox([material_bar_epsilon_dropdown, material_bar_V0e_dropdown, material_bar_V0h_dropdown, material_bar_T_dropdown]),
    HBox([material_bar_nxe_dropdown, material_bar_nye_dropdown, material_bar_nze_dropdown]),
    HBox([material_bar_nxh_dropdown, material_bar_nyh_dropdown, material_bar_nzh_dropdown]),
    material_bar_output
]))
material_bar_plot()

### 5. Intensity Study

In [98]:
intensity_material_dropdown = Dropdown(
    options=materials,
    value=materials[0],
    description='Material:'
)

intensity_quantity_dropdown = Dropdown(
    options=["|O|²", "I = Eᵧ|O|²"],
    value="I = Eᵧ|O|²",
    description='Quantity:'
)

intensity_x_dropdown = Dropdown(
    options=["Lₓ", "Lᵧ", "Lz", "ε", "V₀ₑ", "V₀ₕ", "T"],
    value="Lₓ",
    description='x-axis:'
)

intensity_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[-1]), description='Lₓ,max (nm):')
intensity_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
intensity_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
intensity_epsilon_dropdown = Dropdown(options=[float(v) for v in epsilon_vals], value=float(epsilon_vals[0]), description='ε (kV/cm):')
intensity_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
intensity_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
intensity_T_dropdown = Dropdown(options=[float(v) for v in T_C_vals], value=float(T_C_vals[0]), description='T (°C):')
intensity_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
intensity_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
intensity_nze_dropdown = Dropdown(options=n_vals, value=1, description='nzₑ:')
intensity_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
intensity_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
intensity_nzh_dropdown = Dropdown(options=n_vals, value=1, description='nzₕ:')
intensity_output = Output()

def intensity_plot(*args):
    with intensity_output:
        clear_output(wait=True)

        quantity = intensity_quantity_dropdown.value
        xaxis = intensity_x_dropdown.value
        material = intensity_material_dropdown.value

        if xaxis == "Lₓ":
            xs = np.linspace(1.0, intensity_Lx_dropdown.value, 300)
        elif xaxis == "Lᵧ":
            xs = np.linspace(1.0, intensity_Ly_dropdown.value, 300)
        elif xaxis == "Lz":
            xs = np.linspace(0.5, intensity_Lz_dropdown.value, 300)
        elif xaxis == "ε":
            xs = np.linspace(1.0, intensity_epsilon_dropdown.value, 300)
        elif xaxis == "V₀ₑ":
            xs = np.linspace(1.0, intensity_V0e_dropdown.value, 300)
        elif xaxis == "V₀ₕ":
            xs = np.linspace(1.0, intensity_V0h_dropdown.value, 300)
        else:
            xs = np.linspace(1.0, intensity_T_dropdown.value, 300)

        ys = []

        for x in xs:
            Lx = x if xaxis == "Lₓ" else intensity_Lx_dropdown.value
            Ly = x if xaxis == "Lᵧ" else intensity_Ly_dropdown.value
            Lz = x if xaxis == "Lz" else intensity_Lz_dropdown.value
            epsilon = x if xaxis == "ε" else intensity_epsilon_dropdown.value
            V0e = x if xaxis == "V₀ₑ" else intensity_V0e_dropdown.value
            V0h = x if xaxis == "V₀ₕ" else intensity_V0h_dropdown.value
            T = x if xaxis == "T" else intensity_T_dropdown.value
            deltae_nm = float(1e9 * delta_e(material, V0e))
            deltah_nm = float(1e9 * delta_h(material, V0h))
            O = overlap_from_parameters(
                Lx, Ly, Lz,
                deltae_nm, deltah_nm,
                intensity_nxe_dropdown.value, intensity_nye_dropdown.value, intensity_nze_dropdown.value,
                intensity_nxh_dropdown.value, intensity_nyh_dropdown.value, intensity_nzh_dropdown.value
            )
            overlap_sq = np.abs(O)**2
            E_gamma = float(np.asarray(J_to_eV(photon_energy(
                material,
                intensity_nxe_dropdown.value, intensity_nye_dropdown.value, intensity_nze_dropdown.value,
                intensity_nxh_dropdown.value, intensity_nyh_dropdown.value, intensity_nzh_dropdown.value,
                Lx, Ly, Lz,
                epsilon,
                V0e,
                V0h,
                T
            ))))
            if quantity == "|O|²":
                y = overlap_sq
                ylabel = "|O|²"
            else:
                y = E_gamma * overlap_sq
                ylabel = "I ∝ Eᵧ|O|²"

            ys.append(float(y))
        plt.figure(figsize=(9, 6))
        plt.plot(xs, ys, color='mediumorchid', linewidth=2)
        plt.xlabel(xaxis)
        plt.ylabel(ylabel)
        plt.title(f"{quantity} vs {xaxis}")
        plt.grid(True, alpha=0.3)
        plt.show()

for w in [
    intensity_material_dropdown, intensity_quantity_dropdown, intensity_x_dropdown,
    intensity_Lx_dropdown, intensity_Ly_dropdown, intensity_Lz_dropdown,
    intensity_epsilon_dropdown, intensity_V0e_dropdown, intensity_V0h_dropdown, intensity_T_dropdown,
    intensity_nxe_dropdown, intensity_nye_dropdown, intensity_nze_dropdown,
    intensity_nxh_dropdown, intensity_nyh_dropdown, intensity_nzh_dropdown
]:
    w.observe(intensity_plot, names='value')
display(VBox([
    HBox([intensity_material_dropdown, intensity_quantity_dropdown, intensity_x_dropdown]),
    HBox([intensity_Lx_dropdown, intensity_Ly_dropdown, intensity_Lz_dropdown]),
    HBox([intensity_epsilon_dropdown, intensity_V0e_dropdown, intensity_V0h_dropdown, intensity_T_dropdown]),
    HBox([intensity_nxe_dropdown, intensity_nye_dropdown, intensity_nze_dropdown]),
    HBox([intensity_nxh_dropdown, intensity_nyh_dropdown, intensity_nzh_dropdown]),
    intensity_output
]))

intensity_plot()

In [99]:
def gaussian_emission_spectrum(lambda0_nm, sigma_nm=12.0, amp=1.0, lam_min=100.0, lam_max=2000.0, npts=4000):
    lam = np.linspace(lam_min, lam_max, npts)
    intensity = amp * np.exp(-0.5 * ((lam - lambda0_nm) / sigma_nm)**2)
    return lam, intensity

material_dropdown = Dropdown(options=materials, value=materials[0], description='Material:', layout={'width': '250px'})
T_dropdown = Dropdown(options=[float(v) for v in T_C], value=float(T_C[0]), description='T (°C):', layout={'width': '220px'})
Lx_dropdown = Dropdown(options=[float(v) for v in Lx], value=float(Lx[0]), description='Lₓ (nm):', layout={'width': '220px'})
Ly_dropdown = Dropdown(options=[float(v) for v in Ly], value=float(Ly[0]), description='Lᵧ (nm):', layout={'width': '220px'})
Lz_dropdown = Dropdown(options=[float(v) for v in Lz], value=float(Lz[0]), description='L𝓏 (nm):', layout={'width': '220px'})
epsilon_dropdown = Dropdown(options=[float(v) for v in epsilon], value=float(epsilon[0]), description='ε (kV/cm):', layout={'width': '220px'})
V0e_dropdown = Dropdown(options=[float(v) for v in V0e], value=float(V0e[0]), description='V₀ₑ (meV):', layout={'width': '220px'})
V0h_dropdown = Dropdown(options=[float(v) for v in V0h], value=float(V0h[0]), description='V₀ₕ (meV):', layout={'width': '220px'})
n_options = [1, 2, 3, 4]
nxe_dropdown = Dropdown(options=n_options, value=1, description='nₓₑ:', layout={'width': '160px'})
nye_dropdown = Dropdown(options=n_options, value=1, description='nᵧₑ:', layout={'width': '160px'})
nze_dropdown = Dropdown(options=n_options, value=1, description='n𝓏ₑ:', layout={'width': '160px'})
nxh_dropdown = Dropdown(options=n_options, value=1, description='nₓₕ:', layout={'width': '160px'})
nyh_dropdown = Dropdown(options=n_options, value=1, description='nᵧₕ:', layout={'width': '160px'})
nzh_dropdown = Dropdown(options=n_options, value=1, description='n𝓏ₕ:', layout={'width': '160px'})
sigma_dropdown = Dropdown(options=[5.0, 8.0, 10.0, 12.0, 15.0, 20.0, 25.0, 30.0], value=20.0, description='σ (nm):', layout={'width': '220px'})
output = Output()

def update_output(*args):
    with output:
        clear_output(wait=True)

        material = material_dropdown.value
        Lx_val = Lx_dropdown.value
        Ly_val = Ly_dropdown.value
        Lz_val = Lz_dropdown.value
        eps_val = epsilon_dropdown.value
        V0e_val = V0e_dropdown.value
        V0h_val = V0h_dropdown.value
        T_val = T_dropdown.value
        nx_e_val = nxe_dropdown.value
        ny_e_val = nye_dropdown.value
        nz_e_val = nze_dropdown.value
        nx_h_val = nxh_dropdown.value
        ny_h_val = nyh_dropdown.value
        nz_h_val = nzh_dropdown.value
        sigma_nm = sigma_dropdown.value

        lam_nm = float(np.asarray(
            photon_wavelength(material,
                                 nx_e_val, ny_e_val, nz_e_val,
                                 nx_h_val, ny_h_val, nz_h_val,
                                 Lx_val, Ly_val, Lz_val,
                                 eps_val, V0e_val, V0h_val, T_val)
        ))

        E_photon_eV = 1240.0 / lam_nm
        spectrum_region = em_spectrum_region(lam_nm)

        lam_min = 0.0
        lam_max = 2000.0

        lam_grid, intensity = gaussian_emission_spectrum(
            lambda0_nm=lam_nm,
            sigma_nm=sigma_nm,
            amp=1.0,
            lam_min=lam_min,
            lam_max=lam_max,
            npts=4000
        )
        plt.figure(figsize=(11, 6))
        plt.plot(lam_grid, intensity, color='darkmagenta', linewidth=4.0)
        plt.axvline(lam_nm, color='darkmagenta', linestyle='--', linewidth=2.0)
        plt.xlabel("Wavelength λ (nm)")
        plt.ylabel("Normalized Intensity (a.u.)")
        plt.title(f"Emission Spectrum of {material}")
        plt.grid(True, alpha=0.3)
        x_text = lam_nm + 0.03 * (lam_max - lam_min)
        y_text1 = 0.92 * np.max(intensity)
        y_text2 = 0.80 * np.max(intensity)
        plt.text(x_text, y_text1,
                 f"State: ({nx_e_val},{ny_e_val},{nz_e_val}) → ({nx_h_val},{ny_h_val},{nz_h_val})",
                 color='darkmagenta', fontsize=11, ha='left', va='top')
        plt.text(x_text, y_text2,
                 f"Eᵧ = {E_photon_eV:.4f} eV",
                 color='darkmagenta', fontsize=11, ha='left', va='top')
        plt.show()
        print("=" * 120)
        print("EMISSION SPECTRUM OF INDIVIDUAL RECOMBINATION STATE")
        print("=" * 120)
        print(f"Material: {material}")
        print(f"Lₓ = {Lx_val:.3f} nm,  Lᵧ = {Ly_val:.3f} nm,  L𝓏 = {Lz_val:.3f} nm")
        print(f"ε = {eps_val:.3f} kV/cm,  V₀ₑ = {V0e_val:.3f} meV,  V₀ₕ = {V0h_val:.3f} meV")
        print(f"T = {T_val:.3f} °C")
        print(f"Electron Quantum State: ({nx_e_val},{ny_e_val},{nz_e_val})")
        print(f"Hole Quantum State: ({nx_h_val},{ny_h_val},{nz_h_val})")
        print(f"λ = {lam_nm:.6f} nm")
        print(f"Eᵧ = {E_photon_eV:.6f} eV")
        print(f"σ = {sigma_nm:.3f} nm")
        print(f"EM spectrum region = {spectrum_region}")

controls_row1 = HBox([material_dropdown, T_dropdown])
controls_row2 = HBox([Lx_dropdown, Ly_dropdown, Lz_dropdown])
controls_row3 = HBox([epsilon_dropdown, V0e_dropdown, V0h_dropdown])
controls_row4 = HBox([nxe_dropdown, nye_dropdown, nze_dropdown])
controls_row5 = HBox([nxh_dropdown, nyh_dropdown, nzh_dropdown])
controls_row6 = HBox([sigma_dropdown])

ui = VBox([
    controls_row1,
    controls_row2,
    controls_row3,
    controls_row4,
    controls_row5,
    controls_row6,
    output
])
for w in [
    material_dropdown, Lx_dropdown, Ly_dropdown, Lz_dropdown,
    epsilon_dropdown, V0e_dropdown, V0h_dropdown, T_dropdown,
    nxe_dropdown, nye_dropdown, nze_dropdown,
    nxh_dropdown, nyh_dropdown, nzh_dropdown,
    sigma_dropdown
]:
    w.observe(updat_output, names='value')
display(ui)
update_output()

NameError: name 'updat_output' is not defined

In [101]:
def photon_energy_eV_from_lambda_nm(lambda_nm):
    return 1240.0 / lambda_nm

def overlap_strength_from_project_code(material, Lx_val, Ly_val, Lz_val, V0e_val, V0h_val,
                                       nx_e, ny_e, nz_e, nx_h, ny_h, nz_h):
    deltae_nm = float(1e9 * delta_e(material, V0e_val))
    deltah_nm = float(1e9 * delta_h(material, V0h_val))
    O = overlap_from_parameters(
        Lx_val, Ly_val, Lz_val,
        deltae_nm, deltah_nm,
        nx_e, ny_e, nz_e,
        nx_h, ny_h, nz_h
    )
    return float(np.abs(np.asarray(O))**2)

def build_transition_list_for_one_material(material, Lx_val, Ly_val, Lz_val, eps_val, V0e_val, V0h_val, T_val):
    state_vals = [1, 2, 3, 4]
    transitions = []
    for nx_e in state_vals:
        for ny_e in state_vals:
            for nz_e in state_vals:
                for nx_h in state_vals:
                    for ny_h in state_vals:
                        for nz_h in state_vals:
                            lam_nm = float(np.asarray(
                                photon_wavelength(
                                    material,
                                    nx_e, ny_e, nz_e,
                                    nx_h, ny_h, nz_h,
                                    Lx_val, Ly_val, Lz_val,
                                    eps_val, V0e_val, V0h_val, T_val
                                )
                            ))
                            if np.isfinite(lam_nm) and lam_nm > 0:
                                E_photon_eV = photon_energy_eV_from_lambda_nm(lam_nm)
                                overlap_sq = overlap_strength_from_project_code(
                                    material, Lx_val, Ly_val, Lz_val, V0e_val, V0h_val,
                                    nx_e, ny_e, nz_e, nx_h, ny_h, nz_h
                                )
                                intensity_weight = overlap_sq * E_photon_eV
                                region = em_spectrum_region(lam_nm)
                                if np.isfinite(intensity_weight) and intensity_weight > 0:
                                    transitions.append({
                                        'material': material,
                                        'e_state': (nx_e, ny_e, nz_e),
                                        'h_state': (nx_h, ny_h, nz_h),
                                        'lambda_nm': lam_nm,
                                        'E_photon_eV': E_photon_eV,
                                        'overlap_sq': overlap_sq,
                                        'weight': intensity_weight,
                                        'region': region
                                    })
    return transitions

def build_weighted_spectrum(transitions, sigma_nm, lam_min=None, lam_max=None, npts=6000):
    wavelengths = [t['lambda_nm'] for t in transitions]
    if lam_min is None:
        lam_min = max(0.01, min(wavelengths) - 8.0 * sigma_nm)
    if lam_max is None:
        lam_max = max(2000.0, max(wavelengths) + 8.0 * sigma_nm)
    lam_grid = np.linspace(lam_min, lam_max, npts)
    total_intensity = np.zeros_like(lam_grid)
    for t in transitions:
        lam0 = t['lambda_nm']
        total_intensity += t['weight'] * np.exp(-0.5 * ((lam_grid - lam0) / sigma_nm)**2)
    raw_intensity = total_intensity.copy()
    if np.max(total_intensity) > 0:
        total_intensity = total_intensity / np.max(total_intensity)
    return lam_grid, total_intensity, raw_intensity

def find_peak_indices(total_intensity, max_peaks=5):
    peak_indices = []
    for i in range(1, len(total_intensity) - 1):
        if total_intensity[i] > total_intensity[i - 1] and total_intensity[i] >= total_intensity[i + 1]:
            peak_indices.append(i)
    peak_indices = sorted(peak_indices, key=lambda i: total_intensity[i], reverse=True)
    return peak_indices[:max_peaks]

def print_first_20_valid_transitions(transitions):
    print("First 20 Valid Transitions:")
    print("-" * 120)
    for i, t in enumerate(transitions[:20], 1):
        print(f"{i:2d}. {t['e_state']} --> {t['h_state']}, λ = {t['lambda_nm']:.6f} nm, EM Spectrum Region = {t['region']}")

def print_top_20_closest_to_tallest_peak(transitions, lam_grid, total_intensity):
    tallest_idx = np.argmax(total_intensity)
    tallest_lambda = lam_grid[tallest_idx]
    tallest_height = total_intensity[tallest_idx]
    ranked = sorted(transitions, key=lambda t: abs(t['lambda_nm'] - tallest_lambda))
    print("Top 20 Contributing Transitions:")
    print("-" * 120)
    print(f"Peak Spectrum Location: λ = {tallest_lambda:.6f} nm, Normalized Intensity = {tallest_height:.6f}")
    print("-" * 120)
    for i, t in enumerate(ranked[:20], 1):
        print(f"{i:2d}. {t['e_state']} --> {t['h_state']}, λ = {t['lambda_nm']:.6f} nm, EM Spectrum Region = {t['region']}")

def find_peak_transitions(transitions, sigma_nm, lam_grid, total_intensity, max_peaks=5, window_factor=1.0, max_lines_per_peak=12):
    peak_indices = find_peak_indices(total_intensity, max_peaks=max_peaks)
    print("Peak Transitions:")
    print("-" * 120)
    for j, idx in enumerate(peak_indices, 1):
        peak_lambda = lam_grid[idx]
        peak_height = total_intensity[idx]
        window_nm = window_factor * sigma_nm

        contributing = [t for t in transitions if abs(t['lambda_nm'] - peak_lambda) <= window_nm]
        contributing = sorted(contributing, key=lambda t: abs(t['lambda_nm'] - peak_lambda))
        print(f"\nPeak {j}: λ = {peak_lambda:.6f} nm, Normalized Intensity = {peak_height:.6f}")
        print(f"Contributing Transitions within ±{window_nm:.3f} nm")
        print("-" * 120)
        if len(contributing) == 0:
            print("No transitions found in this peak window.")
        else:
            for k, t in enumerate(contributing[:max_lines_per_peak], 1):
                print(f"{k:2d}. {t['e_state']} --> {t['h_state']}, λ = {t['lambda_nm']:.6f} nm, EM Spectrum Region = {t['region']}")
            if len(contributing) > max_lines_per_peak:
                print(f"... {len(contributing) - max_lines_per_peak} more transitions in this peak window")

base_material_options = list(materials)
material_options_with_all = ['All Materials'] + base_material_options

material_dropdown = Dropdown(
    options=material_options_with_all,
    value='All Materials',
    description='Material:',
    layout={'width': '250px'}
)

T_dropdown = Dropdown(
    options=[float(v) for v in T_C],
    value=float(T_C[0]),
    description='T (°C):',
    layout={'width': '220px'}
)

Lx_dropdown = Dropdown(
    options=[float(v) for v in Lx],
    value=float(Lx[0]),
    description='Lₓ (nm):',
    layout={'width': '220px'}
)

Ly_dropdown = Dropdown(
    options=[float(v) for v in Ly],
    value=float(Ly[0]),
    description='Lᵧ (nm):',
    layout={'width': '220px'}
)

Lz_dropdown = Dropdown(
    options=[float(v) for v in Lz],
    value=float(Lz[0]),
    description='L𝓏 (nm):',
    layout={'width': '220px'}
)

epsilon_dropdown = Dropdown(
    options=[float(v) for v in epsilon],
    value=float(epsilon[0]),
    description='ε (kV/cm):',
    layout={'width': '220px'}
)

V0e_dropdown = Dropdown(
    options=[float(v) for v in V0e],
    value=float(V0e[0]),
    description='V₀ₑ (meV):',
    layout={'width': '220px'}
)

V0h_dropdown = Dropdown(
    options=[float(v) for v in V0h],
    value=float(V0h[0]),
    description='V₀ₕ (meV):',
    layout={'width': '220px'}
)

sigma_dropdown = Dropdown(
    options=[5.0, 8.0, 10.0, 12.0, 15.0, 20.0, 25.0, 30.0],
    value=12.0,
    description='σ (nm):',
    layout={'width': '220px'}
)

summary_mode_dropdown = Dropdown(
    options=[
        ('First 20 Valid Transitions', 'first20'),
        ('Top 20 Transitions Corresponding to Largest Contributions', 'top20closest'),
        ('Peak Transitions', 'peaks')
    ],
    value='first20',
    description='Printout:',
    layout={'width': '520px'}
)
output = Output()

def update_output(*args):
    with output:
        clear_output(wait=True)
        selected_material = material_dropdown.value
        Lx_val = Lx_dropdown.value
        Ly_val = Ly_dropdown.value
        Lz_val = Lz_dropdown.value
        eps_val = epsilon_dropdown.value
        V0e_val = V0e_dropdown.value
        V0h_val = V0h_dropdown.value
        T_val = T_dropdown.value
        sigma_nm = sigma_dropdown.value
        materials_to_use = base_material_options if selected_material == 'All Materials' else [selected_material]
        per_material_transitions = {}
        all_transitions = []
        for mat in materials_to_use:
            mat_transitions = build_transition_list_for_one_material(
                mat, Lx_val, Ly_val, Lz_val, eps_val, V0e_val, V0h_val, T_val
            )
            per_material_transitions[mat] = mat_transitions
            all_transitions.extend(mat_transitions)
        if len(all_transitions) == 0:
            print("No valid transitions found.")
            return
        all_wavelengths = [t['lambda_nm'] for t in all_transitions]
        global_lam_min = max(0.01, min(all_wavelengths) - 8.0 * sigma_nm)
        global_lam_max = max(2000.0, max(all_wavelengths) + 8.0 * sigma_nm)
        spectra_by_material = {}
        for mat in materials_to_use:
            mat_transitions = per_material_transitions[mat]
            lam_grid, total_intensity, raw_intensity = build_weighted_spectrum(
                mat_transitions, sigma_nm, lam_min=global_lam_min, lam_max=global_lam_max, npts=6000
            )
            spectra_by_material[mat] = {
                'lam_grid': lam_grid,
                'total_intensity': total_intensity,
                'raw_intensity': raw_intensity
            }
        if selected_material == 'All Materials':
            plt.figure(figsize=(11, 6))
            for mat in materials_to_use:
                plt.plot(
                    spectra_by_material[mat]['lam_grid'],
                    spectra_by_material[mat]['total_intensity'],
                    linewidth=3.0,
                    label=mat
                )
            plt.xlabel("Wavelength λ (nm)")
            plt.ylabel("Normalized Intensity (a.u.)")
            plt.title("Full Emission Spectrum: All Materials")
            plt.grid(True, alpha=0.3)
            plt.legend(loc='upper right', fontsize=10, title='Material', facecolor='lightgray', edgecolor='black')
            plt.show()
            reference_material = materials_to_use[0]
            lam_grid = spectra_by_material[reference_material]['lam_grid']
            total_intensity = spectra_by_material[reference_material]['total_intensity']
        else:
            lam_grid = spectra_by_material[selected_material]['lam_grid']
            total_intensity = spectra_by_material[selected_material]['total_intensity']
            peak_indices = find_peak_indices(total_intensity, max_peaks=5)
            plt.figure(figsize=(11, 6))
            plt.plot(lam_grid, total_intensity, color='darkmagenta', linewidth=4.0)
            plt.xlabel("Wavelength λ (nm)")
            plt.ylabel("Normalized Intensity (a.u.)")
            plt.title(f"Full Emission Spectrum: {selected_material}")
            plt.grid(True, alpha=0.3)
            for idx in peak_indices:
                peak_lambda = lam_grid[idx]
                peak_height = total_intensity[idx]
                plt.text(
                    peak_lambda,
                    peak_height + 0.03,
                    f"{peak_lambda:.1f} nm",
                    color='darkmagenta',
                    fontsize=10,
                    ha='center',
                    va='bottom'
                )
            plt.show()
        wavelengths = [t['lambda_nm'] for t in all_transitions]
        weights = [t['weight'] for t in all_transitions]
        overlaps_sq = [t['overlap_sq'] for t in all_transitions]
        photon_energies = [t['E_photon_eV'] for t in all_transitions]
        print("=" * 120)
        print("FULL EMISSION SPECTRUM")
        print("=" * 120)
        print(f"Material: {selected_material}")
        print(f"Lₓ = {Lx_val:.3f} nm,  Lᵧ = {Ly_val:.3f} nm,  L𝓏 = {Lz_val:.3f} nm")
        print(f"ε = {eps_val:.3f} kV/cm,  V₀ₑ = {V0e_val:.3f} meV,  V₀ₕ = {V0h_val:.3f} meV")
        print(f"T = {T_val:.3f} °C")
        print(f"σ = {sigma_nm:.3f} nm")
        print(f"Number of transitions included = {len(all_transitions)}")
        print(f"Minimum wavelength = {min(wavelengths):.6f} nm")
        print(f"Maximum wavelength = {max(wavelengths):.6f} nm")
        print(f"Minimum photon energy = {min(photon_energies):.6f} eV")
        print(f"Maximum photon energy = {max(photon_energies):.6f} eV")
        print(f"Minimum |O|² = {min(overlaps_sq):.6e}")
        print(f"Maximum |O|² = {max(overlaps_sq):.6e}")
        print(f"Minimum transition strength = {min(weights):.6e}")
        print(f"Maximum transition strength = {max(weights):.6e}")
        if selected_material == 'All Materials':
            print("\nTransitions per material")
            print("-" * 120)
            for mat in materials_to_use:
                print(f"{mat}: {len(per_material_transitions[mat])}")
            print("\nPeak Transitions (by material):")
            print("-" * 120)
            for mat in materials_to_use:
                mat_grid = spectra_by_material[mat]['lam_grid']
                mat_intensity = spectra_by_material[mat]['total_intensity']
                mat_peak_indices = find_peak_indices(mat_intensity, max_peaks=3)
                peak_text = ", ".join([f"{mat_grid[i]:.2f} nm" for i in mat_peak_indices])
                print(f"{mat}: {peak_text}")
        else:
            if summary_mode_dropdown.value == 'first20':
                print_first_20_valid_transitions(all_transitions)
            elif summary_mode_dropdown.value == 'top20closest':
                print_top_20_closest_to_tallest_peak(all_transitions, lam_grid, total_intensity)
            elif summary_mode_dropdown.value == 'peaks':
                find_peak_transitions(all_transitions, sigma_nm, lam_grid, total_intensity)

controls_row1 = HBox([material_dropdown, T_dropdown])
controls_row2 = HBox([Lx_dropdown, Ly_dropdown, Lz_dropdown])
controls_row3 = HBox([epsilon_dropdown, V0e_dropdown, V0h_dropdown])
controls_row4 = HBox([sigma_dropdown])
controls_row5 = HBox([summary_mode_dropdown])

ui = VBox([
    controls_row1,
    controls_row2,
    controls_row3,
    controls_row4,
    controls_row5,
    output
])

for w in [
    material_dropdown, T_dropdown,
    Lx_dropdown, Ly_dropdown, Lz_dropdown,
    epsilon_dropdown, V0e_dropdown, V0h_dropdown,
    sigma_dropdown, summary_mode_dropdown
]:
    w.observe(update_output, names='value')

display(ui)
update_output()

### 6. Animating the Quantum Dot Laser

In [102]:
laser_material_dropdown = Dropdown(
    options=materials,
    value=materials[0],
    description='Material:'
)

laser_mode_dropdown = Dropdown(
    options=["Beam only", "Wave only", "Dot + beam", "Dot + wave"],
    value="Dot + beam",
    description='Mode:'
)

laser_Lx_dropdown = Dropdown(options=[float(v) for v in Lx_vals], value=float(Lx_vals[0]), description='Lₓ (nm):')
laser_Ly_dropdown = Dropdown(options=[float(v) for v in Ly_vals], value=float(Ly_vals[0]), description='Lᵧ (nm):')
laser_Lz_dropdown = Dropdown(options=[float(v) for v in Lz_vals], value=float(Lz_vals[0]), description='Lz (nm):')
laser_epsilon_dropdown = Dropdown(options=[float(v) for v in epsilon_vals], value=float(epsilon_vals[0]), description='ε (kV/cm):')
laser_V0e_dropdown = Dropdown(options=[float(v) for v in V0e_vals], value=float(V0e_vals[0]), description='V₀ₑ (meV):')
laser_V0h_dropdown = Dropdown(options=[float(v) for v in V0h_vals], value=float(V0h_vals[0]), description='V₀ₕ (meV):')
laser_T_dropdown = Dropdown(options=[float(v) for v in T_C_vals], value=float(T_C_vals[0]), description='T (°C):')
laser_nxe_dropdown = Dropdown(options=n_vals, value=1, description='nₓₑ:')
laser_nye_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₑ:')
laser_nze_dropdown = Dropdown(options=n_vals, value=1, description='nzₑ:')
laser_nxh_dropdown = Dropdown(options=n_vals, value=1, description='nₓₕ:')
laser_nyh_dropdown = Dropdown(options=n_vals, value=1, description='nᵧₕ:')
laser_nzh_dropdown = Dropdown(options=n_vals, value=1, description='nzₕ:')
laser_frames_dropdown = Dropdown(options=[40, 60, 80, 100, 120], value=80, description='Frames:')
laser_output = Output()

def wavelength_region(lambda_nm):
    if lambda_nm < 0.01:
        return "Gamma ray"
    elif lambda_nm < 10.0:
        return "X-ray"
    elif lambda_nm < 400.0:
        return "Ultraviolet"
    elif lambda_nm < 700.0:
        return "Visible light"
    elif lambda_nm < 1.0e6:
        return "Infrared"
    elif lambda_nm < 1.0e9:
        return "Microwave"
    else:
        return "Radio wave"

def visible_color_from_wavelength(lambda_nm):
    lam = float(lambda_nm)
    if lam < 380 or lam > 780:
        return (1.0, 1.0, 1.0)

    if 380 <= lam < 440:
        r = -(lam - 440) / (440 - 380)
        g = 0.0
        b = 1.0
    elif 440 <= lam < 490:
        r = 0.0
        g = (lam - 440) / (490 - 440)
        b = 1.0
    elif 490 <= lam < 510:
        r = 0.0
        g = 1.0
        b = -(lam - 510) / (510 - 490)
    elif 510 <= lam < 580:
        r = (lam - 510) / (580 - 510)
        g = 1.0
        b = 0.0
    elif 580 <= lam < 645:
        r = 1.0
        g = -(lam - 645) / (645 - 580)
        b = 0.0
    else:
        r = 1.0
        g = 0.0
        b = 0.0

    if 380 <= lam < 420:
        factor = 0.3 + 0.7 * (lam - 380) / (420 - 380)
    elif 420 <= lam < 701:
        factor = 1.0
    else:
        factor = 0.3 + 0.7 * (780 - lam) / (780 - 700)

    return (r * factor, g * factor, b * factor)

def laser_style_from_wavelength(lambda_nm):
    region = wavelength_region(lambda_nm)

    if region == "Gamma ray":
        return {
            "region": region,
            "style": "seismic",
            "color": "lime",
            "glow_color": "lime",
            "linewidth": 2.2,
            "alpha": 0.95
        }
    elif region == "X-ray":
        return {
            "region": region,
            "style": "wave",
            "color": "teal",
            "glow_color": "cyan",
            "linewidth": 2.0,
            "alpha": 0.95
        }
    elif region == "Ultraviolet":
        return {
            "region": region,
            "style": "uv_plasma",
            "color": "lime",
            "glow_color": "cyan",
            "linewidth": 2.6,
            "alpha": 1.0
        }
    elif region == "Visible light":
        col = visible_color_from_wavelength(lambda_nm)
        return {
            "region": region,
            "style": "beam",
            "color": col,
            "glow_color": col,
            "linewidth": 2.6,
            "alpha": 0.95
        }
    elif region == "Infrared":
        return {
            "region": region,
            "style": "infrared_shimmer",
            "color": "lightgray",
            "glow_color": "white",
            "linewidth": 2.0,
            "alpha": 0.6
        }
    elif region == "Microwave":
        return {
            "region": region,
            "style": "dashed_beam",
            "color": "orange",
            "glow_color": "gold",
            "linewidth": 2.2,
            "alpha": 0.9
        }
    else:
        return {
            "region": region,
            "style": "radio",
            "color": "deepskyblue",
            "glow_color": "cyan",
            "linewidth": 2.0,
            "alpha": 0.9
        }

def overlap_intensity(material, Lx, Ly, Lz, V0e, V0h, nxe, nye, nze, nxh, nyh, nzh):
    deltae_nm = float(1e9 * delta_e(material, V0e))
    deltah_nm = float(1e9 * delta_h(material, V0h))
    O = overlap_from_parameters(
        Lx, Ly, Lz,
        deltae_nm, deltah_nm,
        nxe, nye, nze,
        nxh, nyh, nzh
    )
    return float(np.abs(O) ** 2)

def laser_animation(*args):
    with laser_output:
        clear_output(wait=True)
        material = laser_material_dropdown.value
        Lx = laser_Lx_dropdown.value
        Ly = laser_Ly_dropdown.value
        Lz = laser_Lz_dropdown.value
        epsilon = laser_epsilon_dropdown.value
        V0e = laser_V0e_dropdown.value
        V0h = laser_V0h_dropdown.value
        T = laser_T_dropdown.value
        nxe = laser_nxe_dropdown.value
        nye = laser_nye_dropdown.value
        nze = laser_nze_dropdown.value
        nxh = laser_nxh_dropdown.value
        nyh = laser_nyh_dropdown.value
        nzh = laser_nzh_dropdown.value
        mode = laser_mode_dropdown.value
        nframes = laser_frames_dropdown.value
        lambda_nm = float(np.asarray(photon_wavelength(
            material,
            nxe, nye, nze,
            nxh, nyh, nzh,
            Lx, Ly, Lz,
            epsilon,
            V0e,
            V0h,
            T
        )))
        E_gamma = float(np.asarray(J_to_eV(photon_energy(
            material,
            nxe, nye, nze,
            nxh, nyh, nzh,
            Lx, Ly, Lz,
            epsilon,
            V0e,
            V0h,
            T
        ))))
        intensity = overlap_intensity(material, Lx, Ly, Lz, V0e, V0h, nxe, nye, nze, nxh, nyh, nzh)
        style = laser_style_from_wavelength(lambda_nm)
        I_norm = intensity / (1.0 + intensity)
        glow_alpha = 0.25 + 0.60 * I_norm
        source_size = 80 + 500 * I_norm
        fig, ax = plt.subplots(figsize=(9, 5), facecolor='black')
        ax.set_facecolor('black')
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 6)
        ax.axis('off')
        electron_x, electron_y = 1.15, 4.50
        hole_x, hole_y = 1.15, 3.80
        recomb_x, recomb_y = 1.35, 4.15
        x1, y1 = 9.0, 0.7
        beam_back, = ax.plot([], [], color=style["glow_color"], lw=10, alpha=0.08, solid_capstyle='round')
        beam_mid, = ax.plot([], [], color=style["glow_color"], lw=6, alpha=0.14, solid_capstyle='round')
        beam_main, = ax.plot([], [], color=style["color"], lw=style["linewidth"], alpha=style["alpha"], solid_capstyle='round')
        wave_line, = ax.plot([], [], color=style["color"], lw=style["linewidth"], alpha=style["alpha"])
        electron_glow = ax.scatter([electron_x], [electron_y], s=source_size * 1.6, c=['deepskyblue'], alpha=0.10, edgecolors='none')
        electron_core = ax.scatter([electron_x], [electron_y], s=source_size * 0.45, c=['deepskyblue'], alpha=0.95, edgecolors='white', linewidths=0.6)
        hole_glow = ax.scatter([hole_x], [hole_y], s=source_size * 1.6, c=['hotpink'], alpha=0.10, edgecolors='none')
        hole_core = ax.scatter([hole_x], [hole_y], s=source_size * 0.45, c=['hotpink'], alpha=0.95, edgecolors='white', linewidths=0.6)
        recomb_glow = ax.scatter([recomb_x], [recomb_y], s=source_size * 4.0, c=[style["glow_color"]], alpha=0.00, edgecolors='none')
        recomb_mid = ax.scatter([recomb_x], [recomb_y], s=source_size * 1.8, c=[style["glow_color"]], alpha=0.00, edgecolors='none')
        recomb_core = ax.scatter([recomb_x], [recomb_y], s=source_size, c=[style["color"]], alpha=0.00, edgecolors='none')
        e_label = ax.text(
            electron_x - 0.45, electron_y + 0.28, "Electron quantum dot",
            color='white', fontsize=10, ha='left', va='center'
        )
        h_label = ax.text(
            hole_x - 0.45, hole_y - 0.28, "Hole quantum dot",
            color='white', fontsize=10, ha='left', va='center'
        )
        r_label = ax.text(
            recomb_x - 0.10, recomb_y + 0.38, "Recombination",
            color='white', fontsize=10, ha='left', va='center'
        )
        for txt in [e_label, h_label, r_label]:
            txt.set_path_effects([pe.withStroke(linewidth=2, foreground='black')])
        e_arrow = ax.annotate(
            "", xy=(electron_x, electron_y), xytext=(electron_x - 0.32, electron_y + 0.18),
            arrowprops=dict(arrowstyle='-', color='white', lw=1.0)
        )
        h_arrow = ax.annotate(
            "", xy=(hole_x, hole_y), xytext=(hole_x - 0.32, hole_y - 0.18),
            arrowprops=dict(arrowstyle='-', color='white', lw=1.0)
        )
        r_arrow = ax.annotate(
            "", xy=(recomb_x, recomb_y), xytext=(recomb_x + 0.06, recomb_y + 0.24),
            arrowprops=dict(arrowstyle='-', color='white', lw=1.0)
        )
        info = ax.text(
            0.60, 0.95, "",
            transform=ax.transAxes,
            color='white',
            fontsize=11,
            va='top',
            ha='left'
        )
        info.set_path_effects([pe.withStroke(linewidth=2, foreground='black')])
        t = np.linspace(0.0, 1.0, 700)
        xline = recomb_x + (x1 - recomb_x) * t
        yline = recomb_y + (y1 - recomb_y) * t
        dx = x1 - recomb_x
        dy = y1 - recomb_y
        L = np.sqrt(dx**2 + dy**2)
        nx_perp = -dy / L
        ny_perp = dx / L
        def animate(frame):
            phase = 2.0 * np.pi * frame / nframes
            progress = frame / (nframes - 1)
            beam_back.set_data([], [])
            beam_mid.set_data([], [])
            beam_main.set_data([], [])
            wave_line.set_data([], [])
            beam_main.set_linestyle('-')
            wave_line.set_linestyle('-')
            show_dot = ("Dot" in mode)
            show_beam = ("beam" in mode.lower())
            show_wave = ("wave" in mode.lower())
            e_pulse = 0.65 + 0.35 * np.sin(phase + 0.4)**2
            h_pulse = 0.65 + 0.35 * np.cos(phase + 0.7)**2
            electron_glow.set_sizes([source_size * 1.6 * e_pulse])
            electron_core.set_sizes([source_size * 0.45 * (0.9 + 0.2 * np.sin(phase)**2)])
            hole_glow.set_sizes([source_size * 1.6 * h_pulse])
            hole_core.set_sizes([source_size * 0.45 * (0.9 + 0.2 * np.cos(phase)**2)])
            electron_glow.set_alpha(0.12 if show_dot else 0.0)
            electron_core.set_alpha(0.95 if show_dot else 0.0)
            hole_glow.set_alpha(0.12 if show_dot else 0.0)
            hole_core.set_alpha(0.95 if show_dot else 0.0)
            e_label.set_alpha(1.0 if show_dot else 0.0)
            h_label.set_alpha(1.0 if show_dot else 0.0)
            e_arrow.arrow_patch.set_alpha(1.0 if show_dot else 0.0)
            h_arrow.arrow_patch.set_alpha(1.0 if show_dot else 0.0)
            recomb_flash = np.exp(-((progress - 0.14) / 0.09) ** 2)
            recomb_glow.set_sizes([source_size * 4.0 * (1.0 + 1.8 * recomb_flash)])
            recomb_mid.set_sizes([source_size * 1.8 * (1.0 + 1.4 * recomb_flash)])
            recomb_core.set_sizes([source_size * (1.0 + 0.9 * recomb_flash)])
            recomb_glow.set_alpha((0.22 + 0.30 * recomb_flash) if show_dot else 0.0)
            recomb_mid.set_alpha((0.32 + 0.32 * recomb_flash) if show_dot else 0.0)
            recomb_core.set_alpha((0.75 + 0.25 * recomb_flash) if show_dot else 0.0)
            r_label.set_alpha(1.0 if show_dot else 0.0)
            r_arrow.arrow_patch.set_alpha(1.0 if show_dot else 0.0)
            xseg = recomb_x + (x1 - recomb_x) * t[t <= progress]
            yseg = recomb_y + (y1 - recomb_y) * t[t <= progress]
            if style["style"] == "beam":
                if show_beam or not show_wave:
                    beam_back.set_data(xseg, yseg)
                    beam_mid.set_data(xseg, yseg)
                    beam_main.set_data(xseg, yseg)
            elif style["style"] == "uv_plasma":
                if show_beam or not show_wave:
                    flicker = 0.65 + 0.35 * np.sin(8 * phase)**2
                    beam_back.set_data(xseg, yseg)
                    beam_mid.set_data(xseg, yseg)
                    beam_main.set_data(xseg, yseg)
                    beam_back.set_color("white")
                    beam_mid.set_color("cyan")
                    beam_main.set_color("lime")
                    beam_back.set_linewidth(16)
                    beam_mid.set_linewidth(9)
                    beam_main.set_linewidth(3.5)
                    beam_back.set_alpha(0.12 * flicker)
                    beam_mid.set_alpha(0.40 * flicker)
                    beam_main.set_alpha(1.0)
            elif style["style"] == "dashed_beam":
                if show_beam or not show_wave:
                    beam_back.set_data(xseg, yseg)
                    beam_mid.set_data(xseg, yseg)
                    beam_main.set_data(xseg, yseg)
                    beam_main.set_linestyle((0, (7, 6)))
            elif style["style"] == "wave":
                if show_wave or not show_beam:
                    amp = 0.10 + 0.05 * np.sin(phase)
                    offset = amp * np.sin(2 * np.pi * 7 * t - 3 * phase)
                    xw = xline + nx_perp * offset
                    yw = yline + ny_perp * offset
                    mask = t <= progress
                    wave_line.set_data(xw[mask], yw[mask])
            elif style["style"] == "seismic":
                if show_wave or not show_beam:
                    amp = 0.16
                    jag = (
                        0.55 * np.sin(2 * np.pi * 10 * t - 4 * phase)
                        + 0.30 * np.sin(2 * np.pi * 23 * t + 1.7 * phase)
                        + 0.15 * np.sign(np.sin(2 * np.pi * 8 * t - 2 * phase))
                    )
                    offset = amp * jag
                    xw = xline + nx_perp * offset
                    yw = yline + ny_perp * offset
                    mask = t <= progress
                    wave_line.set_data(xw[mask], yw[mask])
            elif style["style"] == "infrared_shimmer":
                if show_beam or not show_wave:
                    amp = 0.05
                    shimmer = amp * np.sin(2 * np.pi * 12 * t - 5 * phase)
                    xw = xline + nx_perp * shimmer
                    yw = yline + ny_perp * shimmer
                    mask = t <= progress
                    beam_main.set_data(xw[mask], yw[mask])
                    beam_main.set_color("lightgray")
                    beam_main.set_alpha(0.5 + 0.2 * np.sin(phase)**2)
                    beam_main.set_linewidth(2.0)
            elif style["style"] == "radio":
                if show_wave or not show_beam:
                    amp = 0.22
                    offset = amp * np.sin(2 * np.pi * 4 * t - 2.5 * phase)
                    xw = xline + nx_perp * offset
                    yw = yline + ny_perp * offset
                    mask = t <= progress
                    wave_line.set_data(xw[mask], yw[mask])
                    wave_line.set_linestyle((0, (6, 5)))
            if style["style"] not in ["uv_plasma", "infrared_shimmer"]:
                beam_back.set_color(style["glow_color"])
                beam_mid.set_color(style["glow_color"])
                beam_main.set_color(style["color"])
                wave_line.set_color(style["color"])
                beam_main.set_linewidth(style["linewidth"])
                wave_line.set_linewidth(style["linewidth"])
                beam_back.set_alpha(0.10 + 0.06 * np.sin(phase)**2)
                beam_mid.set_alpha(0.18 + 0.08 * np.cos(phase)**2)
                beam_main.set_alpha(style["alpha"])
                wave_line.set_alpha(style["alpha"])
            info.set_text(
                f"Region: {style['region']}\n"
                f"λ = {lambda_nm:.3e} nm\n"
                f"Eᵧ = {E_gamma:.4f} eV\n"
                f"|O|² = {intensity:.4e}\n"
                f"Material: {material}\n"
                f"(nₑ, nₕ) = ({nxe},{nye},{nze}); ({nxh},{nyh},{nzh})"
            )
            return (
                beam_back, beam_mid, beam_main, wave_line,
                electron_glow, electron_core, hole_glow, hole_core,
                recomb_glow, recomb_mid, recomb_core,
                info, e_label, h_label, r_label
            )
        anim = FuncAnimation(fig, animate, frames=nframes, interval=45, blit=False, repeat=True)
        plt.close(fig)
        display(HTML(anim.to_jshtml()))

for widget in [
    laser_material_dropdown, laser_mode_dropdown,
    laser_Lx_dropdown, laser_Ly_dropdown, laser_Lz_dropdown,
    laser_epsilon_dropdown, laser_V0e_dropdown, laser_V0h_dropdown, laser_T_dropdown,
    laser_nxe_dropdown, laser_nye_dropdown, laser_nze_dropdown,
    laser_nxh_dropdown, laser_nyh_dropdown, laser_nzh_dropdown,
    laser_frames_dropdown
]:
    widget.observe(laser_animation, names='value')
display(VBox([
    HBox([laser_material_dropdown, laser_mode_dropdown, laser_frames_dropdown]),
    HBox([laser_Lx_dropdown, laser_Ly_dropdown, laser_Lz_dropdown]),
    HBox([laser_epsilon_dropdown, laser_V0e_dropdown, laser_V0h_dropdown, laser_T_dropdown]),
    HBox([laser_nxe_dropdown, laser_nye_dropdown, laser_nze_dropdown]),
    HBox([laser_nxh_dropdown, laser_nyh_dropdown, laser_nzh_dropdown]),
    laser_output
]))
laser_animation()